# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d09_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # water_quality_ewqi train
    shutil.copy(glob.glob("/kaggle/input/**/d09_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # water_quality_ewqi test
    print("Kaggle: water_quality_ewqi data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: water_quality_ewqi data copied to ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel.
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9 mapie==0.6.0 puncc==0.8.0
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
import ngboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from openpyxl.utils.dataframe import dataframe_to_rows
from scipy.stats import genpareto
from pytorch_tabnet import TabNetRegressor
import torch.nn as nn
import warnings
import io
import time
import numpy as np
import pandas as pd
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from openpyxl.utils.dataframe import dataframe_to_rows

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795378933.369607      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795378953.121036      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (water_quality_ewqi_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['pH', 'TDS', 'EC', 'TH', 'ca', 'mg', 'CO3', 'HCO3', 'NA', 'K', 'CHLORIDE', 'SULPHATE', 'NITRATE', 'FLUORIDE']
TARGET_LABEL = 'Pre-monsoon EWQI'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (47, 15)
First 5 rows of training data:
      pH   TDS    EC    TH   ca    mg  CO3  HCO3   NA    K  CHLORIDE  SULPHATE  \
0  7.90   552   865   452  272   180  0.2   4.5  260   44       375       681   
1  7.52  1985  2256  1054  529   525  0.0  15.0  272   47       728       324   
2  7.65  1865  2032   636  371   265  0.0   9.0  373   25       713       503   
3  7.25  4855  7856  2050  795  1255  0.0  12.5  335  106        81       676   
4  7.96  2855  3696   865  310   555  0.3   4.6  184  106       698       324   

   NITRATE  FLUORIDE  Pre monsoon EWQI   
0       60      0.40              70.52  
1       10      0.65             138.78  
2       54      0.85             113.07  
3       17      0.69             266.32  
4       25      1.20             158.05  

Shape of test data: (47, 15)
First 5 rows of test data:
      pH   TDS    EC    TH     ca      mg  CO3  HCO3   NA    K  CHLORIDE  \
0  8.38   625   975   320  120.0   200.0  0.0   5.8  280   35 

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (47, 14)
Shape of y_train: (47,)
Shape of X_test: (47, 14)
Shape of y_test: (47,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.3797749  -1.15230942 -0.94309903 -0.47923131 -0.09229732 -0.56852482
  -0.03922784 -1.18768739 -0.02713192 -0.87884263 -0.2770724   1.15873613
   1.41276153 -1.25316406]
 [-0.78765707 -0.2953837  -0.52059702  0.45803078  1.31357822  0.10564733
  -0.50015494  1.61405127  0.07159325 -0.77136925  1.29937074 -0.45273785
  -1.61366471 -0.53733141]
 [-0.38827245 -0.36714301 -0.58863473 -0.19275917  0.44926563 -0.40242444
  -0.50015494  0.01305775  0.90253007 -1.55950739  1.23238307  0.35525611
   1.04959038  0.03533472]
 [-1.61714822  1.42085972  1.18034563  2.00871688  2.76868675  1.53215652
  -0.50015494  0.94697063  0.58990037  1.34227396 -1.59003071  1.13616647
  -1.18996504 -0.42279818]
 [ 0.56410627  0.22487128 -0.08321177  0.16377408  0.11557533  0.16427099
   0.19123571 -1.16100417 -0.6523913   1.34227396  1.1653954  -0.45273785
  -0.70573684  1.03750043]]

First five rows of normalized X_test:
[[ 1.85442582 -1.10865585 -0.90968766 -0.6847

# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred): # Corrected: Removed y_test argument as it's not defined and not used
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}

    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=N_BINS):
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_test.to_numpy()[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) & (y_test.to_numpy()[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan) # or handle empty bins as needed
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages


class visualize:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test.to_numpy() >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = visualize.width(int_pred)
        coverages = visualize.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        n_samples = len(y_true)
        y_true_np = y_true.to_numpy().ravel() # Ensure y_true is numpy array and flattened

        # Determine points inside and outside the interval
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval

        # Plot points inside the interval in green
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        # Plot points outside the interval in orange
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')

        # Plot all ground truth values as blue crosses
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')


        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)], linestyle='--', color='blue', label='Ideal Prediction Line') # Changed label for clarity
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=N_BINS, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()

        # Calculate interval widths for test and train sets
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])

        # Plot histograms for both train and test widths
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')


        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=N_BINS, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()

        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]

        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')

        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax


# FIX N2: the interpolating version below is NOT the NexCP weighted quantile (no +inf mass for the
# test point, linear interpolation between residuals). Renamed so it no longer overrides the
# corrected `weighted_quantile` in the helper cell; kept only for reference.
def _old_weighted_quantile_interp(values, weights, q):
    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]
    cum_weights = np.cumsum(weights)
    cum_weights = cum_weights / cum_weights[-1]
    return np.interp(q, cum_weights, values)

In [13]:
# FIX N1-N9: helpers for the corrected NEXCP / online CP / COP / Extreme CP methods.
# Sources (see REFERENCES.md):
#   NexCP  - Barber, Candes, Ramdas & Tibshirani (2023), "Conformal prediction beyond exchangeability",
#            eq. (10)-(11) weighted split, eq. (12) weighted full, eq. (33) weighted jackknife+ / CV+.
#   J+aB   - Kim, Xu & Barber (2020), "Predictive inference is free with the jackknife+-after-bootstrap".
#   SF-OGD, SAOCP - Bhatnagar et al. (2023), ported line by line from the authors' code
#            github.com/salesforce/online_conformal (ogd.py, saocp.py).
#   FACI   - Gibbs & Candes (2024), ported from the same repo (faci.py); ACI - Gibbs & Candes (2021).
#   Extreme CP - Pasche, Lam & Engelke (2025), official R package ExtremeConformal (GPD_quantiles,
#            conformalize_GPD_max).
import math
from scipy.special import logsumexp

ALPHA = 0.1   # FIX M4: one miscoverage level for every method in this notebook (target coverage 90%)


# ---------------------------------------------------------------- N1: out-of-fold residuals
def _oof_fold_predictions(model_class, params, X_train, y_train, X_test, n_folds=5, seed=123):
    """Out-of-fold absolute residuals for every training row (each row scored by a model that never
    saw it), plus the fold models' predictions on X_test.
    Returns R (n,), mu_rows (n, n_test) = prediction at each test x of the fold model that left row i
    out, and fold_preds (K, n_test)."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    R = np.zeros(len(y))
    mu_rows = np.zeros((len(y), len(Xt)))
    fold_preds = []
    for tr, cal in KFold(n_splits=n_folds, shuffle=True, random_state=seed).split(X):
        m = model_class(**params)
        m.fit(X[tr], y[tr])
        R[cal] = np.abs(y[cal] - np.asarray(m.predict(X[cal])).ravel())
        p = np.asarray(m.predict(Xt)).ravel()
        mu_rows[cal] = p[None, :]
        fold_preds.append(p)
    return R, mu_rows, np.asarray(fold_preds)


def _jab_oob(model_class, params, X_train, y_train, X_test, n_bootstrap, seed=123):
    """Jackknife+-after-bootstrap: each training row is scored only by the bootstrap models whose
    sample did NOT contain it (out-of-bag). Rows that were in every bootstrap sample are dropped."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    n = len(y)
    rng = np.random.default_rng(seed)
    in_bag = np.zeros((n_bootstrap, n), dtype=bool)
    P_tr = np.zeros((n_bootstrap, n))
    P_te = np.zeros((n_bootstrap, len(Xt)))
    for b in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        in_bag[b, idx] = True
        m = model_class(**params)
        m.fit(X[idx], y[idx])
        P_tr[b] = np.asarray(m.predict(X)).ravel()
        P_te[b] = np.asarray(m.predict(Xt)).ravel()
    oob = ~in_bag
    has = oob.any(axis=0)
    cnt = np.maximum(oob.sum(axis=0), 1)
    mu_tr = (P_tr * oob).sum(axis=0) / cnt
    R = np.abs(y - mu_tr)
    mu_rows = (oob.T.astype(float) @ P_te) / cnt[:, None]
    return R[has], mu_rows[has], P_te.mean(axis=0), has


# ---------------------------------------------------------------- N2: NexCP weighted quantile
def weighted_quantile(values, weights, q):
    """NexCP eq. (10)-(11): Q_q( sum_i w~_i delta_{R_i} + w~_{n+1} delta_{+inf} ),
    w~_i = w_i / (w_1+...+w_n+1). Returns the smallest value whose cumulative normalised weight reaches q
    (a step quantile, no interpolation), or +inf when the test point's own mass is needed.
    With all weights 1 this is the usual split-conformal ceil((n+1)(1-alpha))-th smallest residual.
    Was: normalised by w_1+...+w_n only (no test-point mass) and interpolated with np.interp."""
    v = np.asarray(values, dtype=float).ravel()
    w = np.asarray(weights, dtype=float).ravel()
    order = np.argsort(v)
    v, w = v[order], w[order]
    cum = np.cumsum(w) / (w.sum() + 1.0)
    k = np.searchsorted(cum, q - 1e-12, side="left")
    return np.inf if k >= len(v) else float(v[k])


CAP_NOTE = "interval capped: weighted quantile reached +inf (set to the largest calibration error)"


def _cap_q(q, R):
    if np.isfinite(q):
        return q, ""
    return float(np.max(R)), CAP_NOTE


def _jackknife_plus(mu_rows, R, w, alpha):
    """Weighted jackknife+ / CV+ (NexCP eq. 33): for each test point,
    lower = Q_alpha( sum w~_i delta_{mu_-i(x) - R_i} + w~_{n+1} delta_{-inf} ),
    upper = Q_{1-alpha}( sum w~_i delta_{mu_-i(x) + R_i} + w~_{n+1} delta_{+inf} ).
    Infinite bounds are capped at mu +/- max(R) and counted."""
    lo, hi, n_cap = [], [], 0
    for j in range(mu_rows.shape[1]):
        mu = mu_rows[:, j]
        l = -weighted_quantile(-(mu - R), w, 1 - alpha)
        h = weighted_quantile(mu + R, w, 1 - alpha)
        if not (np.isfinite(l) and np.isfinite(h)):
            n_cap += 1
            l = l if np.isfinite(l) else float(np.mean(mu) - np.max(R))
            h = h if np.isfinite(h) else float(np.mean(mu) + np.max(R))
        lo.append(l)
        hi.append(h)
    return np.asarray(lo), np.asarray(hi), n_cap


def _cap_count_note(n_cap, n):
    return "" if n_cap == 0 else f"{n_cap} of {n} intervals capped: weighted quantile reached +inf"


# ---------------------------------------------------------------- N3: NexCP full conformal (eq. 12)
def _nexcp_full(model_class, params, X_train, y_train, X_test, yhat_test, w, alpha, half_range,
                grid_size=50, label=""):
    """For each test x and each candidate y on a grid of `grid_size` values (prediction +/- half_range):
    refit on train + (x, y), compute all residuals R^y, and keep y if the test residual is not above
    the weighted (1-alpha) quantile of { R^y_i (weight w~_i), R^y_{n+1} (weight w~_{n+1}) }.
    The interval is [smallest kept y, largest kept y]."""
    X = np.asarray(X_train)
    y = np.asarray(y_train).ravel()
    Xt = np.asarray(X_test)
    wt = np.asarray(w, dtype=float) / (np.sum(w) + 1.0)
    lo, hi, n_edge, n_empty = [], [], 0, 0
    t0 = time.time()
    for j in range(len(Xt)):
        grid = np.linspace(yhat_test[j] - half_range, yhat_test[j] + half_range, grid_size)
        X_aug = np.vstack([X, Xt[j:j + 1]])
        keep = []
        for yc in grid:
            y_aug = np.append(y, yc)
            m = model_class(**params)
            m.fit(X_aug, y_aug)
            r = np.abs(y_aug - np.asarray(m.predict(X_aug)).ravel())
            if wt[r[:-1] < r[-1]].sum() < 1 - alpha:
                keep.append(yc)
        if keep:
            lo.append(min(keep))
            hi.append(max(keep))
            n_edge += int(np.isclose(keep[0], grid[0]) or np.isclose(keep[-1], grid[-1]))
        else:
            lo.append(yhat_test[j])
            hi.append(yhat_test[j])
            n_empty += 1
        if (j + 1) % 5 == 0 or j == len(Xt) - 1:
            print(f"  NexCP-Full {label}: {j + 1}/{len(Xt)} test points, {(time.time() - t0) / 60:.1f} min")
    notes = []
    if n_edge:
        notes.append(f"{n_edge} of {len(Xt)} intervals reach the edge of the y-grid (true interval may be wider)")
    if n_empty:
        notes.append(f"{n_empty} empty sets (set to the point prediction)")
    return np.asarray(lo), np.asarray(hi), "; ".join(notes)


# ---------------------------------------------------------------- N4/N6: SF-OGD and SAOCP (salesforce/online_conformal)
def _pinball(y, yhat, q):
    return np.maximum(q * (y - yhat), (1 - q) * (yhat - y))


def _pinball_grad(y, yhat, q):
    return -q * (y > yhat) + (1 - q) * (y < yhat)


def _sf_ogd(calib, stream, alpha):
    """Scale-free OGD on the pinball loss (ogd.py ScaleFreeOGD): warm-started on the calibration
    residuals, then for each test point the margin is OUTPUT first and updated after its error is seen."""
    coverage = 1 - alpha
    scale = float(np.max(np.abs(calib))) * np.sqrt(3) if len(calib) else 1.0
    state = {"delta": 0.0, "gn": 0.0}

    def update(s):
        g = _pinball_grad(abs(s), state["delta"], coverage)
        state["gn"] += g ** 2
        if state["gn"] != 0:
            state["delta"] = max(0.0, state["delta"] - scale / np.sqrt(3 * state["gn"]) * g)

    for s in calib:
        update(s)
    out = []
    for s in stream:
        out.append(state["delta"])
        update(s)
    return np.asarray(out)


class _SAOCPExpert:
    """saocp.py _OGD: scale-free OGD expert started at time t with lifetime g * 2^u."""

    def __init__(self, t, scale, alpha, yhat_0, g=8):
        self.scale = scale
        self.base_lr = scale / np.sqrt(3)
        self.alpha = alpha
        self.yhat = yhat_0
        self.grad_norm = 0
        u = 0
        while t % 2 == 0:
            t /= 2
            u += 1
        self.lifetime = g * 2 ** u
        self.z = 0
        self.wz = 0
        self.s_t = 0

    @property
    def expired(self):
        return self.s_t > self.lifetime

    def loss(self, y):
        return _pinball(y, self.yhat, 1 - self.alpha)

    @property
    def w(self):
        return 0 if self.s_t == 0 else self.z / self.s_t * (1 + self.wz)

    def update(self, y, meta_loss):
        w = self.w
        g = np.clip((meta_loss - self.loss(y)) / self.scale / max(self.alpha, 1 - self.alpha), -1 * (w > 0), 1)
        self.z += g
        self.wz += g * w
        self.s_t += 1
        grad = _pinball_grad(y, self.yhat, 1 - self.alpha)
        self.grad_norm += grad ** 2
        if self.grad_norm != 0:
            self.yhat = max(0, self.yhat - self.base_lr / np.sqrt(self.grad_norm) * grad)


def _saocp(calib, stream, alpha, lifetime=8):
    """saocp.py SAOCP (coin-betting combination of SF-OGD experts started at every step)."""
    coverage = 1 - alpha
    scale = float(np.max(np.abs(calib))) * np.sqrt(3) if len(calib) else 1.0
    experts = {}
    state = {"t": 1}

    def get_p():
        if not experts:
            return {}
        prior = {t: 1 / (t ** 2 * (1 + np.floor(np.log2(t)))) for t in experts}
        z = sum(prior.values())
        prior = {t: v / z for t, v in prior.items()}
        p = {t: prior[t] * max(0, e.w) for t, e in experts.items()}
        sp = sum(p.values())
        return {t: v / sp for t, v in p.items()} if sp > 0 else prior

    def predict():
        p = get_p()
        return sum(p[t] * e.yhat for t, e in experts.items())

    def update(s):
        s = abs(s)
        s_hat = predict()
        for t in [k for k, e in experts.items() if e.expired]:
            experts.pop(t)
        experts[state["t"]] = _SAOCPExpert(state["t"], scale, 1 - coverage, yhat_0=s_hat, g=lifetime)
        meta_loss = _pinball(s, predict(), coverage)
        for e in experts.values():
            e.update(s, meta_loss)
        state["t"] += 1

    for s in calib:
        update(s)
    out = []
    for s in stream:
        out.append(predict())
        update(s)
    return np.asarray(out)


# ---------------------------------------------------------------- N5: FACI (faci.py) and ACI
def _faci(calib, stream, alpha):
    """Fully Adaptive Conformal Inference: 8 ACI experts with step sizes 0.001 * 2^k, combined with
    exponential weights on the pinball loss (Gibbs & Candes 2024, Algorithm 1; defaults of faci.py)."""
    coverage = 1 - alpha
    gammas = np.asarray([0.001 * 2 ** k for k in range(8)])
    k = len(gammas)
    I = 100
    sigma = 1 / (2 * I)
    a = 1 - coverage
    denom = ((1 - a) ** 2 * a ** 3 + a ** 2 * (1 - a) ** 3) / 3
    eta = np.sqrt(3 / I) * np.sqrt((np.log(I * k) + 2) / denom)
    alphas = np.full(k, a)
    log_w = np.zeros(k)
    residuals = list(np.abs(calib))
    out = []
    for s in np.abs(stream):
        w = np.exp(log_w - logsumexp(log_w))
        a_t = float(np.dot(w, alphas))
        out.append(float(np.quantile(residuals, np.clip(1 - a_t, 0, 1), method="inverted_cdf")))
        if len(residuals) > math.floor(1 / (1 - coverage)):
            beta = np.mean(np.asarray(residuals) >= s)
            losses = _pinball(beta, alphas, 1 - coverage)
            wbar = log_w - eta * losses
            log_w = logsumexp([wbar, np.full(k, logsumexp(wbar))], b=[[1 - sigma], [sigma / k]], axis=0)
            log_w = log_w - logsumexp(log_w)
            err = alphas > beta
            alphas = np.clip(alphas + gammas * ((1 - coverage) - err), 0, 1)
        residuals.append(s)
    return np.asarray(out)


def _aci(calib, stream, alpha, gamma=0.005):
    """Adaptive Conformal Inference (Gibbs & Candes 2021): alpha_{t+1} = alpha_t + gamma (alpha - err_t).
    If alpha_t <= 0 the ACI set is the whole line; it is capped at the largest error seen and counted."""
    a_t = alpha
    residuals = list(np.abs(calib))
    out, n_cap = [], 0
    for s in np.abs(stream):
        level = 1 - a_t
        if level >= 1:
            q = float(np.max(residuals))
            n_cap += 1
        elif level <= 0:
            q = 0.0
        else:
            q = float(np.quantile(residuals, level, method="inverted_cdf"))
        out.append(q)
        a_t = a_t + gamma * (alpha - float(s > q))
        residuals.append(s)
    return np.asarray(out), n_cap


# ---------------------------------------------------------------- N9: Extreme CP (ExtremeConformal GPD_max)
def _gpd_quantile(p, p0, u, sigma, xi):
    """ExtremeConformal GPD_quantiles: q = u + (sigma/xi) [((1-p)/(1-p0))^(-xi) - 1]  (xi = 0: log form)."""
    if abs(xi) < 1e-12:
        return u + sigma * np.log((1 - p0) / (1 - p))
    return u + (sigma / xi) * (((1 - p) / (1 - p0)) ** (-xi) - 1)


def _conformalize_gpd_max(s_cal, coverage, threshold_lvls=None, min_obs_gpd=10):
    """ExtremeConformal conformalize_GPD_max: for each threshold level p0 (default 0.80..0.99, capped so
    at least `min_obs_gpd` scores lie above it) fit a GPD to the exceedances over u = quantile(p0),
    compute the extrapolated `coverage` quantile, and return the maximum over thresholds."""
    s_cal = np.asarray(s_cal, dtype=float).ravel()
    n = len(s_cal)
    lv = np.round(np.arange(0.80, 0.9901, 0.01), 2) if threshold_lvls is None else np.asarray(threshold_lvls, float)
    if min_obs_gpd > 0:
        lv = np.minimum(lv, 1 - min_obs_gpd / n)
    lv = np.unique(lv)
    rows = []
    for p0 in lv:
        u = float(np.quantile(s_cal, p0))
        exc = s_cal[s_cal > u] - u
        try:
            xi, _, sigma = genpareto.fit(exc, floc=0)
            dq = float(_gpd_quantile(coverage, p0, u, sigma, xi))
        except Exception:
            xi = sigma = dq = np.nan
        rows.append({"threshold_level": float(p0), "threshold_u": u, "n_exceedances": int(len(exc)),
                     "sigma": float(sigma), "xi": float(xi), "q_threshold": dq})
    table = pd.DataFrame(rows)
    finite = table["q_threshold"].replace([np.inf, -np.inf], np.nan).dropna()
    return (float(finite.max()) if len(finite) else np.nan), table


In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_NEXCP(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    decay=0.99,
    n_folds=5,
    n_bootstrap=50,
    full_grid_size=50,   # FIX N3: y-grid points per test point for NexCP-Full
):

    warnings.filterwarnings("ignore")

    SEED = 123
    np.random.seed(SEED)

    strategies_names = {
        "NexCP-Split": "NexCP Split",
        "NexCP-Full": "NexCP Full",
        "NexCP-J+aB": "NexCP Jackknife+ aB",
        "NexCP-CV+": "NexCP CV+",
    }

    y_pred, int_pred = {}, {}

    # =========================================================
    # FIX N1-N3: corrected NexCP methods (Barber et al. 2023). Weights w_i = decay^(n-i): training rows
    # in their original (time) order, newest row weight 1. Residuals are OUT-OF-FOLD (each row scored by
    # a model that never saw it). Was: in-sample training residuals for every variant, an unweighted
    # "Full", bounds averaged (CV+) or min/max-ed (J+aB) without weights.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    n_tr = len(y_tr)
    weights = decay ** np.arange(n_tr - 1, -1, -1)
    notes = {}

    model = model_class(**best_params)
    model.fit(X_tr, y_tr)
    yhat_test = np.asarray(model.predict(X_te)).ravel()
    R_oof, mu_te_folds, fold_preds = _oof_fold_predictions(model_class, best_params, X_tr, y_tr, X_te, n_folds, SEED)

    # NexCP - Split, eq. (11)
    q, notes["NexCP-Split"] = _cap_q(weighted_quantile(R_oof, weights, 1 - alpha), R_oof)
    y_pred["NexCP-Split"] = yhat_test
    int_pred["NexCP-Split"] = np.c_[yhat_test - q, yhat_test + q]

    # NexCP - Full, eq. (12), on a grid of candidate y values around the prediction (+/- 2 x split margin)
    lo, hi, notes["NexCP-Full"] = _nexcp_full(model_class, best_params, X_tr, y_tr, X_te, yhat_test, weights,
                                              alpha, half_range=2.0 * q, grid_size=full_grid_size, label=model_name)
    y_pred["NexCP-Full"] = yhat_test
    int_pred["NexCP-Full"] = np.c_[lo, hi]

    # NexCP - Jackknife+ after bootstrap: out-of-bag residuals + eq. (33) weighted quantiles
    R_oob, mu_te_oob, bag_mean, has = _jab_oob(model_class, best_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
    lo, hi, n_cap = _jackknife_plus(mu_te_oob, R_oob, weights[has], alpha)
    notes["NexCP-J+aB"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-J+aB"] = bag_mean
    int_pred["NexCP-J+aB"] = np.c_[lo, hi]

    # NexCP - CV+, eq. (33) with K folds
    lo, hi, n_cap = _jackknife_plus(mu_te_folds, R_oof, weights, alpha)
    notes["NexCP-CV+"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-CV+"] = np.mean(fold_preds, axis=0)
    int_pred["NexCP-CV+"] = np.c_[lo, hi]

    # =========================================================
    # Metrics
    # =========================================================
    def coverage(y, l, u):
        return np.mean((y >= l) & (y <= u))

    def width(l, u):
        return np.mean(u - l)

    def cwc(l, u, y, alpha):
        cov = coverage(y, l, u)
        w = width(l, u)
        penalty = np.exp(-50 * (cov - (1 - alpha)))
        return w * (1 + penalty if cov < (1 - alpha) else 1)

    y_true = y_test.to_numpy()

    summary_rows = []

    print("\n===== NEXCP RESULTS =====")

    for s in strategies_names:
        cov = coverage(y_true, int_pred[s][:, 0], int_pred[s][:, 1])
        wid = width(int_pred[s][:, 0], int_pred[s][:, 1])
        rmse = np.sqrt(mean_squared_error(y_true, y_pred[s]))
        cwc_val = cwc(int_pred[s][:, 0], int_pred[s][:, 1], y_true, alpha)

        summary_rows.append([
            strategies_names[s],
            cov,
            wid,
            rmse,
            cwc_val,
        ])

        print(f"{strategies_names[s]} - Coverage: {cov*100:.3f}% | Width: {wid:.3f}")

    summary_df = pd.DataFrame(
        summary_rows,
        columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"],
    )
    summary_df["Note"] = [notes.get(s, "") for s in strategies_names]  # FIX N10: caps / limits per method

    # =========================================================
    # Plotting (SAVE ONLY — NO SHOW)
    # =========================================================
    idx = np.arange(len(y_test))
    plot_buffers = {}

    for s in strategies_names:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, y_test, color="blue", s=10, alpha=0.7, label="True")
        ax.fill_between(
            idx,
            int_pred[s][:, 0],
            int_pred[s][:, 1],
            color="gray",
            alpha=0.5,
            label="Prediction interval",
        )
        ax.plot(idx, y_pred[s], color="red", linewidth=1, label="Predicted mean")

        ax.set_title(strategies_names[s])
        ax.set_xlabel("Sample Number")
        ax.set_ylabel(f"{TARGET_LABEL}")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # =========================================================
    # Excel Export
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary sheet ----
        sheet_name = "NEXCP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name, 0)
        for r_idx, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c_idx, val in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=val)

        # ---- Plot sheets ----
        for s, buf in plot_buffers.items():
            img = Image(buf)

            sheet_name = f"NEXCP_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1

            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values sheet ----
        results_dict = {}
        for s in strategies_names:
            results_dict[f"{s}_true"] = y_true
            results_dict[f"{s}_pred"] = y_pred[s]
            results_dict[f"{s}_lower"] = int_pred[s][:, 0]
            results_dict[f"{s}_upper"] = int_pred[s][:, 1]

        df_vals = pd.DataFrame(results_dict)

        sheet_name = "NEXCP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r_idx, row in enumerate(dataframe_to_rows(df_vals, index=False, header=True), 1):
            for c_idx, val in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_online_super(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    strategies="all",
    window_min=50,
    window_max=500,
    window_grid=(25, 50, 100, 200, 400),
    n_bootstrap=30,
):

    warnings.filterwarnings("ignore")

    SEED = 123
    np.random.seed(SEED)

    STRATEGY_NAMES = {
        "Online-Split": "Online Split",
        "Online-CV+": "Online CV+",
        "Online-J+aB": "Online Jackknife+ aB",
        "SAOCP": "SAOCP",
        "SF-OGD": "SF-OGD",
        "FACI": "FACI",
        "ACI": "ACI",   # FIX N5: added (Gibbs & Candes 2021, gamma = 0.005)
        "MFCS-Split": "MFCS Split",
        "MFCS-Full": "MFCS Full",
    }

    ALL_STRATEGIES = list(STRATEGY_NAMES.keys())
    if strategies == "all":
        strategies = ALL_STRATEGIES

    # =========================================================
    # FIX N1, N4-N6: corrected online methods. Every method starts from OUT-OF-FOLD training residuals;
    # test points arrive one at a time and each interval is OUTPUT before that point's error is used.
    # Was: in-sample training residuals; SF-OGD with flipped gradient sign and look-ahead; "FACI" and
    # "SAOCP" were window heuristics (FACI also looked ahead); J+aB scored models on their own samples.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    y_te = np.asarray(y_test).ravel()
    n_train, n_test = len(y_tr), len(y_te)

    base_model = model_class(**model_params)
    base_model.fit(X_tr, y_tr)
    yhat_te = np.asarray(base_model.predict(X_te)).ravel()
    res_te = np.abs(y_te - yhat_te)
    R_oof, mu_te_folds, _ = _oof_fold_predictions(model_class, model_params, X_tr, y_tr, X_te, 5, SEED)

    y_pred, int_pred, notes = {}, {}, {}

    def _band(name, centre, q):
        y_pred[name] = np.asarray(centre)
        int_pred[name] = np.c_[centre - q, centre + q]

    if "Online-Split" in strategies:
        q, notes["Online-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("Online-Split", yhat_te, q)

    if "Online-CV+" in strategies:
        # CV+ (Barber et al. 2021); past test points join the calibration set once their error is known
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_te_folds[:, j], np.full(j, yhat_te[j])]
            R = np.r_[R_oof, res_te[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-CV+"] = yhat_te
        int_pred["Online-CV+"] = np.c_[lo, hi]
        notes["Online-CV+"] = _cap_count_note(n_cap, n_test)

    if "Online-J+aB" in strategies:
        # J+aB (Kim, Xu & Barber 2020): out-of-bag residuals; past test points join as above
        R_oob, mu_oob, bag_mean, has = _jab_oob(model_class, model_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
        res_bag = np.abs(y_te - bag_mean)
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_oob[:, j], np.full(j, bag_mean[j])]
            R = np.r_[R_oob, res_bag[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-J+aB"] = bag_mean
        int_pred["Online-J+aB"] = np.c_[lo, hi]
        notes["Online-J+aB"] = _cap_count_note(n_cap, n_test)

    if "SAOCP" in strategies:
        _band("SAOCP", yhat_te, _saocp(R_oof, res_te, alpha))

    if "SF-OGD" in strategies:
        _band("SF-OGD", yhat_te, _sf_ogd(R_oof, res_te, alpha))

    if "FACI" in strategies:
        _band("FACI", yhat_te, _faci(R_oof, res_te, alpha))

    if "ACI" in strategies:
        q_seq, n_cap = _aci(R_oof, res_te, alpha, gamma=0.005)
        _band("ACI", yhat_te, q_seq)
        notes["ACI"] = "" if n_cap == 0 else f"{n_cap} of {n_test} steps had alpha_t <= 0 (infinite ACI set, capped at the largest error)"

    if "MFCS-Split" in strategies:
        # kept as written by the user (no published source found); same computation as Online-Split
        q, notes["MFCS-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("MFCS-Split", yhat_te, q)

    if "MFCS-Full" in strategies:
        # refit on all points seen so far; margin from that refit's OUT-OF-FOLD residuals (FIX N1)
        lo, hi, preds, n_cap = [], [], [], 0
        for j in range(n_test):
            X_h = np.vstack([X_tr, X_te[:j]])
            y_h = np.r_[y_tr, y_te[:j]]
            m = model_class(**model_params)
            m.fit(X_h, y_h)
            yhat = float(np.asarray(m.predict(X_te[j:j + 1])).ravel()[0])
            R_h, _, _ = _oof_fold_predictions(model_class, model_params, X_h, y_h, X_te[j:j + 1], 5, SEED)
            q = weighted_quantile(R_h, np.ones(len(R_h)), 1 - alpha)
            if not np.isfinite(q):
                q, n_cap = float(np.max(R_h)), n_cap + 1
            preds.append(yhat); lo.append(yhat - q); hi.append(yhat + q)
        y_pred["MFCS-Full"] = np.asarray(preds)
        int_pred["MFCS-Full"] = np.c_[lo, hi]
        notes["MFCS-Full"] = _cap_count_note(n_cap, n_test)

    # =========================================================
    # METRICS
    # =========================================================
    y_true = y_test.to_numpy().ravel()

    rows = []
    for s in y_pred:
        ymin, ymax = int_pred[s].T
        cov = np.mean((y_true >= ymin) & (y_true <= ymax))
        wid = np.mean(ymax - ymin)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred[s]))
        cwc = wid * (1 + (cov < (1 - alpha)) * (1 - cov))

        rows.append([STRATEGY_NAMES[s], cov, wid, rmse, cwc])

    summary_df = pd.DataFrame(
        rows, columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"]
    )
    summary_df["Note"] = [notes.get(s, "") for s in y_pred]  # FIX N10: caps / limits per method

    print("\n===== ONLINE CP RESULTS =====")
    print(summary_df)

    # =========================================================
    # PLOTTING (SAVE ONLY)
    # =========================================================
    plot_buffers = {}
    idx = np.arange(len(y_test))

    for s in y_pred:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, y_true, color="blue", s=10, alpha=0.7, label="True")
        ax.fill_between(idx,
                        int_pred[s][:, 0],
                        int_pred[s][:, 1],
                        color="gray",
                        alpha=0.5,
                        label="Prediction interval")
        ax.plot(idx, y_pred[s], color="red", linewidth=1, label="Predicted mean")

        ax.set_title(f"{STRATEGY_NAMES[s]} ({model_name})")
        ax.set_xlabel("Sample Number")
        ax.set_ylabel(f"{TARGET_LABEL}")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # =========================================================
    # EXCEL EXPORT (APPEND SAFE)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "ONLINECP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plots ----
        for s, buf in plot_buffers.items():
            img = Image(buf)
            sheet_name = f"ONLINECP_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1
            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values ----
        values = {}
        for s in y_pred:
            values[f"{s}_true"] = y_true
            values[f"{s}_pred"] = y_pred[s]
            values[f"{s}_lower"] = int_pred[s][:, 0]
            values[f"{s}_upper"] = int_pred[s][:, 1]

        df_vals = pd.DataFrame(values)

        sheet_name = "ONLINECP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(df_vals, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_COP(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    lr=0.05,          # Learning rate for COP
    T_burnin=COP_T_BURNIN,      # Window size for distribution estimation
    scale=0.5,        # Scale factor for the integrator
    random_state=123,
):
    """
    Conformal Optimistic Prediction (COP)
    ✔ Implements the Distribution-Informed Gradient logic from the repository.
    ✔ Matches the "Excel-safe" and plotting structure of your previous functions.
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # 1. Train Base Model & Generate Test Predictions
    # =========================================================
    # COP is an online calibration method, but we usually start with a pre-trained model
    # or an online model. Here we fit on Train and calibrate on Test stream.
    model = model_class(**model_params)
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred_test = model.predict(X_test).ravel()
    y_true_test = y_test.to_numpy().ravel()

    # Calculate non-conformity scores (Absolute Residuals)
    # This assumes symmetric intervals.
    scores = np.abs(y_true_test - y_pred_test)
    T_test = len(scores)

    # =========================================================
    # 2. COP Initialization
    # =========================================================
    # Initialize quantiles based on training data residuals if possible
    y_pred_train = model.predict(X_train).ravel()
    scores_train = np.abs(y_train.to_numpy().ravel() - y_pred_train)
    # FIX N1: start from OUT-OF-FOLD training residuals (was: in-sample residuals, too small)
    R_oof_cop, _, _ = _oof_fold_predictions(model_class, model_params, np.asarray(X_train),
                                            np.asarray(y_train).ravel(), np.asarray(X_test)[:1], 5, 123)
    init_q = np.quantile(R_oof_cop, 1 - alpha)

    # Arrays to store history
    qs = np.zeros(T_test)          # Final adaptive quantile
    qts = np.zeros(T_test)         # Base quantile term
    integrators = np.zeros(T_test) # Optimistic integrator term
    covereds = np.zeros(T_test)    # Coverage boolean

    # Set initial values
    qs[0] = init_q
    qts[0] = init_q

    # =========================================================
    # 3. Online COP Loop
    # =========================================================
    # Logic adapted from `core/methods.py` function `COP`

    for t in range(T_test - 1):
        # 1. Observe coverage at current step t
        covereds[t] = 1 if qs[t] >= scores[t] else 0

        # FIX N8: step size proportional to the score scale, as in the official COP code
        # (lr x (max - min) of the last T_burnin scores). Was: a fixed lr = 0.05 in target units.
        _hist = np.r_[R_oof_cop, scores[:t + 1]][-T_burnin:]
        lr_t = lr * (np.max(_hist) - np.min(_hist))

        # 2. Calculate Standard Gradient
        # If covered, we decrease q (grad is positive). If not, increase q.
        grad = alpha if covereds[t] else -(1 - alpha)

        # 3. Calculate Distribution-Informed Gradient (The "Optimistic" part)
        if t < T_burnin:
            grad_i = 0
        else:
            # Look at the window of recent scores
            window_s = scores[t - T_burnin : t]

            # Calculate ECDF at the predicted next quantile step
            # logic: np.mean(window_s <= qts[t] - lr * grad) - (1 - alpha)
            current_target = qts[t] - lr_t * grad   # FIX N8
            grad_i = np.mean(window_s <= current_target) - (1 - alpha)

        # 4. Update Terms
        integrator = -scale * grad_i

        # Update base quantile (Gradient Descent)
        qts[t + 1] = qts[t] - lr_t * grad   # FIX N8

        # Update integrator state
        integrators[t + 1] = lr_t * integrator   # FIX N8

        # Combine for final quantile
        qs[t + 1] = qts[t + 1] + integrators[t + 1]

        # Safety: Ensure quantile implies non-negative width
        qs[t + 1] = max(0, qs[t + 1])

    # Handle the final step coverage for metrics
    covereds[-1] = 1 if qs[-1] >= scores[-1] else 0

    # =========================================================
    # 4. Construct Prediction Intervals
    # =========================================================
    ymin = y_pred_test - qs
    ymax = y_pred_test + qs

    # =========================================================
    # 5. Metrics
    # =========================================================
    rmse = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
    coverage = np.mean(covereds)
    avg_width = np.mean(ymax - ymin)

    print(f"\n===== CONFORMAL OPTIMISTIC PREDICTION ({model_name}) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Avg Width: {avg_width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"Final q  : {qs[-1]:.3f}")

    # =========================================================
    # 6. Summary Tables
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "COP",
        "Coverage": coverage,
        "Width": avg_width,
        "RMSE": rmse,
        "Alpha": alpha,
        "LR": lr,
        "Scale": scale
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true_test)),
        "y_true": y_true_test,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": ymax - ymin,
        "is_covered": covereds.astype(int),
        "score_residual": scores,
        "adaptive_q": qs
    })

    # =========================================================
    # 7. Plotting
    # =========================================================
    idx = np.arange(len(y_true_test))
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(idx, y_true_test, color="blue", s=10, alpha=0.7, label="True")
    ax.fill_between(idx, ymin, ymax, color="gray", alpha=0.5, label="COP Interval")
    ax.plot(idx, y_pred_test, color="red", linewidth=1, label="Predicted mean")

    ax.set_title(f"Conformal Optimistic Prediction (COP) - {model_name}")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel(f"{TARGET_LABEL}")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # 8. Excel Export
    # =========================================================
    if excel_file_path:
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            # Create a new workbook if it doesn't exist
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active) # Remove default sheet

        # ---- Summary ----
        sheet_name = "COP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "COP_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "COP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_EXTREME(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    calib_size=0.2,
    gpd_threshold_quantile=0.95,
    n_bootstrap=30,
    random_state=123,
):
    """
    Extreme Conformal Prediction (upper-tail, single-sided)
    ✔ Same structure as NexCP & OnlineCP
    ✔ No return values
    ✔ Excel-safe
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # Train / calibration split
    # =========================================================
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train,
        y_train,
        test_size=calib_size,
        random_state=random_state,
        shuffle=True,
    )

    # =========================================================
    # Train model
    # =========================================================
    model = model_class(**model_params)
    model.fit(X_fit, y_fit)

    y_pred_test = model.predict(X_test).ravel()
    y_pred_calib = model.predict(X_calib).ravel()

    # =========================================================
    # Nonconformity scores (upper tail only)
    # =========================================================
    # FIX N9: Extreme CP as in the official ExtremeConformal code (conformalize_GPD_max):
    # signed scores y - y_hat on ALL calibration points, thresholds 0.80..0.99 (capped so >= 10 scores
    # exceed them), official GPD quantile formula, maximum over thresholds; ONE-SIDED upper bound.
    # Was: only positive scores kept, (1/alpha)^xi in the formula (missing the threshold share), a hard
    # error with < 30 exceedances, and a two-sided band reported for a one-sided method.
    scores = np.asarray(y_calib).ravel() - y_pred_calib
    q_extreme, gpd_table = _conformalize_gpd_max(scores, 1 - alpha, min_obs_gpd=10)
    extreme_note = ""
    if not np.isfinite(q_extreme):
        q_extreme = weighted_quantile(scores, np.ones(len(scores)), 1 - alpha)
        extreme_note = "GPD fit failed at every threshold; empirical conformal quantile used"
    best_row = gpd_table.loc[gpd_table["q_threshold"].idxmax()] if gpd_table["q_threshold"].notna().any() else None
    u = float(best_row["threshold_u"]) if best_row is not None else np.nan
    n_exc_used = int(best_row["n_exceedances"]) if best_row is not None else 0
    print(gpd_table.to_string(index=False))

    # one-sided interval (-inf, y_hat + q]
    y_pred_test = np.asarray(y_pred_test).ravel()
    ymin = np.array([None] * len(y_pred_test), dtype=object)   # no lower bound (empty cell in Excel)
    ymax = y_pred_test + q_extreme

    # =========================================================
    # Metrics
    # =========================================================
    y_true = y_test.to_numpy().ravel()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred_test))
    coverage = np.mean(y_true <= ymax)
    width = q_extreme  # meaningful one-sided width

    print(f"\n===== EXTREME CONFORMAL PREDICTION ({model_name}) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Width    : {width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"q_extreme: {q_extreme:.3f}")
    print(f"u        : {u:.3f}")
    print(f"#Exceed. : {n_exc_used}")
    if extreme_note:
        print(f"Note     : {extreme_note}")

    # =========================================================
    # Summary table
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "EXTREME CP",
        "Coverage": coverage,
        "Width": width,
        "RMSE": rmse,
        "q_extreme": q_extreme,
        "threshold_u": u,
        "n_exceedances": n_exc_used,
        "Note": extreme_note,   # FIX N10
    }])

    # =========================================================
    # Values table
    # =========================================================
    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": np.full(len(y_pred_test), q_extreme),   # FIX N9: one-sided width = q
        "is_covered": (y_true <= ymax).astype(int),
        "residual": y_true - y_pred_test,
    })

    # =========================================================
    # Plot (SAVE ONLY)
    # =========================================================
    idx = np.arange(len(y_true))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(idx, y_true, s=10, alpha=0.7, label="True")
    ax.plot(idx, ymax, ls="--", lw=1, label="Upper bound (one-sided)")   # FIX N9
    ax.plot(idx, y_pred_test, color="red", lw=1, label="Prediction")

    ax.set_title(f"Extreme Conformal Prediction ({model_name})")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel(f"{TARGET_LABEL}")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # Excel export (append-safe, SAME FILE)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "EXTREMECP_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "EXTREMECP_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "EXTREMECP_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # FIX N9: GPD fit at every threshold level (the reported q is the maximum)
        sheet_name = "EXTREMECP_GPD_Thresholds"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1
        ws = wb.create_sheet(sheet_name)
        _tbl = gpd_table.astype(object).where(gpd_table.notna(), None)
        for r, row in enumerate(dataframe_to_rows(_tbl, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


In [18]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path



import os
try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("[Info] Install tqdm for progress bars: pip install tqdm")


# ═══════════════════════════════════════════════════════════
# AlphaNet Architecture  (FIXED)
# ═══════════════════════════════════════════════════════════
class AlphaNet(nn.Module):
    """
    Maps a feature vector (sum of LOO scores) → learned α̃.
    Output is clamped to (alpha_min, 1) so that the conformal
    denominator α̃·(n+1)−1 is always positive.
    """

    def __init__(self, input_dim, alpha_min=0.05):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.alpha_min = alpha_min  # lower bound for valid α̃
        self._initialize_near_one()

    def _initialize_near_one(self):
        """Bias output toward α̃ ≈ 1 at init (sigmoid(5) ≈ 0.993)."""
        with torch.no_grad():
            self.fc2.weight.data.normal_(0, 0.01)
            self.fc2.bias.data.fill_(5.0)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        raw = self.sigmoid(self.fc2(x)).squeeze(-1)
        # Remap from (0,1) → (alpha_min, 1) so denominator stays positive
        return self.alpha_min + (1.0 - self.alpha_min) * raw


def _interval_size(scores, alpha, eps=1e-8):
    """
    Conformal interval width = 2 * Σscores / (α̃·(n+1) − 1).
    Denominator is clamped to be strictly positive.
    """
    n = len(scores)
    denom = alpha * (n + 1) - 1
    denom = torch.clamp(denom, min=eps)   # ← KEY FIX: prevent negative/zero
    r = scores.sum() / denom
    return 2 * r


def _fmt_time(seconds):
    """Format seconds into human-readable string."""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}min"
    else:
        return f"{seconds/3600:.1f}hr"


# ═══════════════════════════════════════════════════════════
# Main function  (CORRECTED + PROGRESS TRACKING)
# ═══════════════════════════════════════════════════════════
def conformal_predictions_AlphaNet(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path=None,
    alpha=0.05,
    lambda_reg=20.0,
    num_epochs=50,
    batch_size=32,
    lr_alpha=1e-2,  # FIX A3 (was 1e-3)
    calib_fraction=0.5,
    num_runs=4,
    random_state=123,
):
    """
    Learned-α Conformal Prediction via AlphaNet.
    - FIXED: α̃ clamped to valid range (no negative intervals)
    - FIXED: LOO scores precomputed (massive speedup)
    - ADDED: tqdm progress bars + ETA estimates
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    t_start_total = time.time()

    # --- numpy-ify if pandas ---
    y_train_np = y_train.to_numpy().ravel() if hasattr(y_train, "to_numpy") else np.asarray(y_train).ravel()
    y_test_np  = y_test.to_numpy().ravel()  if hasattr(y_test,  "to_numpy") else np.asarray(y_test).ravel()
    X_train_np = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    X_test_np  = X_test.to_numpy()  if hasattr(X_test,  "to_numpy") else np.asarray(X_test)

    # =================================================================
    # 1.  Split training data → fit + calibration
    # =================================================================
    n_total = len(X_train_np)
    n_fit   = int(n_total * (1 - calib_fraction))
    perm    = np.random.permutation(n_total)

    fit_idx   = perm[:n_fit]
    calib_idx = perm[n_fit:]

    X_fit,   y_fit   = X_train_np[fit_idx],   y_train_np[fit_idx]
    X_calib, y_calib = X_train_np[calib_idx],  y_train_np[calib_idx]
    n_calib = len(X_calib)

    # Compute alpha_min: must satisfy α̃·(n+1) − 1 > 0  →  α̃ > 1/(n+1)
    alpha_min = 1.0 / (n_calib + 1) + 0.01   # small margin above threshold
    print(f"  [Info] n_calib={n_calib} | alpha_min={alpha_min:.4f}")

    # =================================================================
    # 2.  Train base regression model
    # =================================================================
    print(f"  [Step 2/9] Training base {model_name} model...")
    t0 = time.time()
    base_model = model_class(**model_params)
    base_model.fit(X_fit, y_fit)
    print(f"  [Step 2/9] Done in {_fmt_time(time.time()-t0)}")

    # =================================================================
    # 3.  PRECOMPUTE all LOO scores  (KEY SPEEDUP)
    # =================================================================
    print(f"  [Step 3/9] Precomputing {n_calib} LOO score vectors...")
    t0 = time.time()

    precomputed_scores = []
    precomputed_sums   = []

    iterator = tqdm(range(n_calib), desc="    LOO precompute", leave=False) if HAS_TQDM else range(n_calib)

    for i in iterator:
        loo_X = np.delete(X_calib, i, axis=0)
        loo_y = np.delete(y_calib, i, axis=0)
        scores_loo = np.abs(base_model.predict(loo_X).ravel() - loo_y)

        scores_tensor = torch.tensor(scores_loo, dtype=torch.float32).to(device)
        precomputed_scores.append(scores_tensor)
        precomputed_sums.append(scores_loo.sum())

    elapsed_precompute = time.time() - t0
    print(f"  [Step 3/9] Done in {_fmt_time(elapsed_precompute)}")

    # Build feature matrix (sum of LOO scores per sample)
    # FIX A2: divide the feature (sum of LOO scores) by its mean so the network input is O(1)
    # whatever the units of the target (was: raw sums, e.g. ~1e3, which saturate the network).
    feature_scale = float(np.mean(precomputed_sums)) or 1.0
    X_alpha = torch.tensor(
        (np.array(precomputed_sums) / feature_scale).reshape(-1, 1),
        dtype=torch.float32,
    ).to(device)

    # =================================================================
    # 4.  Train AlphaNet (multiple runs, keep best)
    # =================================================================
    # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
    # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
    # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
    # now matches the exact minimiser of the same loss for every lambda.
    num_epochs = max(num_epochs, int(np.ceil(2000 / np.ceil(n_calib / batch_size))))
    print(f"  [Step 4/9] Training AlphaNet: {num_runs} runs × {num_epochs} epochs")

    best_net   = None
    best_loss  = np.inf
    history    = {"losses": [], "sizes": [], "alphas": []}

    t0_train = time.time()

    for run in range(num_runs):
        np.random.seed(run)
        torch.manual_seed(run)

        alpha_net = AlphaNet(input_dim=1, alpha_min=alpha_min).to(device)
        optimizer = optim.Adam(alpha_net.parameters(), lr=lr_alpha)

        Dataset = TensorDataset(X_alpha, torch.arange(n_calib))
        loader  = DataLoader(Dataset, batch_size=batch_size, shuffle=True)

        run_losses, run_sizes, run_alphas = [], [], []

        epoch_iter = (
            tqdm(range(num_epochs), desc=f"    Run {run+1}/{num_runs}", leave=False)
            if HAS_TQDM else range(num_epochs)
        )

        for epoch in epoch_iter:
            t_epoch_start = time.time()
            epoch_loss = epoch_size = epoch_alpha = 0.0

            for x_batch, idx_batch in loader:
                x_batch = x_batch.to(device)
                alpha_pred = alpha_net(x_batch)

                batch_sizes = []
                for j, idx in enumerate(idx_batch):
                    # USE PRECOMPUTED scores instead of calling predict()
                    scores_t = precomputed_scores[idx.item()]
                    size_j = _interval_size(scores_t, alpha_pred[j])
                    batch_sizes.append(size_j)

                batch_sizes = torch.stack(batch_sizes)
                loss = (batch_sizes + lambda_reg * alpha_pred).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                epoch_size  += batch_sizes.mean().item()
                epoch_alpha += alpha_pred.mean().item()

            n_batches = len(loader)
            run_losses.append(epoch_loss  / n_batches)
            run_sizes.append(epoch_size   / n_batches)
            run_alphas.append(epoch_alpha / n_batches)

            epoch_time = time.time() - t_epoch_start

            if epoch % max(10, num_epochs // 10) == 0 or epoch == num_epochs - 1:  # FIX A3: ~10 log lines per run
                epochs_done = (run * num_epochs) + epoch + 1
                epochs_total = num_runs * num_epochs
                elapsed_train = time.time() - t0_train
                eta = (elapsed_train / epochs_done) * (epochs_total - epochs_done)

                print(
                    f"  λ={lambda_reg} | Run {run+1}/{num_runs} | "
                    f"Epoch {epoch}/{num_epochs} | "
                    f"Loss: {run_losses[-1]:.4f} | "
                    f"Size: {run_sizes[-1]:.4f} | "
                    f"α̃: {run_alphas[-1]:.4f} | "
                    f"⏱ {epoch_time:.2f}s/epoch | "
                    f"ETA: {_fmt_time(eta)}"
                )

        if run_losses[-1] < best_loss:
            best_loss = run_losses[-1]
            best_net  = alpha_net

        history["losses"].append(run_losses)
        history["sizes"].append(run_sizes)
        history["alphas"].append(run_alphas)

        print(f"  ✓ Run {run+1} complete | Final loss: {run_losses[-1]:.4f}")

    train_time = time.time() - t0_train
    print(f"  [Step 4/9] Training done in {_fmt_time(train_time)}")

    alpha_net = best_net
    alpha_net.eval()

    # =================================================================
    # 5.  Compute conformal intervals on the TEST set
    # =================================================================
    print(f"  [Step 5/9] Computing conformal intervals on test set...")

    scores_calib = np.abs(base_model.predict(X_calib).ravel() - y_calib)
    sum_scores   = scores_calib.sum()

    with torch.no_grad():
        feat_test = torch.tensor([sum_scores / feature_scale], dtype=torch.float32).to(device)  # FIX A2: same scale; FIX B11: .to(device) was inside this comment
        alpha_val = alpha_net(feat_test).item()

    denom = alpha_val * (n_calib + 1) - 1
    if denom <= 0:
        q = np.quantile(scores_calib, 1 - alpha)
        half_width = q
        print(f"  [Warning] α̃ too small (denom={denom:.4f}); fallback q={q:.4f}")
    else:
        half_width = sum_scores / denom

    y_pred_test = base_model.predict(X_test_np).ravel()
    ymin = y_pred_test - half_width
    ymax = y_pred_test + half_width

    # =================================================================
    # 6.  Metrics
    # =================================================================
    y_true_test = y_test_np
    is_covered  = ((y_true_test >= ymin) & (y_true_test <= ymax)).astype(int)
    coverage    = is_covered.mean()
    avg_width   = np.mean(ymax - ymin)
    rmse        = np.sqrt(mean_squared_error(y_true_test, y_pred_test))

    total_time = time.time() - t_start_total

    print(f"\n{'='*55}")
    print(f"  ALPHANET CONFORMAL ({model_name})")
    print(f"{'='*55}")
    print(f"  Learned α̃     : {alpha_val:.4f}")
    print(f"  Half-width (r) : {half_width:.4f}")
    print(f"  Coverage       : {coverage*100:.3f}%")
    print(f"  Avg Width      : {avg_width:.4f}")
    print(f"  RMSE           : {rmse:.4f}")
    print(f"  Total time     : {_fmt_time(total_time)}")
    print(f"    ├─ Precompute: {_fmt_time(elapsed_precompute)}")
    print(f"    └─ Training  : {_fmt_time(train_time)}")
    print(f"{'='*55}")

    # =================================================================
    # 7.  Summary & values DataFrames
    # =================================================================
    summary_df = pd.DataFrame([{
        "Strategy":       "AlphaNet Conformal",
        "Coverage":       coverage,
        "Width":          avg_width,
        "RMSE":           rmse,
        "Alpha_nominal (not used for the interval)":  alpha,   # FIX A1: the interval uses Alpha_learned
        "Target_coverage": 1 - alpha_val,
        "Miss_over_alpha": (1 - coverage) / alpha_val,
        "Alpha_learned":  alpha_val,
        "Lambda_reg":     lambda_reg,
        "Total_time_sec": round(total_time, 2),
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true_test)),
        "y_true":       y_true_test,
        "y_pred":       y_pred_test,
        "ymin":         ymin,
        "ymax":         ymax,
        "width":        ymax - ymin,
        "is_covered":   is_covered,
    })

    # =================================================================
    # 8.  Plotting (SAVE ONLY — NO SHOW)
    # =================================================================
    print(f"  [Step 8/9] Generating plots...")
    idx = np.arange(len(y_true_test))

    # --- PI plot ---
    fig_pi, ax_pi = plt.subplots(figsize=(10, 4))
    ax_pi.scatter(idx, y_true_test, color="blue", s=10, alpha=0.7, label="True")
    ax_pi.fill_between(idx, ymin, ymax, color="gray", alpha=0.5,
                       label=f"AlphaNet Interval (α̃={alpha_val:.3f})")
    ax_pi.plot(idx, y_pred_test, color="red", linewidth=1, label="Predicted mean")
    ax_pi.set_title(f"AlphaNet Conformal – {model_name}")
    ax_pi.set_xlabel("Sample Number")
    ax_pi.set_ylabel(f"{TARGET_LABEL}")
    ax_pi.legend(loc="upper left")
    plt.tight_layout()

    buf_pi = io.BytesIO()
    fig_pi.savefig(buf_pi, format="png")
    buf_pi.seek(0)
    plt.close(fig_pi)

    # --- Training curves ---
    arr_losses = np.array(history["losses"])
    arr_sizes  = np.array(history["sizes"])
    arr_alphas = np.array(history["alphas"])
    epochs_arr = np.arange(1, num_epochs + 1)

    fig_tc, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, ylabel, title in zip(
        axs,
        [arr_losses, arr_sizes, arr_alphas],
        ["Loss", "Mean Interval Size", r"Mean $\tilde{\alpha}$"],
        ["Training Loss", "Interval Size", r"Learned $\tilde{\alpha}$"],
    ):
        m = data.mean(axis=0)
        s = data.std(axis=0)
        ax.plot(epochs_arr, m, color="#0077bb")
        ax.fill_between(epochs_arr, m - s, m + s, color="#0077bb", alpha=0.25)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.5)
    plt.tight_layout()

    buf_tc = io.BytesIO()
    fig_tc.savefig(buf_tc, format="png")
    buf_tc.seek(0)
    plt.close(fig_tc)

    # =================================================================
    # 9.  Excel Export
    # =================================================================
    if excel_file_path:
        print(f"  [Step 9/9] Saving to Excel...")
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active)

        def _unique_sheet(wb, name):
            base, i = name, 1
            while name in wb.sheetnames:
                name = f"{base}_{i}"; i += 1
            return name

        sn = _unique_sheet(wb, "AlphaNet_Summary")
        ws = wb.create_sheet(sn, 0)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        sn = _unique_sheet(wb, "AlphaNet_Plot_PI")
        ws = wb.create_sheet(sn)
        ws.add_image(Image(buf_pi), "A1")

        sn = _unique_sheet(wb, "AlphaNet_Plot_Train")
        ws = wb.create_sheet(sn)
        ws.add_image(Image(buf_tc), "A1")

        sn = _unique_sheet(wb, "AlphaNet_Values")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))
        print(f"  [Step 9/9] Saved!")

    print(f"\n  ✅ AlphaNet Conformal complete in {_fmt_time(total_time)}")
    return summary_df

# **Tuned Parameters**

In [19]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/water-quality-ewqi-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Conformal Predictions with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

In [21]:
conformal_predictions_NEXCP(
    model_class=LGBMRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full LightGBM: 5/47 test points, 0.2 min


  NexCP-Full LightGBM: 10/47 test points, 0.4 min


  NexCP-Full LightGBM: 15/47 test points, 0.7 min


  NexCP-Full LightGBM: 20/47 test points, 0.9 min


  NexCP-Full LightGBM: 25/47 test points, 1.1 min


  NexCP-Full LightGBM: 30/47 test points, 1.3 min


... (2 outputs left out)

  NexCP-Full LightGBM: 45/47 test points, 2.0 min


  NexCP-Full LightGBM: 47/47 test points, 2.1 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 93.617% | Width: 83.739
NexCP Full - Coverage: 97.872% | Width: 128.791
NexCP Jackknife+ aB - Coverage: 95.745% | Width: 77.591
NexCP CV+ - Coverage: 95.745% | Width: 90.018


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.936170,83.739366,18.136090,83.739366,
1,NexCP Full,0.978723,128.790636,18.136090,128.790636,32 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.957447,77.591157,20.423533,77.591157,
3,NexCP CV+,0.957447,90.017655,23.074100,90.017655,


In [22]:
conformal_predictions_online_super(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage      Width       RMSE        CWC Note
0          Online Split  0.936170  83.739366  18.136090  83.739366     
1            Online CV+  0.936170  82.554964  18.136090  82.554964     
2  Online Jackknife+ aB  0.936170  77.290953  20.486318  77.290953     
3                 SAOCP  0.936170  84.881099  18.136090  84.881099     
4                SF-OGD  0.914894  76.978942  18.136090  76.978942     
5                  FACI  0.914894  70.877839  18.136090  70.877839     
6                   ACI  0.914894  71.684693  18.136090  71.684693     
7            MFCS Split  0.936170  83.739366  18.136090  83.739366     
8             MFCS Full  0.936170  87.622229  19.352375  87.622229     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.936170,83.739366,18.136090,83.739366,
1,Online CV+,0.936170,82.554964,18.136090,82.554964,
2,Online Jackknife+ aB,0.936170,77.290953,20.486318,77.290953,
3,SAOCP,0.936170,84.881099,18.136090,84.881099,
4,SF-OGD,0.914894,76.978942,18.136090,76.978942,
5,FACI,0.914894,70.877839,18.136090,70.877839,
6,ACI,0.914894,71.684693,18.136090,71.684693,
7,MFCS Split,0.936170,83.739366,18.136090,83.739366,
8,MFCS Full,0.936170,87.622229,19.352375,87.622229,


In [23]:
conformal_predictions_COP(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (LightGBM) =====
Coverage : 91.489%
Avg Width: 72.122
RMSE     : 18.136
Final q  : 34.801


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.914894,72.122231,18.13609,0.1,0.05,0.5


In [24]:
conformal_predictions_EXTREME(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances     sigma       xi  q_threshold
        0.333333    -6.529043             10 11.064753 0.299974     21.74949

===== EXTREME CONFORMAL PREDICTION (LightGBM) =====
Coverage : 78.723%
Width    : 21.749
RMSE     : 26.389
q_extreme: 21.749
u        : -6.529
#Exceed. : 10


In [25]:
conformal_predictions_AlphaNet(
    model_class=LGBMRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/LightGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base LightGBM model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 24 LOO score vectors...


    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 2000 epochs


    Run 1/4:   0%|          | 0/2000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/2000 | Loss: 74.3434 | Size: 54.4690 | α̃: 0.9937 | ⏱ 0.19s/epoch | ETA: 34.2min


  λ=20.0 | Run 1/4 | Epoch 200/2000 | Loss: 74.1123 | Size: 54.1124 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 48s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 74.1122 | Size: 54.1122 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 74.1122
  [Step 4/9] Training done in 40s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (LightGBM)
  Learned α̃     : 1.0000
  Half-width (r) : 27.0561
  Coverage       : 74.468%
  Avg Width      : 54.1122
  RMSE           : 26.0834
  Total time     : 40s
    ├─ Precompute: 0s
    └─ Training  : 40s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 40s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.744681,54.112175,26.083406,0.1,0.0,0.255319,1.0,20.0,40.31


# **Conformal Predictions with XGBoost**

In [26]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

In [27]:
conformal_predictions_NEXCP(
    model_class=XGBRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full XGBoost: 5/47 test points, 0.4 min


  NexCP-Full XGBoost: 10/47 test points, 0.7 min


  NexCP-Full XGBoost: 15/47 test points, 1.1 min


  NexCP-Full XGBoost: 20/47 test points, 1.5 min


  NexCP-Full XGBoost: 25/47 test points, 1.9 min


  NexCP-Full XGBoost: 30/47 test points, 2.3 min


... (2 outputs left out)

  NexCP-Full XGBoost: 45/47 test points, 3.4 min


  NexCP-Full XGBoost: 47/47 test points, 3.6 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 95.745% | Width: 81.281
NexCP Full - Coverage: 97.872% | Width: 149.362
NexCP Jackknife+ aB - Coverage: 95.745% | Width: 88.735
NexCP CV+ - Coverage: 97.872% | Width: 97.920


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.957447,81.280907,20.335771,81.280907,
1,NexCP Full,0.978723,149.362061,20.335771,149.362061,40 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.957447,88.734557,22.946151,88.734557,
3,NexCP CV+,0.978723,97.919705,22.394875,97.919705,


In [28]:
conformal_predictions_online_super(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage      Width       RMSE        CWC Note
0          Online Split  0.957447  81.280907  20.335771  81.280907     
1            Online CV+  0.978723  86.305922  20.335771  86.305922     
2  Online Jackknife+ aB  0.936170  81.402906  22.636952  81.402906     
3                 SAOCP  0.936170  86.078453  20.335771  86.078453     
4                SF-OGD  0.914894  93.177401  20.335771  93.177401     
5                  FACI  0.936170  74.373211  20.335771  74.373211     
6                   ACI  0.936170  75.861483  20.335771  75.861483     
7            MFCS Split  0.957447  81.280907  20.335771  81.280907     
8             MFCS Full  0.936170  81.944792  19.388849  81.944792     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.957447,81.280907,20.335771,81.280907,
1,Online CV+,0.978723,86.305922,20.335771,86.305922,
2,Online Jackknife+ aB,0.936170,81.402906,22.636952,81.402906,
3,SAOCP,0.936170,86.078453,20.335771,86.078453,
4,SF-OGD,0.914894,93.177401,20.335771,93.177401,
5,FACI,0.936170,74.373211,20.335771,74.373211,
6,ACI,0.936170,75.861483,20.335771,75.861483,
7,MFCS Split,0.957447,81.280907,20.335771,81.280907,
8,MFCS Full,0.936170,81.944792,19.388849,81.944792,


In [29]:
conformal_predictions_COP(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (XGBoost) =====
Coverage : 91.489%
Avg Width: 73.725
RMSE     : 20.336
Final q  : 35.882


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.914894,73.724973,20.335771,0.1,0.05,0.5


In [30]:
conformal_predictions_EXTREME(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma       xi  q_threshold
        0.333333    -7.285879             10 5.682616 0.709815    15.485324

===== EXTREME CONFORMAL PREDICTION (XGBoost) =====
Coverage : 65.957%
Width    : 15.485
RMSE     : 24.319
q_extreme: 15.485
u        : -7.286
#Exceed. : 10


In [31]:
conformal_predictions_AlphaNet(
    model_class=XGBRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/XGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base XGBoost model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 24 LOO score vectors...


    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 2000 epochs


    Run 1/4:   0%|          | 0/2000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/2000 | Loss: 78.5353 | Size: 58.6608 | α̃: 0.9937 | ⏱ 0.01s/epoch | ETA: 3.3min


  λ=20.0 | Run 1/4 | Epoch 200/2000 | Loss: 78.2768 | Size: 58.2769 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 41s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 78.2766 | Size: 58.2766 | α̃: 1.0000 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 78.2766
  [Step 4/9] Training done in 44s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (XGBoost)
  Learned α̃     : 1.0000
  Half-width (r) : 29.1383
  Coverage       : 65.957%
  Avg Width      : 58.2766
  RMSE           : 28.8208
  Total time     : 44s
    ├─ Precompute: 0s
    └─ Training  : 44s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 44s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.659574,58.27657,28.820829,0.1,0.0,0.340426,1.0,20.0,43.75


# **Conformal Predictions with GPBoost**

In [32]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

In [33]:
conformal_predictions_NEXCP(
    model_class=GPBoostRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full GPBoost: 5/47 test points, 0.3 min


  NexCP-Full GPBoost: 10/47 test points, 0.5 min


  NexCP-Full GPBoost: 15/47 test points, 0.7 min


  NexCP-Full GPBoost: 20/47 test points, 1.0 min


  NexCP-Full GPBoost: 25/47 test points, 1.2 min


  NexCP-Full GPBoost: 30/47 test points, 1.4 min


... (2 outputs left out)

  NexCP-Full GPBoost: 45/47 test points, 2.1 min


  NexCP-Full GPBoost: 47/47 test points, 2.2 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 93.617% | Width: 80.475
NexCP Full - Coverage: 97.872% | Width: 131.947
NexCP Jackknife+ aB - Coverage: 93.617% | Width: 77.698
NexCP CV+ - Coverage: 93.617% | Width: 88.753


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.936170,80.475292,17.946718,80.475292,
1,NexCP Full,0.978723,131.947331,17.946718,131.947331,37 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.936170,77.698387,20.405864,77.698387,
3,NexCP CV+,0.936170,88.752867,22.877651,88.752867,


In [34]:
conformal_predictions_online_super(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage      Width       RMSE        CWC Note
0          Online Split  0.936170  80.475292  17.946718  80.475292     
1            Online CV+  0.936170  82.945669  17.946718  82.945669     
2  Online Jackknife+ aB  0.914894  77.244551  20.476329  77.244551     
3                 SAOCP  0.936170  78.719001  17.946718  78.719001     
4                SF-OGD  0.914894  79.278650  17.946718  79.278650     
5                  FACI  0.914894  70.223847  17.946718  70.223847     
6                   ACI  0.914894  71.461644  17.946718  71.461644     
7            MFCS Split  0.936170  80.475292  17.946718  80.475292     
8             MFCS Full  0.936170  87.626349  19.332590  87.626349     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.936170,80.475292,17.946718,80.475292,
1,Online CV+,0.936170,82.945669,17.946718,82.945669,
2,Online Jackknife+ aB,0.914894,77.244551,20.476329,77.244551,
3,SAOCP,0.936170,78.719001,17.946718,78.719001,
4,SF-OGD,0.914894,79.278650,17.946718,79.278650,
5,FACI,0.914894,70.223847,17.946718,70.223847,
6,ACI,0.914894,71.461644,17.946718,71.461644,
7,MFCS Split,0.936170,80.475292,17.946718,80.475292,
8,MFCS Full,0.936170,87.626349,19.332590,87.626349,


In [35]:
conformal_predictions_COP(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (GPBoost) =====
Coverage : 91.489%
Avg Width: 70.753
RMSE     : 17.947
Final q  : 34.043


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.914894,70.753047,17.946718,0.1,0.05,0.5


In [36]:
conformal_predictions_EXTREME(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances    sigma       xi  q_threshold
        0.333333    -5.378284             10 8.733712 0.413009    19.768782

===== EXTREME CONFORMAL PREDICTION (GPBoost) =====
Coverage : 74.468%
Width    : 19.769
RMSE     : 26.444
q_extreme: 19.769
u        : -5.378
#Exceed. : 10


In [37]:
conformal_predictions_AlphaNet(
    model_class=GPBoostRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GPBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/GPBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base GPBoost model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 24 LOO score vectors...


    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 2000 epochs


    Run 1/4:   0%|          | 0/2000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/2000 | Loss: 74.8487 | Size: 54.9743 | α̃: 0.9937 | ⏱ 0.01s/epoch | ETA: 3.3min


  λ=20.0 | Run 1/4 | Epoch 200/2000 | Loss: 74.6144 | Size: 54.6145 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 39s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 74.6142 | Size: 54.6142 | α̃: 1.0000 | ⏱ 0.01s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 74.6142
  [Step 4/9] Training done in 41s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (GPBoost)
  Learned α̃     : 1.0000
  Half-width (r) : 27.3071
  Coverage       : 74.468%
  Avg Width      : 54.6142
  RMSE           : 25.9015
  Total time     : 41s
    ├─ Precompute: 0s
    └─ Training  : 41s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 41s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.744681,54.614196,25.901481,0.1,0.0,0.255319,1.0,20.0,40.82


# **Conformal Predictions with NGBoost**

In [38]:
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

In [39]:
conformal_predictions_NEXCP(
    model_class=NGBRegressor,
    best_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full NGBoost: 5/47 test points, 18.1 min


  NexCP-Full NGBoost: 10/47 test points, 36.4 min


  NexCP-Full NGBoost: 15/47 test points, 54.6 min


  NexCP-Full NGBoost: 20/47 test points, 72.7 min


  NexCP-Full NGBoost: 25/47 test points, 90.7 min


  NexCP-Full NGBoost: 30/47 test points, 108.5 min


... (2 outputs left out)

  NexCP-Full NGBoost: 45/47 test points, 161.4 min


  NexCP-Full NGBoost: 47/47 test points, 168.4 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 97.872% | Width: 130.710
NexCP Full - Coverage: 100.000% | Width: 253.701
NexCP Jackknife+ aB - Coverage: 95.745% | Width: 101.033
NexCP CV+ - Coverage: 100.000% | Width: 133.614


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.978723,130.709858,23.384239,130.709858,
1,NexCP Full,1.000000,253.700854,23.384239,253.700854,42 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.957447,101.033015,20.227825,101.033015,
3,NexCP CV+,1.000000,133.613775,18.577339,133.613775,


In [40]:
conformal_predictions_online_super(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage       Width       RMSE         CWC Note
0          Online Split  0.978723  130.709858  23.384239  130.709858     
1            Online CV+  1.000000  118.770321  23.384239  118.770321     
2  Online Jackknife+ aB  0.936170   88.850121  20.108095   88.850121     
3                 SAOCP  0.872340  101.033857  23.384239  113.931796     
4                SF-OGD  0.914894  128.927924  23.384239  128.927924     
5                  FACI  0.914894  104.714296  23.384239  104.714296     
6                   ACI  0.936170  108.080821  23.384239  108.080821     
7            MFCS Split  0.978723  130.709858  23.384239  130.709858     
8             MFCS Full  0.978723   92.283712  14.076046   92.283712     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.978723,130.709858,23.384239,130.709858,
1,Online CV+,1.000000,118.770321,23.384239,118.770321,
2,Online Jackknife+ aB,0.936170,88.850121,20.108095,88.850121,
3,SAOCP,0.872340,101.033857,23.384239,113.931796,
4,SF-OGD,0.914894,128.927924,23.384239,128.927924,
5,FACI,0.914894,104.714296,23.384239,104.714296,
6,ACI,0.936170,108.080821,23.384239,108.080821,
7,MFCS Split,0.978723,130.709858,23.384239,130.709858,
8,MFCS Full,0.978723,92.283712,14.076046,92.283712,


In [41]:
conformal_predictions_COP(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (NGBoost) =====
Coverage : 93.617%
Avg Width: 108.889
RMSE     : 23.384
Final q  : 51.479


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.93617,108.888995,23.384239,0.1,0.05,0.5


In [42]:
conformal_predictions_EXTREME(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances     sigma       xi  q_threshold
        0.333333   -12.966444             10 13.927862 0.620534    37.430643

===== EXTREME CONFORMAL PREDICTION (NGBoost) =====
Coverage : 93.617%
Width    : 37.431
RMSE     : 19.876
q_extreme: 37.431
u        : -12.966
#Exceed. : 10


In [43]:
conformal_predictions_AlphaNet(
    model_class=NGBRegressor,
    model_params={**best_params, "Dist": Normal},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="NGBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/NGBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base NGBoost model...


  [Step 2/9] Done in 5s
  [Step 3/9] Precomputing 24 LOO score vectors...


    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

  [Step 3/9] Done in 5s
  [Step 4/9] Training AlphaNet: 4 runs × 2000 epochs


    Run 1/4:   0%|          | 0/2000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/2000 | Loss: 63.6094 | Size: 43.7350 | α̃: 0.9937 | ⏱ 0.03s/epoch | ETA: 7.6min


... (46 outputs left out)


  ALPHANET CONFORMAL (NGBoost)
  Learned α̃     : 1.0000
  Half-width (r) : 21.7242
  Coverage       : 70.213%
  Avg Width      : 43.4485
  RMSE           : 21.7006
  Total time     : 51s
    ├─ Precompute: 5s
    └─ Training  : 41s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 51s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.702128,43.448494,21.700597,0.1,0.0,0.297872,1.0,20.0,51.21


# **Conformal Predictions with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

In [45]:
conformal_predictions_NEXCP(
    model_class=GradientBoostingRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full GradientBoosting: 5/47 test points, 0.3 min


  NexCP-Full GradientBoosting: 10/47 test points, 0.7 min


  NexCP-Full GradientBoosting: 15/47 test points, 1.0 min


  NexCP-Full GradientBoosting: 20/47 test points, 1.3 min


  NexCP-Full GradientBoosting: 25/47 test points, 1.7 min


  NexCP-Full GradientBoosting: 30/47 test points, 2.0 min


... (2 outputs left out)

  NexCP-Full GradientBoosting: 45/47 test points, 3.0 min


  NexCP-Full GradientBoosting: 47/47 test points, 3.1 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 97.872% | Width: 112.379
NexCP Full - Coverage: 93.617% | Width: 165.714
NexCP Jackknife+ aB - Coverage: 95.745% | Width: 86.200
NexCP CV+ - Coverage: 95.745% | Width: 108.483


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.978723,112.378910,22.274845,112.378910,
1,NexCP Full,0.936170,165.713755,22.274845,165.713755,19 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.957447,86.200292,22.553917,86.200292,
3,NexCP CV+,0.957447,108.483490,22.920246,108.483490,


In [46]:
conformal_predictions_online_super(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage       Width       RMSE         CWC Note
0          Online Split  0.978723  112.378910  22.274845  112.378910     
1            Online CV+  0.957447   95.926467  22.274845   95.926467     
2  Online Jackknife+ aB  0.957447   81.814988  22.219015   81.814988     
3                 SAOCP  0.914894   94.136979  22.274845   94.136979     
4                SF-OGD  0.914894   99.732938  22.274845   99.732938     
5                  FACI  0.914894   76.559732  22.274845   76.559732     
6                   ACI  0.914894   79.448395  22.274845   79.448395     
7            MFCS Split  0.978723  112.378910  22.274845  112.378910     
8             MFCS Full  1.000000   86.635486  17.790291   86.635486     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.978723,112.378910,22.274845,112.378910,
1,Online CV+,0.957447,95.926467,22.274845,95.926467,
2,Online Jackknife+ aB,0.957447,81.814988,22.219015,81.814988,
3,SAOCP,0.914894,94.136979,22.274845,94.136979,
4,SF-OGD,0.914894,99.732938,22.274845,99.732938,
5,FACI,0.914894,76.559732,22.274845,76.559732,
6,ACI,0.914894,79.448395,22.274845,79.448395,
7,MFCS Split,0.978723,112.378910,22.274845,112.378910,
8,MFCS Full,1.000000,86.635486,17.790291,86.635486,


In [47]:
conformal_predictions_COP(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (GradientBoosting) =====
Coverage : 91.489%
Avg Width: 77.705
RMSE     : 22.275
Final q  : 37.576


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.914894,77.704704,22.274845,0.1,0.05,0.5


In [48]:
conformal_predictions_EXTREME(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances     sigma       xi  q_threshold
        0.333333   -22.685866             10 23.057606 0.138506    27.342422

===== EXTREME CONFORMAL PREDICTION (GradientBoosting) =====
Coverage : 87.234%
Width    : 27.342
RMSE     : 24.821
q_extreme: 27.342
u        : -22.686
#Exceed. : 10


In [49]:
conformal_predictions_AlphaNet(
    model_class=GradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="GradientBoosting",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/Gradient Boosting.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base GradientBoosting model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 24 LOO score vectors...


    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 2000 epochs


    Run 1/4:   0%|          | 0/2000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/2000 | Loss: 82.2108 | Size: 62.3364 | α̃: 0.9937 | ⏱ 0.01s/epoch | ETA: 3.1min


  λ=20.0 | Run 1/4 | Epoch 200/2000 | Loss: 81.9282 | Size: 61.9283 | α̃: 1.0000 | ⏱ 0.01s/epoch | ETA: 42s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 81.9280 | Size: 61.9280 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 81.9280
  [Step 4/9] Training done in 42s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (GradientBoosting)
  Learned α̃     : 1.0000
  Half-width (r) : 30.9640
  Coverage       : 70.213%
  Avg Width      : 61.9280
  RMSE           : 29.8619
  Total time     : 42s
    ├─ Precompute: 0s
    └─ Training  : 42s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 42s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.702128,61.928031,29.861877,0.1,0.0,0.297872,1.0,20.0,42.05


# **Conformal Predictions with CatBoost**

In [50]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

In [51]:
conformal_predictions_NEXCP(
    model_class=CatBoostRegressor,
    best_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full CatBoost: 5/47 test points, 0.7 min


  NexCP-Full CatBoost: 10/47 test points, 1.4 min


  NexCP-Full CatBoost: 15/47 test points, 2.1 min


  NexCP-Full CatBoost: 20/47 test points, 2.9 min


  NexCP-Full CatBoost: 25/47 test points, 3.6 min


  NexCP-Full CatBoost: 30/47 test points, 4.3 min


... (2 outputs left out)

  NexCP-Full CatBoost: 45/47 test points, 6.4 min


  NexCP-Full CatBoost: 47/47 test points, 6.7 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 97.872% | Width: 117.411
NexCP Full - Coverage: 89.362% | Width: 192.201
NexCP Jackknife+ aB - Coverage: 93.617% | Width: 86.996
NexCP CV+ - Coverage: 97.872% | Width: 114.285


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.978723,117.411105,19.407678,117.411105,
1,NexCP Full,0.893617,192.201418,19.407678,456.662157,37 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.936170,86.996115,25.093631,86.996115,
3,NexCP CV+,0.978723,114.285494,21.632813,114.285494,


In [52]:
conformal_predictions_online_super(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage       Width       RMSE         CWC Note
0          Online Split  0.978723  117.411105  19.407678  117.411105     
1            Online CV+  0.957447   98.288169  19.407678   98.288169     
2  Online Jackknife+ aB  0.893617   86.257865  25.046094   95.434234     
3                 SAOCP  0.914894   89.137116  19.407678   89.137116     
4                SF-OGD  0.914894   99.688227  19.407678   99.688227     
5                  FACI  0.936170   78.270426  19.407678   78.270426     
6                   ACI  0.936170   82.189793  19.407678   82.189793     
7            MFCS Split  0.978723  117.411105  19.407678  117.411105     
8             MFCS Full  0.893617   78.843310  19.541200   87.230896     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.978723,117.411105,19.407678,117.411105,
1,Online CV+,0.957447,98.288169,19.407678,98.288169,
2,Online Jackknife+ aB,0.893617,86.257865,25.046094,95.434234,
3,SAOCP,0.914894,89.137116,19.407678,89.137116,
4,SF-OGD,0.914894,99.688227,19.407678,99.688227,
5,FACI,0.936170,78.270426,19.407678,78.270426,
6,ACI,0.936170,82.189793,19.407678,82.189793,
7,MFCS Split,0.978723,117.411105,19.407678,117.411105,
8,MFCS Full,0.893617,78.843310,19.541200,87.230896,


In [53]:
conformal_predictions_COP(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (CatBoost) =====
Coverage : 93.617%
Avg Width: 81.006
RMSE     : 19.408
Final q  : 38.307


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.93617,81.006416,19.407678,0.1,0.05,0.5


In [54]:
conformal_predictions_EXTREME(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances     sigma        xi  q_threshold
        0.333333   -11.333018             10 32.907075 -0.013183    50.321471

===== EXTREME CONFORMAL PREDICTION (CatBoost) =====
Coverage : 95.745%
Width    : 50.321
RMSE     : 24.036
q_extreme: 50.321
u        : -11.333
#Exceed. : 10


In [55]:
conformal_predictions_AlphaNet(
    model_class=CatBoostRegressor,
    model_params={**best_params, "verbose": False},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="CatBoost",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/CatBoost.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base CatBoost model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 24 LOO score vectors...


    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 2000 epochs


    Run 1/4:   0%|          | 0/2000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/2000 | Loss: 55.2757 | Size: 35.4012 | α̃: 0.9937 | ⏱ 0.03s/epoch | ETA: 6.9min


  λ=20.0 | Run 1/4 | Epoch 200/2000 | Loss: 55.1694 | Size: 35.1695 | α̃: 1.0000 | ⏱ 0.01s/epoch | ETA: 43s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 55.1693 | Size: 35.1693 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 55.1693
  [Step 4/9] Training done in 43s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (CatBoost)
  Learned α̃     : 1.0000
  Half-width (r) : 17.5847
  Coverage       : 68.085%
  Avg Width      : 35.1693
  RMSE           : 27.2524
  Total time     : 43s
    ├─ Precompute: 0s
    └─ Training  : 43s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 43s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.680851,35.169325,27.252402,0.1,0.0,0.319149,1.0,20.0,42.96


# **Conformal Predictions with HistGradientBoosting**

In [56]:
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

In [57]:
conformal_predictions_NEXCP(
    model_class=HistGradientBoostingRegressor,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


  NexCP-Full HGBR: 5/47 test points, 0.4 min


  NexCP-Full HGBR: 10/47 test points, 0.8 min


  NexCP-Full HGBR: 15/47 test points, 1.2 min


  NexCP-Full HGBR: 20/47 test points, 1.6 min


  NexCP-Full HGBR: 25/47 test points, 2.0 min


  NexCP-Full HGBR: 30/47 test points, 2.4 min


... (2 outputs left out)

  NexCP-Full HGBR: 45/47 test points, 3.6 min


  NexCP-Full HGBR: 47/47 test points, 3.7 min



===== NEXCP RESULTS =====
NexCP Split - Coverage: 100.000% | Width: 112.758
NexCP Full - Coverage: 95.745% | Width: 160.495
NexCP Jackknife+ aB - Coverage: 93.617% | Width: 91.264
NexCP CV+ - Coverage: 100.000% | Width: 124.388


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,1.000000,112.758082,18.719715,112.758082,
1,NexCP Full,0.957447,160.495438,18.719715,160.495438,16 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.936170,91.264321,21.092195,91.264321,
3,NexCP CV+,1.000000,124.387766,19.439032,124.387766,


In [58]:
conformal_predictions_online_super(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== ONLINE CP RESULTS =====
               Strategy  Coverage       Width       RMSE         CWC Note
0          Online Split  1.000000  112.758082  18.719715  112.758082     
1            Online CV+  0.957447  105.612982  18.719715  105.612982     
2  Online Jackknife+ aB  0.957447   88.231116  21.047950   88.231116     
3                 SAOCP  0.936170   83.557492  18.719715   83.557492     
4                SF-OGD  0.914894   86.381354  18.719715   86.381354     
5                  FACI  0.978723   78.194764  18.719715   78.194764     
6                   ACI  0.978723   89.824766  18.719715   89.824766     
7            MFCS Split  1.000000  112.758082  18.719715  112.758082     
8             MFCS Full  0.957447   90.674193  20.061251   90.674193     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,1.000000,112.758082,18.719715,112.758082,
1,Online CV+,0.957447,105.612982,18.719715,105.612982,
2,Online Jackknife+ aB,0.957447,88.231116,21.047950,88.231116,
3,SAOCP,0.936170,83.557492,18.719715,83.557492,
4,SF-OGD,0.914894,86.381354,18.719715,86.381354,
5,FACI,0.978723,78.194764,18.719715,78.194764,
6,ACI,0.978723,89.824766,18.719715,89.824766,
7,MFCS Split,1.000000,112.758082,18.719715,112.758082,
8,MFCS Full,0.957447,90.674193,20.061251,90.674193,


In [59]:
conformal_predictions_COP(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)



===== CONFORMAL OPTIMISTIC PREDICTION (HGBR) =====
Coverage : 95.745%
Avg Width: 104.545
RMSE     : 18.720
Final q  : 49.845


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.957447,104.545443,18.719715,0.1,0.05,0.5


In [60]:
conformal_predictions_EXTREME(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)


 threshold_level  threshold_u  n_exceedances     sigma       xi  q_threshold
        0.333333   -15.603072             10 12.461838 0.447243    21.625008

===== EXTREME CONFORMAL PREDICTION (HGBR) =====
Coverage : 78.723%
Width    : 21.625
RMSE     : 25.986
q_extreme: 21.625
u        : -15.603
#Exceed. : 10


In [61]:
conformal_predictions_AlphaNet(
    model_class=HistGradientBoostingRegressor,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="HGBR",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/HGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base HGBR model...
  [Step 2/9] Done in 0s
  [Step 3/9] Precomputing 24 LOO score vectors...


    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

  [Step 3/9] Done in 0s
  [Step 4/9] Training AlphaNet: 4 runs × 2000 epochs


    Run 1/4:   0%|          | 0/2000 [00:00<?, ?it/s]

  λ=20.0 | Run 1/4 | Epoch 0/2000 | Loss: 75.9103 | Size: 56.0359 | α̃: 0.9937 | ⏱ 0.01s/epoch | ETA: 3.6min


  λ=20.0 | Run 1/4 | Epoch 200/2000 | Loss: 75.6690 | Size: 55.6691 | α̃: 1.0000 | ⏱ 0.01s/epoch | ETA: 40s


... (44 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 75.6688 | Size: 55.6688 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 75.6688
  [Step 4/9] Training done in 41s
  [Step 5/9] Computing conformal intervals on test set...

  ALPHANET CONFORMAL (HGBR)
  Learned α̃     : 1.0000
  Half-width (r) : 27.8344
  Coverage       : 76.596%
  Avg Width      : 55.6688
  RMSE           : 29.6376
  Total time     : 41s
    ├─ Precompute: 0s
    └─ Training  : 41s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet Conformal complete in 41s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.765957,55.668808,29.637644,0.1,0.0,0.234043,1.0,20.0,41.23


# **Conformal Predictions with PGBM**

In [62]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [63]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual str to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/300, Train metric: 55.0218
Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31.7545
Estimator 

Estimator 28/300, Train metric: 29.1221
Estimator 29/300, Train metric: 28.3471
Estimator 30/300, Train metric: 27.8655
Estimator 31/300, Train metric: 27.3146
Estimator 32/300, Train metric: 26.7418
Estimator 33/300, Train metric: 26.1088
Estimator 34/300, Train metric: 25.2632
Estimator 35/300, Train metric: 24.6503
Estimator 36/300, Train metric: 24.2793
Estimator 37/300, Train metric: 23.8609
Estimator 38/300, Train metric: 23.3188
Estimator 39/300, Train metric: 22.8892
Estimator 40/300, Train metric: 22.4073
Estimator 41/300, Train metric: 21.8657
Estimator 42/300, Train metric: 21.3036
Estimator 43/300, Train metric: 21.0520
Estimator 44/300, Train metric: 20.6351
Estimator 45/300, Train metric: 20.4684
Estimator 46/300, Train metric: 20.1000
Estimator 47/300, Train metric: 19.8026
Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178


Estimator 60/300, Train metric: 16.9987
Estimator 61/300, Train metric: 16.7382
Estimator 62/300, Train metric: 16.5740
Estimator 63/300, Train metric: 16.3673
Estimator 64/300, Train metric: 16.3097
Estimator 65/300, Train metric: 16.0251
Estimator 66/300, Train metric: 15.7774
Estimator 67/300, Train metric: 15.7003
Estimator 68/300, Train metric: 15.4424
Estimator 69/300, Train metric: 15.2547
Estimator 70/300, Train metric: 15.1427
Estimator 71/300, Train metric: 15.0232
Estimator 72/300, Train metric: 14.7207
Estimator 73/300, Train metric: 14.5002
Estimator 74/300, Train metric: 14.4790
Estimator 75/300, Train metric: 14.3939
Estimator 76/300, Train metric: 14.2124
Estimator 77/300, Train metric: 14.1670
Estimator 78/300, Train metric: 14.0933
Estimator 79/300, Train metric: 13.9494
Estimator 80/300, Train metric: 13.9759
Estimator 81/300, Train metric: 13.8087
Estimator 82/300, Train metric: 13.6064
Estimator 83/300, Train metric: 13.4929
Estimator 84/300, Train metric: 13.3332


Estimator 100/300, Train metric: 11.7421
Estimator 101/300, Train metric: 11.6483
Estimator 102/300, Train metric: 11.4578
Estimator 103/300, Train metric: 11.3104
Estimator 104/300, Train metric: 11.1709
Estimator 105/300, Train metric: 11.1320
Estimator 106/300, Train metric: 11.1323
Estimator 107/300, Train metric: 11.1133
Estimator 108/300, Train metric: 11.0399
Estimator 109/300, Train metric: 10.8530
Estimator 110/300, Train metric: 10.7520
Estimator 111/300, Train metric: 10.6698
Estimator 112/300, Train metric: 10.6707
Estimator 113/300, Train metric: 10.5762
Estimator 114/300, Train metric: 10.5468
Estimator 115/300, Train metric: 10.4995
Estimator 116/300, Train metric: 10.4526
Estimator 117/300, Train metric: 10.4320
Estimator 118/300, Train metric: 10.4033
Estimator 119/300, Train metric: 10.2728
Estimator 120/300, Train metric: 10.2700
Estimator 121/300, Train metric: 10.1300
Estimator 122/300, Train metric: 10.0422
Estimator 123/300, Train metric: 9.9921
Estimator 124/300

Estimator 137/300, Train metric: 9.2752
Estimator 138/300, Train metric: 9.1582
Estimator 139/300, Train metric: 9.1144
Estimator 140/300, Train metric: 9.1211
Estimator 141/300, Train metric: 9.0928
Estimator 142/300, Train metric: 9.0618
Estimator 143/300, Train metric: 9.0502
Estimator 144/300, Train metric: 8.9818
Estimator 145/300, Train metric: 8.9173
Estimator 146/300, Train metric: 8.8772
Estimator 147/300, Train metric: 8.7642
Estimator 148/300, Train metric: 8.7541
Estimator 149/300, Train metric: 8.7641
Estimator 150/300, Train metric: 8.7435
Estimator 151/300, Train metric: 8.6394
Estimator 152/300, Train metric: 8.6179
Estimator 153/300, Train metric: 8.5644
Estimator 154/300, Train metric: 8.5225
Estimator 155/300, Train metric: 8.3490
Estimator 156/300, Train metric: 8.3194
Estimator 157/300, Train metric: 8.2799
Estimator 158/300, Train metric: 8.2496
Estimator 159/300, Train metric: 8.2563
Estimator 160/300, Train metric: 8.2039
Estimator 161/300, Train metric: 8.1449


Estimator 174/300, Train metric: 7.8243
Estimator 175/300, Train metric: 7.8278
Estimator 176/300, Train metric: 7.8433
Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264
Estimator 179/300, Train metric: 7.7773
Estimator 180/300, Train metric: 7.6972
Estimator 181/300, Train metric: 7.6182
Estimator 182/300, Train metric: 7.5688
Estimator 183/300, Train metric: 7.5420
Estimator 184/300, Train metric: 7.4684
Estimator 185/300, Train metric: 7.4722
Estimator 186/300, Train metric: 7.4772
Estimator 187/300, Train metric: 7.3982
Estimator 188/300, Train metric: 7.3940
Estimator 189/300, Train metric: 7.3914
Estimator 190/300, Train metric: 7.3487
Estimator 191/300, Train metric: 7.2955
Estimator 192/300, Train metric: 7.2783
Estimator 193/300, Train metric: 7.2857
Estimator 194/300, Train metric: 7.2736
Estimator 195/300, Train metric: 7.2801
Estimator 196/300, Train metric: 7.2204
Estimator 197/300, Train metric: 7.1948
Estimator 198/300, Train metric: 7.1825


Estimator 213/300, Train metric: 6.8710
Estimator 214/300, Train metric: 6.8053
Estimator 215/300, Train metric: 6.7838
Estimator 216/300, Train metric: 6.7737
Estimator 217/300, Train metric: 6.7405
Estimator 218/300, Train metric: 6.6536
Estimator 219/300, Train metric: 6.6429
Estimator 220/300, Train metric: 6.6378
Estimator 221/300, Train metric: 6.6014
Estimator 222/300, Train metric: 6.6011
Estimator 223/300, Train metric: 6.6056
Estimator 224/300, Train metric: 6.5337
Estimator 225/300, Train metric: 6.4972
Estimator 226/300, Train metric: 6.4960
Estimator 227/300, Train metric: 6.4901
Estimator 228/300, Train metric: 6.4128
Estimator 229/300, Train metric: 6.3822
Estimator 230/300, Train metric: 6.3718
Estimator 231/300, Train metric: 6.3704
Estimator 232/300, Train metric: 6.3162
Estimator 233/300, Train metric: 6.3248
Estimator 234/300, Train metric: 6.3246
Estimator 235/300, Train metric: 6.3318
Estimator 236/300, Train metric: 6.3256
Estimator 237/300, Train metric: 6.3282


Estimator 250/300, Train metric: 6.0913
Estimator 251/300, Train metric: 6.0789
Estimator 252/300, Train metric: 6.0865
Estimator 253/300, Train metric: 6.0385
Estimator 254/300, Train metric: 6.0455
Estimator 255/300, Train metric: 6.0139
Estimator 256/300, Train metric: 5.9809
Estimator 257/300, Train metric: 5.9833
Estimator 258/300, Train metric: 5.9267
Estimator 259/300, Train metric: 5.8744
Estimator 260/300, Train metric: 5.8320
Estimator 261/300, Train metric: 5.8273
Estimator 262/300, Train metric: 5.8217
Estimator 263/300, Train metric: 5.8125
Estimator 264/300, Train metric: 5.8159
Estimator 265/300, Train metric: 5.8148
Estimator 266/300, Train metric: 5.7728
Estimator 267/300, Train metric: 5.7383
Estimator 268/300, Train metric: 5.7390
Estimator 269/300, Train metric: 5.7183
Estimator 270/300, Train metric: 5.6924
Estimator 271/300, Train metric: 5.6954
Estimator 272/300, Train metric: 5.6684
Estimator 273/300, Train metric: 5.6512
Estimator 274/300, Train metric: 5.6027


Estimator 288/300, Train metric: 5.3943
Estimator 289/300, Train metric: 5.3629
Estimator 290/300, Train metric: 5.3620
Estimator 291/300, Train metric: 5.3140
Estimator 292/300, Train metric: 5.3226
Estimator 293/300, Train metric: 5.3074
Estimator 294/300, Train metric: 5.2770
Estimator 295/300, Train metric: 5.2490
Estimator 296/300, Train metric: 5.2060
Estimator 297/300, Train metric: 5.1671
Estimator 298/300, Train metric: 5.1673
Estimator 299/300, Train metric: 5.1422
         0.05         0.1        0.15         0.2         0.3         0.4  \
0   89.368225   89.712395   89.884720   90.171989   90.389870   90.629974   
1  144.479233  144.579300  144.624924  144.643768  144.674820  144.697357   
2   98.188240   98.245857   98.256577   98.268898   98.295525   98.308388   
3  265.277527  265.480011  266.200867  266.369965  266.852905  267.042664   
4  164.376160  164.866272  165.195801  165.458145  165.614960  165.836090   

          0.5         0.6         0.7         0.8        

In [64]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test >= int_pred[strat_name][:, 0]) & (y_test <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred): # Corrected: Removed y_test argument as it's not defined and not used
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}

    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=N_BINS):
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_test[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) & (y_test[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan) # or handle empty bins as needed
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages


class visualize:
    @staticmethod
    def coverage(int_pred, y_test):
        return {strat_name: np.mean((y_test >= int_pred[strat_name][:, 0]) & (y_test.to_numpy() <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        return {strat_name: np.sqrt(mean_squared_error(y_test, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = visualize.width(int_pred)
        coverages = visualize.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        n_samples = len(y_true)
        y_true_np = y_true.ravel() # Ensure y_true is numpy array and flattened

        # Determine points inside and outside the interval
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval

        # Plot points inside the interval in green
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        # Plot points outside the interval in orange
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')

        # Plot all ground truth values as blue crosses
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')


        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)], linestyle='--', color='blue', label='Ideal Prediction Line') # Changed label for clarity
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=N_BINS, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()

        # Calculate interval widths for test and train sets
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])

        # Plot histograms for both train and test widths
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')


        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=N_BINS, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()

        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]

        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')

        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax





In [65]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path


import os
def to_numpy_1d(y):
    return y.to_numpy().ravel() if hasattr(y, "to_numpy") else np.asarray(y).ravel()


# FIX N2: the interpolating version below is NOT the NexCP weighted quantile (no +inf mass for the
# test point, linear interpolation between residuals). Renamed so it no longer overrides the
# corrected `weighted_quantile` in the helper cell; kept only for reference.
def _old_weighted_quantile_interp(values, weights, q):
    sorter = np.argsort(values)
    values, weights = values[sorter], weights[sorter]
    cdf = np.cumsum(weights) / np.sum(weights)
    return np.interp(q, cdf, values)



# ============================================================
# NexCP for PGBM (Excel-safe, NexCP-style, NO RETURNS)
# ============================================================

def conformal_predictions_NEXCP_PGBM(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    decay=0.99,
    n_folds=5,
    n_bootstrap=50,
    full_grid_size=50,   # FIX N3: y-grid points per test point for NexCP-Full
    seed=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(seed)

    # ========================================================
    # data conversion (PGBM-specific)
    # ========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    STRATEGIES = {
        "NexCP-Split": "NexCP Split",
        "NexCP-Full": "NexCP Full",
        "NexCP-J+aB": "NexCP Jackknife+ aB",
        "NexCP-CV+": "NexCP CV+",
    }

    y_pred, int_pred = {}, {}

    # =========================================================
    # FIX N1-N3: corrected NexCP methods (Barber et al. 2023). Weights w_i = decay^(n-i): training rows
    # in their original (time) order, newest row weight 1. Residuals are OUT-OF-FOLD (each row scored by
    # a model that never saw it). Was: in-sample training residuals for every variant, an unweighted
    # "Full", bounds averaged (CV+) or min/max-ed (J+aB) without weights.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    n_tr = len(y_tr)
    weights = decay ** np.arange(n_tr - 1, -1, -1)
    notes = {}

    model = model_class(**best_params)
    model.fit(X_tr, y_tr)
    yhat_test = np.asarray(model.predict(X_te)).ravel()
    R_oof, mu_te_folds, fold_preds = _oof_fold_predictions(model_class, best_params, X_tr, y_tr, X_te, n_folds, seed)

    # NexCP - Split, eq. (11)
    q, notes["NexCP-Split"] = _cap_q(weighted_quantile(R_oof, weights, 1 - alpha), R_oof)
    y_pred["NexCP-Split"] = yhat_test
    int_pred["NexCP-Split"] = np.c_[yhat_test - q, yhat_test + q]

    # NexCP - Full, eq. (12), on a grid of candidate y values around the prediction (+/- 2 x split margin)
    lo, hi, notes["NexCP-Full"] = _nexcp_full(model_class, best_params, X_tr, y_tr, X_te, yhat_test, weights,
                                              alpha, half_range=2.0 * q, grid_size=full_grid_size, label=model_name)
    y_pred["NexCP-Full"] = yhat_test
    int_pred["NexCP-Full"] = np.c_[lo, hi]

    # NexCP - Jackknife+ after bootstrap: out-of-bag residuals + eq. (33) weighted quantiles
    R_oob, mu_te_oob, bag_mean, has = _jab_oob(model_class, best_params, X_tr, y_tr, X_te, n_bootstrap, seed)
    lo, hi, n_cap = _jackknife_plus(mu_te_oob, R_oob, weights[has], alpha)
    notes["NexCP-J+aB"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-J+aB"] = bag_mean
    int_pred["NexCP-J+aB"] = np.c_[lo, hi]

    # NexCP - CV+, eq. (33) with K folds
    lo, hi, n_cap = _jackknife_plus(mu_te_folds, R_oof, weights, alpha)
    notes["NexCP-CV+"] = _cap_count_note(n_cap, len(lo))
    y_pred["NexCP-CV+"] = np.mean(fold_preds, axis=0)
    int_pred["NexCP-CV+"] = np.c_[lo, hi]

    # ========================================================
    # Metrics + Summary
    # ========================================================
    summary_rows = []
    values_dict = {}

    print("\n===== NEXCP (PGBM) RESULTS =====")

    for s in STRATEGIES:
        ymin, ymax = int_pred[s].T
        yhat = y_pred[s]

        coverage = np.mean((yte >= ymin) & (yte <= ymax))
        width = np.mean(ymax - ymin)
        rmse = np.sqrt(mean_squared_error(yte, yhat))

        print(f"{STRATEGIES[s]} - Coverage: {coverage*100:.3f}% | Width: {width:.3f}")

        summary_rows.append({
            "Strategy": STRATEGIES[s],
            "Coverage": coverage,
            "Width": width,
            "RMSE": rmse,
        })

        values_dict[f"{s}_true"] = yte
        values_dict[f"{s}_pred"] = yhat
        values_dict[f"{s}_lower"] = ymin
        values_dict[f"{s}_upper"] = ymax

    summary_df = pd.DataFrame(summary_rows)
    summary_df["Note"] = [notes.get(s, "") for s in STRATEGIES]  # FIX N10: caps / limits per method
    values_df = pd.DataFrame(values_dict)

    # ========================================================
    # Plotting (SAVE ONLY — NO SHOW)
    # ========================================================
    idx = np.arange(len(yte))
    plot_buffers = {}

    for s in STRATEGIES:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
        ax.fill_between(idx, int_pred[s][:, 0], int_pred[s][:, 1],
                        color="gray", alpha=0.5, label="Prediction interval")
        ax.plot(idx, y_pred[s], color="red", lw=1, label="Predicted mean")

        ax.set_title(f"{STRATEGIES[s]} ({model_name})")
        ax.set_xlabel("Sample Number")
        ax.set_ylabel(f"{TARGET_LABEL}")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # ========================================================
    # Excel Export (append-safe)
    # ========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary sheet ----
        sheet_name = "NEXCP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot sheets ----
        for s, buf in plot_buffers.items():
            img = Image(buf)
            sheet_name = f"NEXCP_PGBM_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1
            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values sheet ----
        sheet_name = "NEXCP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


conformal_predictions_NEXCP_PGBM(
    model_class=PGBMWrapper,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


Training on CPU
Estimator 0/300, Train metric: 55.0218
Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31

Estimator 37/300, Train metric: 23.8609
Estimator 38/300, Train metric: 23.3188
Estimator 39/300, Train metric: 22.8892
Estimator 40/300, Train metric: 22.4073
Estimator 41/300, Train metric: 21.8657
Estimator 42/300, Train metric: 21.3036
Estimator 43/300, Train metric: 21.0520
Estimator 44/300, Train metric: 20.6351
Estimator 45/300, Train metric: 20.4684
Estimator 46/300, Train metric: 20.1000
Estimator 47/300, Train metric: 19.8026
Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178
Estimator 53/300, Train metric: 18.5688
Estimator 54/300, Train metric: 18.3391
Estimator 55/300, Train metric: 17.9783
Estimator 56/300, Train metric: 17.8861
Estimator 57/300, Train metric: 17.7476
Estimator 58/300, Train metric: 17.4496
Estimator 59/300, Train metric: 17.1360
Estimator 60/300, Train metric: 16.9987
Estimator 61/300, Train metric: 16.7382


Estimator 77/300, Train metric: 14.1670
Estimator 78/300, Train metric: 14.0933
Estimator 79/300, Train metric: 13.9494
Estimator 80/300, Train metric: 13.9759
Estimator 81/300, Train metric: 13.8087
Estimator 82/300, Train metric: 13.6064
Estimator 83/300, Train metric: 13.4929
Estimator 84/300, Train metric: 13.3332
Estimator 85/300, Train metric: 13.2440
Estimator 86/300, Train metric: 13.0907
Estimator 87/300, Train metric: 13.0373
Estimator 88/300, Train metric: 13.0103
Estimator 89/300, Train metric: 12.8905
Estimator 90/300, Train metric: 12.8355
Estimator 91/300, Train metric: 12.7364
Estimator 92/300, Train metric: 12.5994
Estimator 93/300, Train metric: 12.4648
Estimator 94/300, Train metric: 12.2250
Estimator 95/300, Train metric: 12.0125
Estimator 96/300, Train metric: 11.9530
Estimator 97/300, Train metric: 11.9207
Estimator 98/300, Train metric: 11.8422
Estimator 99/300, Train metric: 11.7715
Estimator 100/300, Train metric: 11.7421
Estimator 101/300, Train metric: 11.648

Estimator 116/300, Train metric: 10.4526
Estimator 117/300, Train metric: 10.4320
Estimator 118/300, Train metric: 10.4033
Estimator 119/300, Train metric: 10.2728
Estimator 120/300, Train metric: 10.2700
Estimator 121/300, Train metric: 10.1300
Estimator 122/300, Train metric: 10.0422
Estimator 123/300, Train metric: 9.9921
Estimator 124/300, Train metric: 9.9513
Estimator 125/300, Train metric: 9.9225
Estimator 126/300, Train metric: 9.8781
Estimator 127/300, Train metric: 9.8022
Estimator 128/300, Train metric: 9.8036
Estimator 129/300, Train metric: 9.6728
Estimator 130/300, Train metric: 9.6110
Estimator 131/300, Train metric: 9.5791
Estimator 132/300, Train metric: 9.5569
Estimator 133/300, Train metric: 9.5385
Estimator 134/300, Train metric: 9.4700
Estimator 135/300, Train metric: 9.4091
Estimator 136/300, Train metric: 9.4051
Estimator 137/300, Train metric: 9.2752
Estimator 138/300, Train metric: 9.1582
Estimator 139/300, Train metric: 9.1144
Estimator 140/300, Train metric: 

Estimator 154/300, Train metric: 8.5225
Estimator 155/300, Train metric: 8.3490
Estimator 156/300, Train metric: 8.3194
Estimator 157/300, Train metric: 8.2799
Estimator 158/300, Train metric: 8.2496
Estimator 159/300, Train metric: 8.2563
Estimator 160/300, Train metric: 8.2039
Estimator 161/300, Train metric: 8.1449
Estimator 162/300, Train metric: 8.1309
Estimator 163/300, Train metric: 8.1263
Estimator 164/300, Train metric: 8.1300
Estimator 165/300, Train metric: 8.1188
Estimator 166/300, Train metric: 8.1135
Estimator 167/300, Train metric: 8.0945
Estimator 168/300, Train metric: 7.9866
Estimator 169/300, Train metric: 7.9883
Estimator 170/300, Train metric: 7.9698
Estimator 171/300, Train metric: 7.9719
Estimator 172/300, Train metric: 7.9138
Estimator 173/300, Train metric: 7.8723
Estimator 174/300, Train metric: 7.8243
Estimator 175/300, Train metric: 7.8278
Estimator 176/300, Train metric: 7.8433
Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264


Estimator 191/300, Train metric: 7.2955
Estimator 192/300, Train metric: 7.2783
Estimator 193/300, Train metric: 7.2857
Estimator 194/300, Train metric: 7.2736
Estimator 195/300, Train metric: 7.2801
Estimator 196/300, Train metric: 7.2204
Estimator 197/300, Train metric: 7.1948
Estimator 198/300, Train metric: 7.1825
Estimator 199/300, Train metric: 7.1782
Estimator 200/300, Train metric: 7.1704
Estimator 201/300, Train metric: 7.0732
Estimator 202/300, Train metric: 7.0839
Estimator 203/300, Train metric: 7.0691
Estimator 204/300, Train metric: 7.0687
Estimator 205/300, Train metric: 7.0643
Estimator 206/300, Train metric: 7.0131
Estimator 207/300, Train metric: 7.0047
Estimator 208/300, Train metric: 6.9527
Estimator 209/300, Train metric: 6.9280
Estimator 210/300, Train metric: 6.9172
Estimator 211/300, Train metric: 6.9223
Estimator 212/300, Train metric: 6.8699
Estimator 213/300, Train metric: 6.8710
Estimator 214/300, Train metric: 6.8053
Estimator 215/300, Train metric: 6.7838


... (19,868 outputs left out)

Estimator 192/300, Train metric: 4.3978
Estimator 193/300, Train metric: 4.3880
Estimator 194/300, Train metric: 4.3073
Estimator 195/300, Train metric: 4.2661
Estimator 196/300, Train metric: 4.2291
Estimator 197/300, Train metric: 4.1876
Estimator 198/300, Train metric: 4.1416
Estimator 199/300, Train metric: 4.1441
Estimator 200/300, Train metric: 4.1539
Estimator 201/300, Train metric: 4.1197
Estimator 202/300, Train metric: 4.0539
Estimator 203/300, Train metric: 4.0261
Estimator 204/300, Train metric: 4.0024
Estimator 205/300, Train metric: 3.9163
Estimator 206/300, Train metric: 3.9046
Estimator 207/300, Train metric: 3.8321
Estimator 208/300, Train metric: 3.7943
Estimator 209/300, Train metric: 3.7670
Estimator 210/300, Train metric: 3.7609
Estimator 211/300, Train metric: 3.7660
Estimator 212/300, Train metric: 3.7233
Estimator 213/300, Train metric: 3.6981
Estimator 214/300, Train metric: 3.6759
Estimator 215/300, Train metric: 3.6779
Estimator 216/300, Train metric: 3.6502


Estimator 230/300, Train metric: 3.2260
Estimator 231/300, Train metric: 3.2023
Estimator 232/300, Train metric: 3.2038
Estimator 233/300, Train metric: 3.2093
Estimator 234/300, Train metric: 3.2039
Estimator 235/300, Train metric: 3.1934
Estimator 236/300, Train metric: 3.1852
Estimator 237/300, Train metric: 3.1731
Estimator 238/300, Train metric: 3.1381
Estimator 239/300, Train metric: 3.1430
Estimator 240/300, Train metric: 3.0784
Estimator 241/300, Train metric: 3.0519
Estimator 242/300, Train metric: 3.0152
Estimator 243/300, Train metric: 2.9645
Estimator 244/300, Train metric: 2.9276
Estimator 245/300, Train metric: 2.8706
Estimator 246/300, Train metric: 2.8219
Estimator 247/300, Train metric: 2.7969
Estimator 248/300, Train metric: 2.7660
Estimator 249/300, Train metric: 2.7274
Estimator 250/300, Train metric: 2.7231
Estimator 251/300, Train metric: 2.6874
Estimator 252/300, Train metric: 2.6797
Estimator 253/300, Train metric: 2.6791
Estimator 254/300, Train metric: 2.6780


Estimator 266/300, Train metric: 2.4205
Estimator 267/300, Train metric: 2.4163
Estimator 268/300, Train metric: 2.4146
Estimator 269/300, Train metric: 2.4130
Estimator 270/300, Train metric: 2.4102
Estimator 271/300, Train metric: 2.3634
Estimator 272/300, Train metric: 2.3571
Estimator 273/300, Train metric: 2.3471
Estimator 274/300, Train metric: 2.3457
Estimator 275/300, Train metric: 2.3154
Estimator 276/300, Train metric: 2.2831
Estimator 277/300, Train metric: 2.2808
Estimator 278/300, Train metric: 2.2774
Estimator 279/300, Train metric: 2.2137
Estimator 280/300, Train metric: 2.1858
Estimator 281/300, Train metric: 2.1665
Estimator 282/300, Train metric: 2.1633
Estimator 283/300, Train metric: 2.1512
Estimator 284/300, Train metric: 2.1200
Estimator 285/300, Train metric: 2.1183
Estimator 286/300, Train metric: 2.1046
Estimator 287/300, Train metric: 2.0854
Estimator 288/300, Train metric: 2.0665
Estimator 289/300, Train metric: 2.0536
Estimator 290/300, Train metric: 2.0312



===== NEXCP (PGBM) RESULTS =====
NexCP Split - Coverage: 100.000% | Width: 101.938
NexCP Full - Coverage: 100.000% | Width: 148.015
NexCP Jackknife+ aB - Coverage: 97.872% | Width: 86.838
NexCP CV+ - Coverage: 97.872% | Width: 104.896


In [66]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_online_super_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    strategies="all",
    window_min=50,
    window_max=500,
    window_grid=(25, 50, 100, 200, 400),
    n_bootstrap=30,
    seed=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(seed)

    STRATEGIES = {
        "Online-Split": "Online Split",
        "Online-CV+": "Online CV+",
        "Online-J+aB": "Online Jackknife+ aB",
        "SAOCP": "SAOCP",
        "SF-OGD": "SF-OGD",
        "FACI": "FACI",
        "ACI": "ACI",   # FIX N5: added (Gibbs & Candes 2021, gamma = 0.005)
        "MFCS-Split": "MFCS Split",
        "MFCS-Full": "MFCS Full",
    }

    if strategies == "all":
        strategies = list(STRATEGIES.keys())

    # --------------------------------------------------------
    # Safe NumPy conversion (PGBM-specific)
    # --------------------------------------------------------
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # FIX N1, N4-N6: corrected online methods. Every method starts from OUT-OF-FOLD training residuals;
    # test points arrive one at a time and each interval is OUTPUT before that point's error is used.
    # Was: in-sample training residuals; SF-OGD with flipped gradient sign and look-ahead; "FACI" and
    # "SAOCP" were window heuristics (FACI also looked ahead); J+aB scored models on their own samples.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    y_te = np.asarray(y_test).ravel()
    n_train, n_test = len(y_tr), len(y_te)

    base_model = model_class(**model_params)
    base_model.fit(X_tr, y_tr)
    yhat_te = np.asarray(base_model.predict(X_te)).ravel()
    res_te = np.abs(y_te - yhat_te)
    R_oof, mu_te_folds, _ = _oof_fold_predictions(model_class, model_params, X_tr, y_tr, X_te, 5, seed)

    y_pred, int_pred, notes = {}, {}, {}

    def _band(name, centre, q):
        y_pred[name] = np.asarray(centre)
        int_pred[name] = np.c_[centre - q, centre + q]

    if "Online-Split" in strategies:
        q, notes["Online-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("Online-Split", yhat_te, q)

    if "Online-CV+" in strategies:
        # CV+ (Barber et al. 2021); past test points join the calibration set once their error is known
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_te_folds[:, j], np.full(j, yhat_te[j])]
            R = np.r_[R_oof, res_te[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-CV+"] = yhat_te
        int_pred["Online-CV+"] = np.c_[lo, hi]
        notes["Online-CV+"] = _cap_count_note(n_cap, n_test)

    if "Online-J+aB" in strategies:
        # J+aB (Kim, Xu & Barber 2020): out-of-bag residuals; past test points join as above
        R_oob, mu_oob, bag_mean, has = _jab_oob(model_class, model_params, X_tr, y_tr, X_te, n_bootstrap, seed)
        res_bag = np.abs(y_te - bag_mean)
        lo, hi, n_cap = [], [], 0
        for j in range(n_test):
            mu = np.r_[mu_oob[:, j], np.full(j, bag_mean[j])]
            R = np.r_[R_oob, res_bag[:j]]
            l, h, c = _jackknife_plus(mu[:, None], R, np.ones(len(R)), alpha)
            lo.append(l[0]); hi.append(h[0]); n_cap += c
        y_pred["Online-J+aB"] = bag_mean
        int_pred["Online-J+aB"] = np.c_[lo, hi]
        notes["Online-J+aB"] = _cap_count_note(n_cap, n_test)

    if "SAOCP" in strategies:
        _band("SAOCP", yhat_te, _saocp(R_oof, res_te, alpha))

    if "SF-OGD" in strategies:
        _band("SF-OGD", yhat_te, _sf_ogd(R_oof, res_te, alpha))

    if "FACI" in strategies:
        _band("FACI", yhat_te, _faci(R_oof, res_te, alpha))

    if "ACI" in strategies:
        q_seq, n_cap = _aci(R_oof, res_te, alpha, gamma=0.005)
        _band("ACI", yhat_te, q_seq)
        notes["ACI"] = "" if n_cap == 0 else f"{n_cap} of {n_test} steps had alpha_t <= 0 (infinite ACI set, capped at the largest error)"

    if "MFCS-Split" in strategies:
        # kept as written by the user (no published source found); same computation as Online-Split
        q, notes["MFCS-Split"] = _cap_q(weighted_quantile(R_oof, np.ones(n_train), 1 - alpha), R_oof)
        _band("MFCS-Split", yhat_te, q)

    if "MFCS-Full" in strategies:
        # refit on all points seen so far; margin from that refit's OUT-OF-FOLD residuals (FIX N1)
        lo, hi, preds, n_cap = [], [], [], 0
        for j in range(n_test):
            X_h = np.vstack([X_tr, X_te[:j]])
            y_h = np.r_[y_tr, y_te[:j]]
            m = model_class(**model_params)
            m.fit(X_h, y_h)
            yhat = float(np.asarray(m.predict(X_te[j:j + 1])).ravel()[0])
            R_h, _, _ = _oof_fold_predictions(model_class, model_params, X_h, y_h, X_te[j:j + 1], 5, seed)
            q = weighted_quantile(R_h, np.ones(len(R_h)), 1 - alpha)
            if not np.isfinite(q):
                q, n_cap = float(np.max(R_h)), n_cap + 1
            preds.append(yhat); lo.append(yhat - q); hi.append(yhat + q)
        y_pred["MFCS-Full"] = np.asarray(preds)
        int_pred["MFCS-Full"] = np.c_[lo, hi]
        notes["MFCS-Full"] = _cap_count_note(n_cap, n_test)

    # ========================================================
    # Metrics + Summary
    # ========================================================
    summary_rows = []
    values_dict = {}

    print("\n===== ONLINE CP (PGBM) RESULTS =====")

    for s in y_pred:
        ymin, ymax = int_pred[s].T
        yhat = y_pred[s]

        coverage = np.mean((yte >= ymin) & (yte <= ymax))
        width = np.mean(ymax - ymin)
        rmse = np.sqrt(mean_squared_error(yte, yhat))

        print(f"{STRATEGIES[s]} - Coverage: {coverage*100:.3f}% | Width: {width:.3f}")

        summary_rows.append({
            "Strategy": STRATEGIES[s],
            "Coverage": coverage,
            "Width": width,
            "RMSE": rmse,
        })

        values_dict[f"{s}_true"] = yte
        values_dict[f"{s}_pred"] = yhat
        values_dict[f"{s}_lower"] = ymin
        values_dict[f"{s}_upper"] = ymax

    summary_df = pd.DataFrame(summary_rows)
    summary_df["Note"] = [notes.get(s, "") for s in y_pred]  # FIX N10: caps / limits per method
    values_df = pd.DataFrame(values_dict)

    # ========================================================
    # Plotting (SAVE ONLY — NO SHOW)
    # ========================================================
    idx = np.arange(len(yte))
    plot_buffers = {}

    for s in y_pred:
        fig, ax = plt.subplots(figsize=(10, 4))

        ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
        ax.fill_between(idx, int_pred[s][:, 0], int_pred[s][:, 1],
                        color="gray", alpha=0.5, label="Prediction interval")
        ax.plot(idx, y_pred[s], color="red", lw=1, label="Predicted mean")

        ax.set_title(f"{STRATEGIES[s]} ({model_name})")
        ax.set_xlabel("Sample Number")
        ax.set_ylabel(f"{TARGET_LABEL}")
        ax.legend(loc="upper left")

        plt.tight_layout()

        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        buf.seek(0)
        plot_buffers[s] = buf

        plt.close(fig)

    # ========================================================
    # Excel Export (append-safe)
    # ========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "ONLINECP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plots ----
        for s, buf in plot_buffers.items():
            img = Image(buf)
            sheet_name = f"ONLINECP_PGBM_Plot_{s}"
            base = sheet_name
            i = 1
            while sheet_name in wb.sheetnames:
                sheet_name = f"{base}_{i}"
                i += 1
            ws = wb.create_sheet(sheet_name)
            ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "ONLINECP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


conformal_predictions_online_super_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    strategies="all",
)


Training on CPU
Estimator 0/300, Train metric: 55.0218
Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31

Estimator 38/300, Train metric: 23.3188
Estimator 39/300, Train metric: 22.8892
Estimator 40/300, Train metric: 22.4073
Estimator 41/300, Train metric: 21.8657
Estimator 42/300, Train metric: 21.3036
Estimator 43/300, Train metric: 21.0520
Estimator 44/300, Train metric: 20.6351
Estimator 45/300, Train metric: 20.4684
Estimator 46/300, Train metric: 20.1000
Estimator 47/300, Train metric: 19.8026
Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178
Estimator 53/300, Train metric: 18.5688
Estimator 54/300, Train metric: 18.3391
Estimator 55/300, Train metric: 17.9783
Estimator 56/300, Train metric: 17.8861
Estimator 57/300, Train metric: 17.7476
Estimator 58/300, Train metric: 17.4496
Estimator 59/300, Train metric: 17.1360
Estimator 60/300, Train metric: 16.9987
Estimator 61/300, Train metric: 16.7382
Estimator 62/300, Train metric: 16.5740


Estimator 75/300, Train metric: 14.3939
Estimator 76/300, Train metric: 14.2124
Estimator 77/300, Train metric: 14.1670
Estimator 78/300, Train metric: 14.0933
Estimator 79/300, Train metric: 13.9494
Estimator 80/300, Train metric: 13.9759
Estimator 81/300, Train metric: 13.8087
Estimator 82/300, Train metric: 13.6064
Estimator 83/300, Train metric: 13.4929
Estimator 84/300, Train metric: 13.3332
Estimator 85/300, Train metric: 13.2440
Estimator 86/300, Train metric: 13.0907
Estimator 87/300, Train metric: 13.0373
Estimator 88/300, Train metric: 13.0103
Estimator 89/300, Train metric: 12.8905
Estimator 90/300, Train metric: 12.8355
Estimator 91/300, Train metric: 12.7364
Estimator 92/300, Train metric: 12.5994
Estimator 93/300, Train metric: 12.4648
Estimator 94/300, Train metric: 12.2250
Estimator 95/300, Train metric: 12.0125
Estimator 96/300, Train metric: 11.9530
Estimator 97/300, Train metric: 11.9207
Estimator 98/300, Train metric: 11.8422
Estimator 99/300, Train metric: 11.7715


Estimator 111/300, Train metric: 10.6698
Estimator 112/300, Train metric: 10.6707
Estimator 113/300, Train metric: 10.5762
Estimator 114/300, Train metric: 10.5468
Estimator 115/300, Train metric: 10.4995
Estimator 116/300, Train metric: 10.4526
Estimator 117/300, Train metric: 10.4320
Estimator 118/300, Train metric: 10.4033
Estimator 119/300, Train metric: 10.2728
Estimator 120/300, Train metric: 10.2700
Estimator 121/300, Train metric: 10.1300
Estimator 122/300, Train metric: 10.0422
Estimator 123/300, Train metric: 9.9921
Estimator 124/300, Train metric: 9.9513
Estimator 125/300, Train metric: 9.9225
Estimator 126/300, Train metric: 9.8781
Estimator 127/300, Train metric: 9.8022
Estimator 128/300, Train metric: 9.8036
Estimator 129/300, Train metric: 9.6728
Estimator 130/300, Train metric: 9.6110
Estimator 131/300, Train metric: 9.5791
Estimator 132/300, Train metric: 9.5569
Estimator 133/300, Train metric: 9.5385
Estimator 134/300, Train metric: 9.4700
Estimator 135/300, Train met

Estimator 144/300, Train metric: 8.9818
Estimator 145/300, Train metric: 8.9173
Estimator 146/300, Train metric: 8.8772
Estimator 147/300, Train metric: 8.7642
Estimator 148/300, Train metric: 8.7541
Estimator 149/300, Train metric: 8.7641
Estimator 150/300, Train metric: 8.7435
Estimator 151/300, Train metric: 8.6394
Estimator 152/300, Train metric: 8.6179
Estimator 153/300, Train metric: 8.5644
Estimator 154/300, Train metric: 8.5225
Estimator 155/300, Train metric: 8.3490
Estimator 156/300, Train metric: 8.3194
Estimator 157/300, Train metric: 8.2799
Estimator 158/300, Train metric: 8.2496
Estimator 159/300, Train metric: 8.2563
Estimator 160/300, Train metric: 8.2039
Estimator 161/300, Train metric: 8.1449
Estimator 162/300, Train metric: 8.1309
Estimator 163/300, Train metric: 8.1263
Estimator 164/300, Train metric: 8.1300
Estimator 165/300, Train metric: 8.1188
Estimator 166/300, Train metric: 8.1135
Estimator 167/300, Train metric: 8.0945
Estimator 168/300, Train metric: 7.9866


Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264
Estimator 179/300, Train metric: 7.7773
Estimator 180/300, Train metric: 7.6972
Estimator 181/300, Train metric: 7.6182
Estimator 182/300, Train metric: 7.5688
Estimator 183/300, Train metric: 7.5420
Estimator 184/300, Train metric: 7.4684
Estimator 185/300, Train metric: 7.4722
Estimator 186/300, Train metric: 7.4772
Estimator 187/300, Train metric: 7.3982
Estimator 188/300, Train metric: 7.3940
Estimator 189/300, Train metric: 7.3914
Estimator 190/300, Train metric: 7.3487
Estimator 191/300, Train metric: 7.2955
Estimator 192/300, Train metric: 7.2783
Estimator 193/300, Train metric: 7.2857
Estimator 194/300, Train metric: 7.2736
Estimator 195/300, Train metric: 7.2801
Estimator 196/300, Train metric: 7.2204
Estimator 197/300, Train metric: 7.1948
Estimator 198/300, Train metric: 7.1825
Estimator 199/300, Train metric: 7.1782
Estimator 200/300, Train metric: 7.1704
Estimator 201/300, Train metric: 7.0732


... (3,174 outputs left out)

Estimator 222/300, Train metric: 3.4848
Estimator 223/300, Train metric: 3.4820
Estimator 224/300, Train metric: 3.4271
Estimator 225/300, Train metric: 3.4018
Estimator 226/300, Train metric: 3.3821
Estimator 227/300, Train metric: 3.3579
Estimator 228/300, Train metric: 3.3073
Estimator 229/300, Train metric: 3.2868
Estimator 230/300, Train metric: 3.2545
Estimator 231/300, Train metric: 3.2322
Estimator 232/300, Train metric: 3.2172
Estimator 233/300, Train metric: 3.2182
Estimator 234/300, Train metric: 3.1976
Estimator 235/300, Train metric: 3.1640
Estimator 236/300, Train metric: 3.1351
Estimator 237/300, Train metric: 3.1060
Estimator 238/300, Train metric: 3.0699
Estimator 239/300, Train metric: 3.0547
Estimator 240/300, Train metric: 3.0396
Estimator 241/300, Train metric: 3.0248
Estimator 242/300, Train metric: 3.0269
Estimator 243/300, Train metric: 3.0098
Estimator 244/300, Train metric: 2.9633


Estimator 245/300, Train metric: 2.9475
Estimator 246/300, Train metric: 2.9465
Estimator 247/300, Train metric: 2.9302
Estimator 248/300, Train metric: 2.9173
Estimator 249/300, Train metric: 2.8651
Estimator 250/300, Train metric: 2.8396
Estimator 251/300, Train metric: 2.8139
Estimator 252/300, Train metric: 2.7765
Estimator 253/300, Train metric: 2.7667
Estimator 254/300, Train metric: 2.7584
Estimator 255/300, Train metric: 2.7231
Estimator 256/300, Train metric: 2.6942
Estimator 257/300, Train metric: 2.6814
Estimator 258/300, Train metric: 2.6637
Estimator 259/300, Train metric: 2.6545
Estimator 260/300, Train metric: 2.6531
Estimator 261/300, Train metric: 2.6104
Estimator 262/300, Train metric: 2.6129
Estimator 263/300, Train metric: 2.6141
Estimator 264/300, Train metric: 2.5663
Estimator 265/300, Train metric: 2.5579
Estimator 266/300, Train metric: 2.5499
Estimator 267/300, Train metric: 2.5529


Estimator 268/300, Train metric: 2.5241
Estimator 269/300, Train metric: 2.5065
Estimator 270/300, Train metric: 2.4962
Estimator 271/300, Train metric: 2.4765
Estimator 272/300, Train metric: 2.4685
Estimator 273/300, Train metric: 2.4301
Estimator 274/300, Train metric: 2.4066
Estimator 275/300, Train metric: 2.3904
Estimator 276/300, Train metric: 2.3897
Estimator 277/300, Train metric: 2.3803
Estimator 278/300, Train metric: 2.3435
Estimator 279/300, Train metric: 2.3274
Estimator 280/300, Train metric: 2.2921
Estimator 281/300, Train metric: 2.2756
Estimator 282/300, Train metric: 2.2740
Estimator 283/300, Train metric: 2.2365
Estimator 284/300, Train metric: 2.2159
Estimator 285/300, Train metric: 2.1955
Estimator 286/300, Train metric: 2.1789
Estimator 287/300, Train metric: 2.1700
Estimator 288/300, Train metric: 2.1728
Estimator 289/300, Train metric: 2.1722
Estimator 290/300, Train metric: 2.1538


Estimator 291/300, Train metric: 2.1271
Estimator 292/300, Train metric: 2.1279
Estimator 293/300, Train metric: 2.1291
Estimator 294/300, Train metric: 2.1183
Estimator 295/300, Train metric: 2.0884
Estimator 296/300, Train metric: 2.0696
Estimator 297/300, Train metric: 2.0585
Estimator 298/300, Train metric: 2.0475
Estimator 299/300, Train metric: 2.0310

===== ONLINE CP (PGBM) RESULTS =====
Online Split - Coverage: 100.000% | Width: 101.938
Online CV+ - Coverage: 97.872% | Width: 88.203
Online Jackknife+ aB - Coverage: 97.872% | Width: 79.042
SAOCP - Coverage: 91.489% | Width: 76.632
SF-OGD - Coverage: 91.489% | Width: 91.522
FACI - Coverage: 93.617% | Width: 72.089
ACI - Coverage: 93.617% | Width: 76.430
MFCS Split - Coverage: 100.000% | Width: 101.938
MFCS Full - Coverage: 97.872% | Width: 81.073


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_COP_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    lr=0.05,
    T_burnin=COP_T_BURNIN,
    scale=0.5,
    random_state=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # =========================================================
    # Safe NumPy conversion (PGBM-specific)
    # =========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # Train base PGBM model
    # =========================================================
    model = model_class(**model_params)
    model.fit(Xtr, ytr)

    y_pred_test = model.predict(Xte).ravel()
    y_pred_train = model.predict(Xtr).ravel()

    # Absolute residuals (symmetric COP)
    scores_test = np.abs(yte - y_pred_test)
    scores_train = np.abs(ytr - y_pred_train)

    T_test = len(scores_test)

    # =========================================================
    # COP initialization
    # =========================================================
    # FIX N1: start from OUT-OF-FOLD training residuals (was: in-sample residuals, too small)
    R_oof_cop, _, _ = _oof_fold_predictions(model_class, model_params, np.asarray(X_train),
                                            np.asarray(y_train).ravel(), np.asarray(X_test)[:1], 5, 123)
    init_q = np.quantile(R_oof_cop, 1 - alpha)

    qs = np.zeros(T_test)
    qts = np.zeros(T_test)
    integrators = np.zeros(T_test)
    covereds = np.zeros(T_test)

    qs[0] = init_q
    qts[0] = init_q

    # =========================================================
    # Online COP loop
    # =========================================================
    for t in range(T_test - 1):

        covereds[t] = 1 if qs[t] >= scores_test[t] else 0

        # FIX N8: step size proportional to the score scale, as in the official COP code
        # (lr x (max - min) of the last T_burnin scores). Was: a fixed lr = 0.05 in target units.
        _hist = np.r_[R_oof_cop, scores_test[:t + 1]][-T_burnin:]
        lr_t = lr * (np.max(_hist) - np.min(_hist))

        # Standard COP gradient
        grad = alpha if covereds[t] else -(1 - alpha)

        # Distribution-informed (optimistic) gradient
        if t < T_burnin:
            grad_i = 0.0
        else:
            window_s = scores_test[t - T_burnin : t]
            target = qts[t] - lr_t * grad   # FIX N8
            grad_i = np.mean(window_s <= target) - (1 - alpha)

        # Updates
        integrator = -scale * grad_i
        qts[t + 1] = qts[t] - lr_t * grad   # FIX N8
        integrators[t + 1] = lr_t * integrator   # FIX N8
        qs[t + 1] = max(0.0, qts[t + 1] + integrators[t + 1])

    covereds[-1] = 1 if qs[-1] >= scores_test[-1] else 0

    # =========================================================
    # Prediction intervals
    # =========================================================
    ymin = y_pred_test - qs
    ymax = y_pred_test + qs

    # =========================================================
    # Metrics
    # =========================================================
    rmse = np.sqrt(mean_squared_error(yte, y_pred_test))
    coverage = np.mean(covereds)
    avg_width = np.mean(ymax - ymin)

    print(f"\n===== COP (PGBM) =====")
    print(f"Coverage : {coverage*100:.3f}%")
    print(f"Width    : {avg_width:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    print(f"Final q  : {qs[-1]:.3f}")

    # =========================================================
    # Tables
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy": "COP",
        "Coverage": coverage,
        "Width": avg_width,
        "RMSE": rmse,
        "Alpha": alpha,
        "LR": lr,
        "Scale": scale,
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true": yte,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": ymax - ymin,
        "is_covered": covereds.astype(int),
        "score": scores_test,
        "adaptive_q": qs,
    })

    # =========================================================
    # Plot (SAVE ONLY)
    # =========================================================
    idx = np.arange(len(yte))
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
    ax.fill_between(idx, ymin, ymax, color="gray", alpha=0.5, label="COP Interval")
    ax.plot(idx, y_pred_test, color="red", lw=1, label="Predicted mean")


    ax.set_title(f"COP ({model_name})")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel(f"{TARGET_LABEL}")
    ax.legend(loc="upper left")

    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # =========================================================
    # Excel export (append-safe)
    # =========================================================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "COP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "COP_PGBM_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1
        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "COP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df

conformal_predictions_COP_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)


Training on CPU
Estimator 0/300, Train metric: 55.0218
Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31

Estimator 34/300, Train metric: 25.2632
Estimator 35/300, Train metric: 24.6503
Estimator 36/300, Train metric: 24.2793
Estimator 37/300, Train metric: 23.8609
Estimator 38/300, Train metric: 23.3188
Estimator 39/300, Train metric: 22.8892
Estimator 40/300, Train metric: 22.4073
Estimator 41/300, Train metric: 21.8657
Estimator 42/300, Train metric: 21.3036
Estimator 43/300, Train metric: 21.0520
Estimator 44/300, Train metric: 20.6351
Estimator 45/300, Train metric: 20.4684
Estimator 46/300, Train metric: 20.1000
Estimator 47/300, Train metric: 19.8026
Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178
Estimator 53/300, Train metric: 18.5688
Estimator 54/300, Train metric: 18.3391
Estimator 55/300, Train metric: 17.9783
Estimator 56/300, Train metric: 17.8861
Estimator 57/300, Train metric: 17.7476
Estimator 58/300, Train metric: 17.4496


Estimator 69/300, Train metric: 15.2547
Estimator 70/300, Train metric: 15.1427
Estimator 71/300, Train metric: 15.0232
Estimator 72/300, Train metric: 14.7207
Estimator 73/300, Train metric: 14.5002
Estimator 74/300, Train metric: 14.4790
Estimator 75/300, Train metric: 14.3939
Estimator 76/300, Train metric: 14.2124
Estimator 77/300, Train metric: 14.1670
Estimator 78/300, Train metric: 14.0933
Estimator 79/300, Train metric: 13.9494
Estimator 80/300, Train metric: 13.9759
Estimator 81/300, Train metric: 13.8087
Estimator 82/300, Train metric: 13.6064
Estimator 83/300, Train metric: 13.4929
Estimator 84/300, Train metric: 13.3332
Estimator 85/300, Train metric: 13.2440
Estimator 86/300, Train metric: 13.0907
Estimator 87/300, Train metric: 13.0373
Estimator 88/300, Train metric: 13.0103
Estimator 89/300, Train metric: 12.8905
Estimator 90/300, Train metric: 12.8355
Estimator 91/300, Train metric: 12.7364
Estimator 92/300, Train metric: 12.5994
Estimator 93/300, Train metric: 12.4648


Estimator 103/300, Train metric: 11.3104
Estimator 104/300, Train metric: 11.1709
Estimator 105/300, Train metric: 11.1320
Estimator 106/300, Train metric: 11.1323
Estimator 107/300, Train metric: 11.1133
Estimator 108/300, Train metric: 11.0399
Estimator 109/300, Train metric: 10.8530
Estimator 110/300, Train metric: 10.7520
Estimator 111/300, Train metric: 10.6698
Estimator 112/300, Train metric: 10.6707
Estimator 113/300, Train metric: 10.5762
Estimator 114/300, Train metric: 10.5468
Estimator 115/300, Train metric: 10.4995
Estimator 116/300, Train metric: 10.4526
Estimator 117/300, Train metric: 10.4320
Estimator 118/300, Train metric: 10.4033
Estimator 119/300, Train metric: 10.2728
Estimator 120/300, Train metric: 10.2700
Estimator 121/300, Train metric: 10.1300
Estimator 122/300, Train metric: 10.0422
Estimator 123/300, Train metric: 9.9921
Estimator 124/300, Train metric: 9.9513
Estimator 125/300, Train metric: 9.9225
Estimator 126/300, Train metric: 9.8781
Estimator 127/300, T

Estimator 135/300, Train metric: 9.4091
Estimator 136/300, Train metric: 9.4051
Estimator 137/300, Train metric: 9.2752
Estimator 138/300, Train metric: 9.1582
Estimator 139/300, Train metric: 9.1144
Estimator 140/300, Train metric: 9.1211
Estimator 141/300, Train metric: 9.0928
Estimator 142/300, Train metric: 9.0618
Estimator 143/300, Train metric: 9.0502
Estimator 144/300, Train metric: 8.9818
Estimator 145/300, Train metric: 8.9173
Estimator 146/300, Train metric: 8.8772
Estimator 147/300, Train metric: 8.7642
Estimator 148/300, Train metric: 8.7541
Estimator 149/300, Train metric: 8.7641
Estimator 150/300, Train metric: 8.7435
Estimator 151/300, Train metric: 8.6394
Estimator 152/300, Train metric: 8.6179
Estimator 153/300, Train metric: 8.5644
Estimator 154/300, Train metric: 8.5225
Estimator 155/300, Train metric: 8.3490
Estimator 156/300, Train metric: 8.3194
Estimator 157/300, Train metric: 8.2799
Estimator 158/300, Train metric: 8.2496
Estimator 159/300, Train metric: 8.2563


Estimator 166/300, Train metric: 8.1135
Estimator 167/300, Train metric: 8.0945
Estimator 168/300, Train metric: 7.9866
Estimator 169/300, Train metric: 7.9883
Estimator 170/300, Train metric: 7.9698
Estimator 171/300, Train metric: 7.9719
Estimator 172/300, Train metric: 7.9138
Estimator 173/300, Train metric: 7.8723
Estimator 174/300, Train metric: 7.8243
Estimator 175/300, Train metric: 7.8278
Estimator 176/300, Train metric: 7.8433
Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264
Estimator 179/300, Train metric: 7.7773
Estimator 180/300, Train metric: 7.6972
Estimator 181/300, Train metric: 7.6182
Estimator 182/300, Train metric: 7.5688
Estimator 183/300, Train metric: 7.5420
Estimator 184/300, Train metric: 7.4684
Estimator 185/300, Train metric: 7.4722
Estimator 186/300, Train metric: 7.4772
Estimator 187/300, Train metric: 7.3982
Estimator 188/300, Train metric: 7.3940
Estimator 189/300, Train metric: 7.3914
Estimator 190/300, Train metric: 7.3487


... (40 outputs left out)

Estimator 212/300, Train metric: 6.8942
Estimator 213/300, Train metric: 6.7845
Estimator 214/300, Train metric: 6.7917
Estimator 215/300, Train metric: 6.7924
Estimator 216/300, Train metric: 6.7957
Estimator 217/300, Train metric: 6.8059
Estimator 218/300, Train metric: 6.8052
Estimator 219/300, Train metric: 6.8087
Estimator 220/300, Train metric: 6.8106
Estimator 221/300, Train metric: 6.7497
Estimator 222/300, Train metric: 6.7012
Estimator 223/300, Train metric: 6.6902
Estimator 224/300, Train metric: 6.6882
Estimator 225/300, Train metric: 6.6842
Estimator 226/300, Train metric: 6.6434
Estimator 227/300, Train metric: 6.6134
Estimator 228/300, Train metric: 6.5890
Estimator 229/300, Train metric: 6.5629
Estimator 230/300, Train metric: 6.5749
Estimator 231/300, Train metric: 6.5470
Estimator 232/300, Train metric: 6.5441
Estimator 233/300, Train metric: 6.5476
Estimator 234/300, Train metric: 6.5218
Estimator 235/300, Train metric: 6.4685
Estimator 236/300, Train metric: 6.4732


Estimator 255/300, Train metric: 6.0294
Estimator 256/300, Train metric: 6.0219
Estimator 257/300, Train metric: 6.0178
Estimator 258/300, Train metric: 5.9615
Estimator 259/300, Train metric: 5.9713
Estimator 260/300, Train metric: 5.9579
Estimator 261/300, Train metric: 5.9114
Estimator 262/300, Train metric: 5.8944
Estimator 263/300, Train metric: 5.8945
Estimator 264/300, Train metric: 5.8903
Estimator 265/300, Train metric: 5.8191
Estimator 266/300, Train metric: 5.8140
Estimator 267/300, Train metric: 5.7898
Estimator 268/300, Train metric: 5.7866
Estimator 269/300, Train metric: 5.7381
Estimator 270/300, Train metric: 5.6916
Estimator 271/300, Train metric: 5.6987
Estimator 272/300, Train metric: 5.6291
Estimator 273/300, Train metric: 5.6267
Estimator 274/300, Train metric: 5.5999
Estimator 275/300, Train metric: 5.5795
Estimator 276/300, Train metric: 5.5807
Estimator 277/300, Train metric: 5.5526
Estimator 278/300, Train metric: 5.5096
Estimator 279/300, Train metric: 5.5008


Estimator 296/300, Train metric: 5.1676
Estimator 297/300, Train metric: 5.1333
Estimator 298/300, Train metric: 5.1313
Estimator 299/300, Train metric: 5.1343

===== COP (PGBM) =====
Coverage : 93.617%
Width    : 76.560
RMSE     : 19.532
Final q  : 36.756


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.93617,76.559699,19.531829,0.1,0.05,0.5


In [68]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_EXTREME_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    calib_size=0.2,
    gpd_threshold_quantile=0.95,
    n_bootstrap=30,
    random_state=123,
):

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)

    # --------------------------------------------------------
    # Safe NumPy conversion (PGBM-specific)
    # --------------------------------------------------------
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy() if hasattr(X_test, "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # --------------------------------------------------------
    # Train / calibration split
    # --------------------------------------------------------
    X_fit, X_cal, y_fit, y_cal = train_test_split(
        Xtr, ytr, test_size=calib_size, random_state=random_state, shuffle=True
    )

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    model = model_class(**model_params)
    model.fit(X_fit, y_fit)

    y_pred_test = model.predict(Xte)
    y_pred_cal = model.predict(X_cal)

    # --------------------------------------------------------
    # Upper-tail nonconformity scores
    # --------------------------------------------------------
    # FIX N9: Extreme CP as in the official ExtremeConformal code (conformalize_GPD_max):
    # signed scores y - y_hat on ALL calibration points, thresholds 0.80..0.99 (capped so >= 10 scores
    # exceed them), official GPD quantile formula, maximum over thresholds; ONE-SIDED upper bound.
    # Was: only positive scores kept, (1/alpha)^xi in the formula (missing the threshold share), a hard
    # error with < 30 exceedances, and a two-sided band reported for a one-sided method.
    scores = np.asarray(y_cal).ravel() - y_pred_cal
    q_extreme, gpd_table = _conformalize_gpd_max(scores, 1 - alpha, min_obs_gpd=10)
    extreme_note = ""
    if not np.isfinite(q_extreme):
        q_extreme = weighted_quantile(scores, np.ones(len(scores)), 1 - alpha)
        extreme_note = "GPD fit failed at every threshold; empirical conformal quantile used"
    best_row = gpd_table.loc[gpd_table["q_threshold"].idxmax()] if gpd_table["q_threshold"].notna().any() else None
    u = float(best_row["threshold_u"]) if best_row is not None else np.nan
    n_exc_used = int(best_row["n_exceedances"]) if best_row is not None else 0
    print(gpd_table.to_string(index=False))

    # one-sided interval (-inf, y_hat + q]
    y_pred_test = np.asarray(y_pred_test).ravel()
    ymin = np.array([None] * len(y_pred_test), dtype=object)   # no lower bound (empty cell in Excel)
    ymax = y_pred_test + q_extreme

    # --------------------------------------------------------
    # Metrics (one-sided coverage)
    # --------------------------------------------------------
    rmse = np.sqrt(mean_squared_error(yte, y_pred_test))
    coverage = np.mean(yte <= ymax)
    width = q_extreme

    print(f"\n===== EXTREME CP (PGBM) =====")
    print(f"Coverage     : {coverage*100:.3f}%")
    print(f"Width        : {width:.3f}")
    print(f"RMSE         : {rmse:.3f}")
    print(f"q_extreme    : {q_extreme:.3f}")
    print(f"threshold_u  : {u:.3f}")
    print(f"#Exceedances : {n_exc_used}")
    if extreme_note:
        print(f"Note     : {extreme_note}")

    # --------------------------------------------------------
    # Summary table
    # --------------------------------------------------------
    summary_df = pd.DataFrame([{
        "Strategy": "EXTREME CP",
        "Coverage": coverage,
        "Width": width,
        "RMSE": rmse,
        "q_extreme": q_extreme,
        "threshold_u": u,
        "n_exceedances": n_exc_used,
        "Note": extreme_note,   # FIX N10
    }])

    # --------------------------------------------------------
    # Values table
    # --------------------------------------------------------
    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true": yte,
        "y_pred": y_pred_test,
        "ymin": ymin,
        "ymax": ymax,
        "width": np.full(len(y_pred_test), q_extreme),   # FIX N9: one-sided width = q
        "is_covered": (yte <= ymax).astype(int),
        "residual": yte - y_pred_test,
    })

    # --------------------------------------------------------
    # Plot (SAVE ONLY — NO SHOW)
    # --------------------------------------------------------
    idx = np.arange(len(yte))
    fig, ax = plt.subplots(figsize=(10, 4))

    ax.scatter(idx, yte, s=10, color="blue", alpha=0.7, label="True")
    ax.plot(idx, ymax, ls="--", lw=1, label="Upper bound (one-sided)")   # FIX N9
    ax.plot(idx, y_pred_test, color="red", lw=1, label="Prediction")

    ax.set_title(f"Extreme CP ({model_name})")
    ax.set_xlabel("Sample Number")
    ax.set_ylabel(f"{TARGET_LABEL}")
    ax.legend(loc="upper left")

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    plt.close(fig)

    # --------------------------------------------------------
    # Excel export (append-safe)
    # --------------------------------------------------------
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        # ---- Summary ----
        sheet_name = "EXTREMECP_PGBM_Summary"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- Plot ----
        img = Image(buf)
        sheet_name = "EXTREMECP_PGBM_Plot"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        ws.add_image(img, "A1")

        # ---- Values ----
        sheet_name = "EXTREMECP_PGBM_Values"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet_name)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # FIX N9: GPD fit at every threshold level (the reported q is the maximum)
        sheet_name = "EXTREMECP_PGBM_GPD_Thresholds"
        base = sheet_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base}_{i}"
            i += 1
        ws = wb.create_sheet(sheet_name)
        _tbl = gpd_table.astype(object).where(gpd_table.notna(), None)
        for r, row in enumerate(dataframe_to_rows(_tbl, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))


conformal_predictions_EXTREME_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.3,
)

Training on CPU
Estimator 0/300, Train metric: 47.9872
Estimator 1/300, Train metric: 46.5230
Estimator 2/300, Train metric: 45.5795
Estimator 3/300, Train metric: 44.6544
Estimator 4/300, Train metric: 43.5806
Estimator 5/300, Train metric: 42.4868
Estimator 6/300, Train metric: 41.9306
Estimator 7/300, Train metric: 40.8864
Estimator 8/300, Train metric: 39.6130
Estimator 9/300, Train metric: 39.1794
Estimator 10/300, Train metric: 38.4707
Estimator 11/300, Train metric: 38.4037
Estimator 12/300, Train metric: 37.3792
Estimator 13/300, Train metric: 36.8350
Estimator 14/300, Train metric: 36.2328
Estimator 15/300, Train metric: 35.6044
Estimator 16/300, Train metric: 34.8059
Estimator 17/300, Train metric: 34.1822
Estimator 18/300, Train metric: 33.3234
Estimator 19/300, Train metric: 32.7742
Estimator 20/300, Train metric: 32.4492
Estimator 21/300, Train metric: 31.7592
Estimator 22/300, Train metric: 31.2400
Estimator 23/300, Train metric: 31.0484
Estimator 24/300, Train metric: 30

Estimator 53/300, Train metric: 20.1548
Estimator 54/300, Train metric: 19.9734
Estimator 55/300, Train metric: 19.5426
Estimator 56/300, Train metric: 19.3560
Estimator 57/300, Train metric: 19.0755
Estimator 58/300, Train metric: 19.0030
Estimator 59/300, Train metric: 18.6201
Estimator 60/300, Train metric: 18.3472
Estimator 61/300, Train metric: 18.2005
Estimator 62/300, Train metric: 17.9819
Estimator 63/300, Train metric: 17.8300
Estimator 64/300, Train metric: 17.5498
Estimator 65/300, Train metric: 17.5225
Estimator 66/300, Train metric: 17.3727
Estimator 67/300, Train metric: 17.3070
Estimator 68/300, Train metric: 17.1678
Estimator 69/300, Train metric: 17.0614
Estimator 70/300, Train metric: 16.7803
Estimator 71/300, Train metric: 16.4749
Estimator 72/300, Train metric: 16.3948
Estimator 73/300, Train metric: 16.2032
Estimator 74/300, Train metric: 15.9211
Estimator 75/300, Train metric: 15.7556
Estimator 76/300, Train metric: 15.4573
Estimator 77/300, Train metric: 15.4301


Estimator 106/300, Train metric: 12.4934
Estimator 107/300, Train metric: 12.4208
Estimator 108/300, Train metric: 12.3121
Estimator 109/300, Train metric: 12.1284
Estimator 110/300, Train metric: 12.0086
Estimator 111/300, Train metric: 11.8016
Estimator 112/300, Train metric: 11.7318
Estimator 113/300, Train metric: 11.6670
Estimator 114/300, Train metric: 11.6887
Estimator 115/300, Train metric: 11.5860
Estimator 116/300, Train metric: 11.5622
Estimator 117/300, Train metric: 11.3964
Estimator 118/300, Train metric: 11.3872
Estimator 119/300, Train metric: 11.1432
Estimator 120/300, Train metric: 11.0086
Estimator 121/300, Train metric: 10.8916
Estimator 122/300, Train metric: 10.7412
Estimator 123/300, Train metric: 10.6977
Estimator 124/300, Train metric: 10.5051
Estimator 125/300, Train metric: 10.3668
Estimator 126/300, Train metric: 10.1861
Estimator 127/300, Train metric: 10.1135
Estimator 128/300, Train metric: 10.1220
Estimator 129/300, Train metric: 10.0885
Estimator 130/30

Estimator 156/300, Train metric: 8.4016
Estimator 157/300, Train metric: 8.3515
Estimator 158/300, Train metric: 8.3379
Estimator 159/300, Train metric: 8.2709
Estimator 160/300, Train metric: 8.1762
Estimator 161/300, Train metric: 8.1119
Estimator 162/300, Train metric: 7.9747
Estimator 163/300, Train metric: 7.8391
Estimator 164/300, Train metric: 7.8195
Estimator 165/300, Train metric: 7.8222
Estimator 166/300, Train metric: 7.7660
Estimator 167/300, Train metric: 7.7863
Estimator 168/300, Train metric: 7.7255
Estimator 169/300, Train metric: 7.5893
Estimator 170/300, Train metric: 7.5417
Estimator 171/300, Train metric: 7.4689
Estimator 172/300, Train metric: 7.4514
Estimator 173/300, Train metric: 7.3731
Estimator 174/300, Train metric: 7.3000
Estimator 175/300, Train metric: 7.2695
Estimator 176/300, Train metric: 7.2219
Estimator 177/300, Train metric: 7.1383
Estimator 178/300, Train metric: 7.0303
Estimator 179/300, Train metric: 7.0361
Estimator 180/300, Train metric: 6.9048


Estimator 204/300, Train metric: 6.0880
Estimator 205/300, Train metric: 6.0382
Estimator 206/300, Train metric: 6.0201
Estimator 207/300, Train metric: 6.0035
Estimator 208/300, Train metric: 6.0043
Estimator 209/300, Train metric: 5.9166
Estimator 210/300, Train metric: 5.8947
Estimator 211/300, Train metric: 5.8349
Estimator 212/300, Train metric: 5.7705
Estimator 213/300, Train metric: 5.7236
Estimator 214/300, Train metric: 5.6761
Estimator 215/300, Train metric: 5.6347
Estimator 216/300, Train metric: 5.6130
Estimator 217/300, Train metric: 5.5424
Estimator 218/300, Train metric: 5.4981
Estimator 219/300, Train metric: 5.4567
Estimator 220/300, Train metric: 5.4494
Estimator 221/300, Train metric: 5.3933
Estimator 222/300, Train metric: 5.3735
Estimator 223/300, Train metric: 5.3358
Estimator 224/300, Train metric: 5.2964
Estimator 225/300, Train metric: 5.2615
Estimator 226/300, Train metric: 5.1988
Estimator 227/300, Train metric: 5.1459
Estimator 228/300, Train metric: 5.1362


Estimator 254/300, Train metric: 4.4071
Estimator 255/300, Train metric: 4.3765
Estimator 256/300, Train metric: 4.3708
Estimator 257/300, Train metric: 4.3648
Estimator 258/300, Train metric: 4.3656
Estimator 259/300, Train metric: 4.3355
Estimator 260/300, Train metric: 4.3162
Estimator 261/300, Train metric: 4.3098
Estimator 262/300, Train metric: 4.3069
Estimator 263/300, Train metric: 4.3089
Estimator 264/300, Train metric: 4.2819
Estimator 265/300, Train metric: 4.2891
Estimator 266/300, Train metric: 4.2473
Estimator 267/300, Train metric: 4.2144
Estimator 268/300, Train metric: 4.1879
Estimator 269/300, Train metric: 4.1611
Estimator 270/300, Train metric: 4.1214
Estimator 271/300, Train metric: 4.1036
Estimator 272/300, Train metric: 4.0990
Estimator 273/300, Train metric: 4.0711
Estimator 274/300, Train metric: 4.0584
Estimator 275/300, Train metric: 4.0464
Estimator 276/300, Train metric: 3.9909
Estimator 277/300, Train metric: 3.9908
Estimator 278/300, Train metric: 3.9303


 threshold_level  threshold_u  n_exceedances     sigma       xi  q_threshold
        0.333333    -9.061442             10 11.850386 0.368527    23.481194

===== EXTREME CP (PGBM) =====
Coverage     : 89.362%
Width        : 23.481
RMSE         : 23.436
q_extreme    : 23.481
threshold_u  : -9.061
#Exceedances : 10


In [69]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_AlphaNet_PGBM(
    model_class,
    model_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="PGBM",
    excel_file_path=None,
    alpha=0.05,
    lambda_reg=20.0,
    num_epochs=50,
    batch_size=32,
    lr_alpha=1e-2,  # FIX A3 (was 1e-3)
    calib_fraction=0.5,
    num_runs=4,
    random_state=123,
):
    warnings.filterwarnings("ignore")
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    t_start_total = time.time()

    # =========================================================
    # data conversion (PGBM-specific)
    # =========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy()  if hasattr(X_test,  "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # 1. Split training data → fit + calibration
    # =========================================================
    n_total = len(Xtr)
    n_fit   = int(n_total * (1 - calib_fraction))
    perm    = np.random.permutation(n_total)

    fit_idx   = perm[:n_fit]
    calib_idx = perm[n_fit:]

    X_fit,   y_fit   = Xtr[fit_idx],   ytr[fit_idx]
    X_calib, y_calib = Xtr[calib_idx],  ytr[calib_idx]
    n_calib = len(X_calib)

    alpha_min = 1.0 / (n_calib + 1) + 0.01
    print(f"  [Info] n_calib={n_calib} | alpha_min={alpha_min:.4f}")

    # =========================================================
    # 2. Train base PGBM model
    # =========================================================
    print(f"  [Step 2/9] Training base {model_name} model...")
    t0 = time.time()
    base_model = model_class(**model_params)
    base_model.fit(X_fit, y_fit)
    print(f"  [Step 2/9] Done in {_fmt_time(time.time()-t0)}")

    def _safe_predict(m, X):
        p = m.predict(X)
        return p.ravel() if hasattr(p, "ravel") else np.asarray(p).ravel()

    # =========================================================
    # 3. PRECOMPUTE all LOO scores
    # =========================================================
    print(f"  [Step 3/9] Precomputing {n_calib} LOO score vectors...")
    t0 = time.time()

    precomputed_scores = []
    precomputed_sums   = []

    iterator = tqdm(range(n_calib), desc="    LOO precompute", leave=False) if HAS_TQDM else range(n_calib)

    for i in iterator:
        loo_X = np.delete(X_calib, i, axis=0)
        loo_y = np.delete(y_calib, i, axis=0)
        scores_loo = np.abs(_safe_predict(base_model, loo_X) - loo_y)

        scores_tensor = torch.tensor(scores_loo, dtype=torch.float32).to(device)
        precomputed_scores.append(scores_tensor)
        precomputed_sums.append(scores_loo.sum())

    elapsed_precompute = time.time() - t0
    print(f"  [Step 3/9] Done in {_fmt_time(elapsed_precompute)}")

    # FIX A2: divide the feature (sum of LOO scores) by its mean so the network input is O(1)
    # whatever the units of the target (was: raw sums, e.g. ~1e3, which saturate the network).
    feature_scale = float(np.mean(precomputed_sums)) or 1.0
    X_alpha = torch.tensor(
        (np.array(precomputed_sums) / feature_scale).reshape(-1, 1),
        dtype=torch.float32,
    ).to(device)

    # =========================================================
    # 4. Train AlphaNet
    # =========================================================
    # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
    # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
    # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
    # now matches the exact minimiser of the same loss for every lambda.
    num_epochs = max(num_epochs, int(np.ceil(2000 / np.ceil(n_calib / batch_size))))
    print(f"  [Step 4/9] Training AlphaNet: {num_runs} runs × {num_epochs} epochs")

    best_net  = None
    best_loss = np.inf
    history   = {"losses": [], "sizes": [], "alphas": []}

    t0_train = time.time()

    for run in range(num_runs):
        np.random.seed(run)
        torch.manual_seed(run)

        alpha_net = AlphaNet(input_dim=1, alpha_min=alpha_min).to(device)
        optimizer = optim.Adam(alpha_net.parameters(), lr=lr_alpha)

        Dataset = TensorDataset(X_alpha, torch.arange(n_calib))
        loader  = DataLoader(Dataset, batch_size=batch_size, shuffle=True)

        run_losses, run_sizes, run_alphas = [], [], []

        epoch_iter = (
            tqdm(range(num_epochs), desc=f"    Run {run+1}/{num_runs}", leave=False)
            if HAS_TQDM else range(num_epochs)
        )

        for epoch in epoch_iter:
            t_epoch_start = time.time()
            epoch_loss = epoch_size = epoch_alpha = 0.0

            for x_batch, idx_batch in loader:
                x_batch = x_batch.to(device)
                alpha_pred = alpha_net(x_batch)

                batch_sizes = []
                for j, idx in enumerate(idx_batch):
                    scores_t = precomputed_scores[idx.item()]
                    size_j = _interval_size(scores_t, alpha_pred[j])
                    batch_sizes.append(size_j)

                batch_sizes = torch.stack(batch_sizes)
                loss = (batch_sizes + lambda_reg * alpha_pred).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                epoch_size  += batch_sizes.mean().item()
                epoch_alpha += alpha_pred.mean().item()

            n_batches = len(loader)
            run_losses.append(epoch_loss  / n_batches)
            run_sizes.append(epoch_size   / n_batches)
            run_alphas.append(epoch_alpha / n_batches)

            epoch_time = time.time() - t_epoch_start

            if epoch % max(10, num_epochs // 10) == 0 or epoch == num_epochs - 1:  # FIX A3: ~10 log lines per run
                epochs_done  = (run * num_epochs) + epoch + 1
                epochs_total = num_runs * num_epochs
                elapsed_train = time.time() - t0_train
                eta = (elapsed_train / epochs_done) * (epochs_total - epochs_done)

                print(
                    f"  λ={lambda_reg} | Run {run+1}/{num_runs} | "
                    f"Epoch {epoch}/{num_epochs} | "
                    f"Loss: {run_losses[-1]:.4f} | "
                    f"Size: {run_sizes[-1]:.4f} | "
                    f"α̃: {run_alphas[-1]:.4f} | "
                    f"⏱ {epoch_time:.2f}s/epoch | "
                    f"ETA: {_fmt_time(eta)}"
                )

        if run_losses[-1] < best_loss:
            best_loss = run_losses[-1]
            best_net  = alpha_net

        history["losses"].append(run_losses)
        history["sizes"].append(run_sizes)
        history["alphas"].append(run_alphas)

        print(f"  ✓ Run {run+1} complete | Final loss: {run_losses[-1]:.4f}")

    train_time = time.time() - t0_train
    print(f"  [Step 4/9] Training done in {_fmt_time(train_time)}")

    alpha_net = best_net
    alpha_net.eval()

    # =========================================================
    # 5. Compute conformal intervals on TEST set
    # =========================================================
    print(f"  [Step 5/9] Computing conformal intervals...")
    scores_calib = np.abs(_safe_predict(base_model, X_calib) - y_calib)
    sum_scores   = scores_calib.sum()

    with torch.no_grad():
        feat_test = torch.tensor([sum_scores / feature_scale], dtype=torch.float32).to(device)  # FIX A2: same scale; FIX B11: .to(device) was inside this comment
        alpha_val = alpha_net(feat_test).item()

    denom = alpha_val * (n_calib + 1) - 1
    if denom <= 0:
        q = np.quantile(scores_calib, 1 - alpha)
        half_width = q
        print(f"  [Warning] α̃ too small; fallback q={q:.4f}")
    else:
        half_width = sum_scores / denom

    y_pred_test = _safe_predict(base_model, Xte)
    ymin = y_pred_test - half_width
    ymax = y_pred_test + half_width

    # =========================================================
    # 6. Metrics
    # =========================================================
    is_covered = ((yte >= ymin) & (yte <= ymax)).astype(int)
    coverage   = is_covered.mean()
    avg_width  = np.mean(ymax - ymin)
    rmse       = np.sqrt(mean_squared_error(yte, y_pred_test))

    total_time = time.time() - t_start_total

    print(f"\n{'='*55}")
    print(f"  ALPHANET CONFORMAL ({model_name})")
    print(f"{'='*55}")
    print(f"  Learned α̃     : {alpha_val:.4f}")
    print(f"  Half-width (r) : {half_width:.4f}")
    print(f"  Coverage       : {coverage*100:.3f}%")
    print(f"  Avg Width      : {avg_width:.4f}")
    print(f"  RMSE           : {rmse:.4f}")
    print(f"  Total time     : {_fmt_time(total_time)}")
    print(f"    ├─ Precompute: {_fmt_time(elapsed_precompute)}")
    print(f"    └─ Training  : {_fmt_time(train_time)}")
    print(f"{'='*55}")

    # =========================================================
    # 7. Summary & Values
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy":       "AlphaNet Conformal",
        "Coverage":       coverage,
        "Width":          avg_width,
        "RMSE":           rmse,
        "Alpha_nominal (not used for the interval)":  alpha,   # FIX A1: the interval uses Alpha_learned
        "Target_coverage": 1 - alpha_val,
        "Miss_over_alpha": (1 - coverage) / alpha_val,
        "Alpha_learned":  alpha_val,
        "Lambda_reg":     lambda_reg,
        "Total_time_sec": round(total_time, 2),
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true":       yte,
        "y_pred":       y_pred_test,
        "ymin":         ymin,
        "ymax":         ymax,
        "width":        ymax - ymin,
        "is_covered":   is_covered,
    })

    # =========================================================
    # 8. Plotting (SAVE ONLY)
    # =========================================================
    print(f"  [Step 8/9] Generating plots...")
    idx_arr = np.arange(len(yte))

    fig_pi, ax_pi = plt.subplots(figsize=(10, 4))
    ax_pi.scatter(idx_arr, yte, color="blue", s=10, alpha=0.7, label="True")
    ax_pi.fill_between(idx_arr, ymin, ymax, color="gray", alpha=0.5,
                       label=f"AlphaNet Interval (α̃={alpha_val:.3f})")
    ax_pi.plot(idx_arr, y_pred_test, color="red", lw=1, label="Predicted mean")
    ax_pi.set_title(f"AlphaNet Conformal ({model_name})")
    ax_pi.set_xlabel("Sample Number")
    ax_pi.set_ylabel(f"{TARGET_LABEL}")
    ax_pi.legend(loc="upper left")
    plt.tight_layout()

    buf_pi = io.BytesIO()
    fig_pi.savefig(buf_pi, format="png")
    buf_pi.seek(0)
    plt.close(fig_pi)

    arr_losses = np.array(history["losses"])
    arr_sizes  = np.array(history["sizes"])
    arr_alphas = np.array(history["alphas"])
    epochs_arr = np.arange(1, num_epochs + 1)

    fig_tc, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, ylabel, title in zip(
        axs,
        [arr_losses, arr_sizes, arr_alphas],
        ["Loss", "Mean Interval Size", r"Mean $\tilde{\alpha}$"],
        ["Training Loss", "Interval Size", r"Learned $\tilde{\alpha}$"],
    ):
        m = data.mean(axis=0)
        s = data.std(axis=0)
        ax.plot(epochs_arr, m, color="#0077bb")
        ax.fill_between(epochs_arr, m - s, m + s, color="#0077bb", alpha=0.25)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.5)
    plt.tight_layout()

    buf_tc = io.BytesIO()
    fig_tc.savefig(buf_tc, format="png")
    buf_tc.seek(0)
    plt.close(fig_tc)

    # =========================================================
    # 9. Excel Export
    # =========================================================
    if excel_file_path:
        print(f"  [Step 9/9] Saving to Excel...")
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active)

        def _unique_sheet(wb, name):
            base, i = name, 1
            while name in wb.sheetnames:
                name = f"{base}_{i}"; i += 1
            return name

        sn = _unique_sheet(wb, "AlphaNet_PGBM_Summary")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        img_pi = Image(buf_pi)
        sn = _unique_sheet(wb, "AlphaNet_PGBM_Plot_PI")
        ws = wb.create_sheet(sn)
        ws.add_image(img_pi, "A1")

        img_tc = Image(buf_tc)
        sn = _unique_sheet(wb, "AlphaNet_PGBM_Plot_Train")
        ws = wb.create_sheet(sn)
        ws.add_image(img_tc, "A1")

        sn = _unique_sheet(wb, "AlphaNet_PGBM_Values")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))
        print(f"  [Step 9/9] Saved!")

    print(f"\n  ✅ AlphaNet PGBM complete in {_fmt_time(total_time)}")
    return summary_df

In [70]:
conformal_predictions_AlphaNet_PGBM(
    model_class=PGBMWrapper,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/PGBM.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base PGBM model...
Training on CPU
Estimator 0/300, Train metric: 59.8080
Estimator 1/300, Train metric: 59.0243
Estimator 2/300, Train metric: 58.7837
Estimator 3/300, Train metric: 57.9027
Estimator 4/300, Train metric: 57.1104
Estimator 5/300, Train metric: 55.4804
Estimator 6/300, Train metric: 54.1079
Estimator 7/300, Train metric: 52.7957
Estimator 8/300, Train metric: 52.2203
Estimator 9/300, Train metric: 50.8190
Estimator 10/300, Train metric: 49.5273
Estimator 11/300, Train metric: 49.0101
Estimator 12/300, Train metric: 48.3958
Estimator 13/300, Train metric: 47.5341
Estimator 14/300, Train metric: 47.4911
Estimator 15/300, Train metric: 46.8011
Estimator 16/300, Train metric: 45.9502
Estimator 17/300, Train metric: 45.3504
Estimator 18/300, Train metric: 44.3521
Estimator 19/300, Train metric: 44.0826
Estimator 20/300, Train metric: 43.5489
Estimator 21/300, Train metric: 42.6037
Estimator 22/300, Train metric: 41

Estimator 61/300, Train metric: 24.9792
Estimator 62/300, Train metric: 24.4912
Estimator 63/300, Train metric: 24.2003
Estimator 64/300, Train metric: 24.1613
Estimator 65/300, Train metric: 24.0539
Estimator 66/300, Train metric: 23.8520
Estimator 67/300, Train metric: 23.7910
Estimator 68/300, Train metric: 23.5422
Estimator 69/300, Train metric: 23.0846
Estimator 70/300, Train metric: 22.5582
Estimator 71/300, Train metric: 22.2556
Estimator 72/300, Train metric: 22.0760
Estimator 73/300, Train metric: 22.0094
Estimator 74/300, Train metric: 22.0055
Estimator 75/300, Train metric: 21.8658
Estimator 76/300, Train metric: 21.5539
Estimator 77/300, Train metric: 21.5112
Estimator 78/300, Train metric: 21.4438
Estimator 79/300, Train metric: 21.4994
Estimator 80/300, Train metric: 21.3521
Estimator 81/300, Train metric: 21.2645
Estimator 82/300, Train metric: 21.1364
Estimator 83/300, Train metric: 20.7742
Estimator 84/300, Train metric: 20.7984
Estimator 85/300, Train metric: 20.8059


Estimator 127/300, Train metric: 15.7608
Estimator 128/300, Train metric: 15.5825
Estimator 129/300, Train metric: 15.5862
Estimator 130/300, Train metric: 15.5938
Estimator 131/300, Train metric: 15.6036
Estimator 132/300, Train metric: 15.4449
Estimator 133/300, Train metric: 15.3597
Estimator 134/300, Train metric: 15.2104
Estimator 135/300, Train metric: 15.2327
Estimator 136/300, Train metric: 15.0813
Estimator 137/300, Train metric: 15.0968
Estimator 138/300, Train metric: 14.9504
Estimator 139/300, Train metric: 14.7869
Estimator 140/300, Train metric: 14.8061
Estimator 141/300, Train metric: 14.8206
Estimator 142/300, Train metric: 14.7879
Estimator 143/300, Train metric: 14.6773
Estimator 144/300, Train metric: 14.7093
Estimator 145/300, Train metric: 14.7124
Estimator 146/300, Train metric: 14.7075
Estimator 147/300, Train metric: 14.7403
Estimator 148/300, Train metric: 14.6121
Estimator 149/300, Train metric: 14.4433
Estimator 150/300, Train metric: 14.4412
Estimator 151/30

Estimator 189/300, Train metric: 13.1706
Estimator 190/300, Train metric: 13.1882
Estimator 191/300, Train metric: 13.0761
Estimator 192/300, Train metric: 12.9508
Estimator 193/300, Train metric: 12.8344
Estimator 194/300, Train metric: 12.8603
Estimator 195/300, Train metric: 12.8738
Estimator 196/300, Train metric: 12.7884
Estimator 197/300, Train metric: 12.8003
Estimator 198/300, Train metric: 12.8322
Estimator 199/300, Train metric: 12.7403
Estimator 200/300, Train metric: 12.6373
Estimator 201/300, Train metric: 12.5175
Estimator 202/300, Train metric: 12.4196
Estimator 203/300, Train metric: 12.4429
Estimator 204/300, Train metric: 12.3249
Estimator 205/300, Train metric: 12.2345
Estimator 206/300, Train metric: 12.1795
Estimator 207/300, Train metric: 12.1482
Estimator 208/300, Train metric: 12.1554
Estimator 209/300, Train metric: 12.1564
Estimator 210/300, Train metric: 12.0683
Estimator 211/300, Train metric: 12.0874
Estimator 212/300, Train metric: 12.0082
Estimator 213/30

Estimator 257/300, Train metric: 10.0762
Estimator 258/300, Train metric: 9.9937
Estimator 259/300, Train metric: 9.9377
Estimator 260/300, Train metric: 9.9352
Estimator 261/300, Train metric: 9.8233
Estimator 262/300, Train metric: 9.8287
Estimator 263/300, Train metric: 9.8308
Estimator 264/300, Train metric: 9.8339
Estimator 265/300, Train metric: 9.8343
Estimator 266/300, Train metric: 9.7456
Estimator 267/300, Train metric: 9.6999
Estimator 268/300, Train metric: 9.6899
Estimator 269/300, Train metric: 9.5890
Estimator 270/300, Train metric: 9.5622
Estimator 271/300, Train metric: 9.5631
Estimator 272/300, Train metric: 9.5695
Estimator 273/300, Train metric: 9.5112
Estimator 274/300, Train metric: 9.5257
Estimator 275/300, Train metric: 9.5224
Estimator 276/300, Train metric: 9.5148
Estimator 277/300, Train metric: 9.4524
Estimator 278/300, Train metric: 9.4242
Estimator 279/300, Train metric: 9.4300
Estimator 280/300, Train metric: 9.4314
Estimator 281/300, Train metric: 9.3864

    LOO precompute:   0%|          | 0/24 [00:00<?, ?it/s]

... (48 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 60.8020 | Size: 40.8020 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 60.8020
  [Step 4/9] Training done in 44s
  [Step 5/9] Computing conformal intervals...

  ALPHANET CONFORMAL (PGBM)
  Learned α̃     : 1.0000
  Half-width (r) : 20.4010
  Coverage       : 63.830%
  Avg Width      : 40.8020
  RMSE           : 23.6046
  Total time     : 45s
    ├─ Precompute: 0s
    └─ Training  : 44s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet PGBM complete in 45s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.638298,40.80202,23.604563,0.1,0.0,0.361702,1.0,20.0,45.1


# **Conformal Predictions with TabNet**

In [71]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')
print(best_params)

{'n_d': 32, 'n_a': 64, 'n_steps': 10, 'gamma': 1.3, 'lambda_sparse': 0.0001, 'optimizer_params': {'lr': 0.02}, 'mask_type': 'entmax', 'n_shared': 1, 'n_independent': 1, 'scheduler_params': {'step_size': 10, 'gamma': 0.9}, 'scheduler_fn': <class 'torch.optim.lr_scheduler.StepLR'>, 'seed': 42, 'verbose': 0}


In [72]:
class validate:
    @staticmethod
    def coverage(int_pred, y_test):
        y_arr = np.array(y_test)
        return {strat_name: np.mean((y_arr >= int_pred[strat_name][:, 0]) & (y_arr <= int_pred[strat_name][:, 1])) for strat_name in int_pred}
    @staticmethod
    def width(int_pred):
        return {strat_name: np.mean(int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]) for strat_name in int_pred}
    @staticmethod
    def rmse(y_pred, y_test):
        y_arr = np.array(y_test)
        return {strat_name: np.sqrt(mean_squared_error(y_arr, y_pred[strat_name])) for strat_name in y_pred}
    @staticmethod
    def cwc(int_pred, y_test, miscoverage):
        widths = validate.width(int_pred)
        coverages = validate.coverage(int_pred, y_test)
        return {strat_name: widths[strat_name] * (1 + (coverages[strat_name] < (1 - miscoverage)) * (1 - coverages[strat_name])) for strat_name in int_pred}
    @staticmethod
    def cond_coverage(int_pred, y_test, num_bins=N_BINS):
        y_arr = np.array(y_test)
        cond_coverages = {}
        for strat_name in int_pred:
            widths = int_pred[strat_name][:, 1] - int_pred[strat_name][:, 0]
            bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
            digitized_widths = np.digitize(widths, bin_edges)
            strategy_cond_coverage = []
            for bin_num in range(1, num_bins + 1):
                indices_in_bin = np.where(digitized_widths == bin_num)[0]
                if len(indices_in_bin) > 0:
                    bin_coverage = np.mean((y_arr[indices_in_bin] >= int_pred[strat_name][indices_in_bin, 0]) &
                                           (y_arr[indices_in_bin] <= int_pred[strat_name][indices_in_bin, 1]))
                    strategy_cond_coverage.append(bin_coverage)
                else:
                    strategy_cond_coverage.append(np.nan)
            cond_coverages[strat_name] = strategy_cond_coverage
        return cond_coverages
class visualize:
    @staticmethod
    def goodness(y_true, y_pred, y_pred_low, y_pred_up, coverage, width, rmse, cwc, ax=None, title="Goodness Plot"):
        if ax is None:
            fig, ax = plt.subplots()
        y_true_np = np.array(y_true).ravel()
        inside_interval = (y_true_np >= y_pred_low) & (y_true_np <= y_pred_up)
        outside_interval = ~inside_interval
        ax.scatter(y_true_np[inside_interval], y_pred[inside_interval], color='green', s=10, label='Inside Interval')
        ax.scatter(y_true_np[outside_interval], y_pred[outside_interval], color='orange', s=10, label='Outside Interval')
        ax.scatter(y_true_np, y_true_np, color='blue', marker='x', s=30, linewidths=0.7, label='Ground Truth Values')
        ax.vlines(y_true_np, y_pred_low, y_pred_up, color='gray', alpha=0.5, label='Prediction Intervals')
        ax.plot([min(y_true_np), max(y_true_np)], [min(y_true_np), max(y_true_np)],
                 linestyle='--', color='blue', label='Ideal Prediction Line')
        ax.set_xlabel('Ground Truth')
        ax.set_ylabel('Predictions')
        ax.set_title(f'{title}\nCoverage: {coverage*100:.2f}%, Width: {width:.2f}, RMSE: {rmse:.2f}, CWC: {cwc:.2f}')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def width_size_occurrence(int_pred, train_intervals, num_bins=N_BINS, ax=None, x_lim=None, title="Width Size Occurrence"):
        if ax is None:
            fig, ax = plt.subplots()
        widths_test = np.abs(int_pred[:, 1] - int_pred[:, 0])
        widths_train = np.abs(train_intervals[:, 1] - train_intervals[:, 0])
        ax.hist(widths_test, bins=num_bins, alpha=0.6, label='Test Interval Widths', color='red')
        ax.hist(widths_train, bins=num_bins, alpha=0.4, label='Train Interval Widths', color='blue')
        if x_lim is not None:
            ax.set_xlim(x_lim)
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Occurrence (Frequency)')
        ax.set_title(f'{title} - Interval Width Occurrence')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

    @staticmethod
    def coverage_by_width(y_test, int_pred, miscoverage, cond_coverages, num_bins=N_BINS, ax=None, title="Coverage vs Width"):
        if ax is None:
            fig, ax = plt.subplots()
        widths = int_pred[:, 1] - int_pred[:, 0]
        bin_edges = np.histogram_bin_edges(widths, bins=num_bins)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]
        ax.bar(bin_centers, cond_coverages, width=bin_width, color='blue', alpha=0.7, edgecolor='black', linewidth=0.8, label='Conditional Coverage')
        ax.axhline(1 - miscoverage, color='red', linestyle='--', label=f'Nominal Coverage ({(1 - miscoverage)*100:.0f}%)')
        ax.set_xlabel('Interval Width')
        ax.set_ylabel('Conditional Coverage')
        ax.set_title(f'{title} - Coverage vs Interval Width')
        ax.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
        return ax

class TabNetRegressorCP(TabNetRegressor):
    def fit(self, X, y, *args, **kwargs):
        y = np.asarray(y)
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        return super().fit(X, y, *args, **kwargs)

    def predict(self, X, *args, **kwargs):
        return super().predict(X).flatten()


In [73]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_NEXCP_tabnet(
    X_train,
    y_train,
    X_test,
    y_test,
    best_params,
    model_name="TabNet",
    excel_file_path=None,
    alpha=0.1,
    decay=0.99,
    n_folds=5,
    n_bootstrap=50,
    full_grid_size=50,   # FIX N3: y-grid points per test point for NexCP-Full
):
    warnings.filterwarnings("ignore")

    SEED = 123
    rng = np.random.default_rng(SEED)
    np.random.seed(SEED)

    if "verbose" in best_params:
        best_params = dict(best_params)
        best_params.pop("verbose", None)

    model_class = TabNetRegressorCP

    strategies = {
        "NEXCP-Split": "NexCP Split",
        "NEXCP-Full": "NexCP Full",
        "NEXCP-J+aB": "NexCP Jackknife+ aB",
        "NEXCP-CV+": "NexCP CV+",
    }

    y_pred, int_pred = {}, {}

    y_train_np = np.asarray(y_train).ravel()
    y_test_np = np.asarray(y_test).ravel()

    # =========================================================
    # FIX N1-N3: corrected NexCP methods (Barber et al. 2023). Weights w_i = decay^(n-i): training rows
    # in their original (time) order, newest row weight 1. Residuals are OUT-OF-FOLD (each row scored by
    # a model that never saw it). Was: in-sample training residuals for every variant, an unweighted
    # "Full", bounds averaged (CV+) or min/max-ed (J+aB) without weights.
    # =========================================================
    X_tr = np.asarray(X_train)
    y_tr = np.asarray(y_train).ravel()
    X_te = np.asarray(X_test)
    n_tr = len(y_tr)
    weights = decay ** np.arange(n_tr - 1, -1, -1)
    notes = {}

    model = model_class(**best_params)
    model.fit(X_tr, y_tr)
    yhat_test = np.asarray(model.predict(X_te)).ravel()
    R_oof, mu_te_folds, fold_preds = _oof_fold_predictions(model_class, best_params, X_tr, y_tr, X_te, n_folds, SEED)

    # NexCP - Split, eq. (11)
    q, notes["NEXCP-Split"] = _cap_q(weighted_quantile(R_oof, weights, 1 - alpha), R_oof)
    y_pred["NEXCP-Split"] = yhat_test
    int_pred["NEXCP-Split"] = np.c_[yhat_test - q, yhat_test + q]

    # NexCP - Full, eq. (12), on a grid of candidate y values around the prediction (+/- 2 x split margin)
    lo, hi, notes["NEXCP-Full"] = _nexcp_full(model_class, best_params, X_tr, y_tr, X_te, yhat_test, weights,
                                              alpha, half_range=2.0 * q, grid_size=full_grid_size, label=model_name)
    y_pred["NEXCP-Full"] = yhat_test
    int_pred["NEXCP-Full"] = np.c_[lo, hi]

    # NexCP - Jackknife+ after bootstrap: out-of-bag residuals + eq. (33) weighted quantiles
    R_oob, mu_te_oob, bag_mean, has = _jab_oob(model_class, best_params, X_tr, y_tr, X_te, n_bootstrap, SEED)
    lo, hi, n_cap = _jackknife_plus(mu_te_oob, R_oob, weights[has], alpha)
    notes["NEXCP-J+aB"] = _cap_count_note(n_cap, len(lo))
    y_pred["NEXCP-J+aB"] = bag_mean
    int_pred["NEXCP-J+aB"] = np.c_[lo, hi]

    # NexCP - CV+, eq. (33) with K folds
    lo, hi, n_cap = _jackknife_plus(mu_te_folds, R_oof, weights, alpha)
    notes["NEXCP-CV+"] = _cap_count_note(n_cap, len(lo))
    y_pred["NEXCP-CV+"] = np.mean(fold_preds, axis=0)
    int_pred["NEXCP-CV+"] = np.c_[lo, hi]

    # =========================
    # Metrics
    # =========================
    def coverage(y, l, u):
        return np.mean((y >= l) & (y <= u))

    def width(l, u):
        return np.mean(u - l)

    def cwc(l, u, y, alpha):
        cov = coverage(y, l, u)
        w = width(l, u)
        penalty = np.exp(-50 * (cov - (1 - alpha)))
        return w * (1 + penalty if cov < (1 - alpha) else 1)

    summary = []

    print("\n===== NEXCP TABNET RESULTS =====")
    for s in strategies:
        cov = coverage(y_test_np, int_pred[s][:, 0], int_pred[s][:, 1])
        wid = width(int_pred[s][:, 0], int_pred[s][:, 1])
        rmse = np.sqrt(mean_squared_error(y_test_np, y_pred[s]))
        cwc_val = cwc(int_pred[s][:, 0], int_pred[s][:, 1], y_test_np, alpha)

        summary.append([strategies[s], cov, wid, rmse, cwc_val])
        print(f"{strategies[s]} | Coverage: {cov*100:.3f}% | Width: {wid:.3f}")

    summary_df = pd.DataFrame(
        summary,
        columns=["Strategy", "Coverage", "Width", "RMSE", "CWC"],
    )
    summary_df["Note"] = [notes.get(s, "") for s in strategies]  # FIX N10: caps / limits per method

    # =========================
    # Plot + Excel Export
    # =========================
    if excel_file_path:
        wb = load_workbook(_ensure_excel_file(excel_file_path))

        sheet = "NEXCP_TabNet_Summary"
        base = sheet
        i = 1
        while sheet in wb.sheetnames:
            sheet = f"{base}_{i}"
            i += 1

        ws = wb.create_sheet(sheet)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        for s in strategies:
            fig, ax = plt.subplots(figsize=(10, 4))
            idx = np.arange(len(y_test_np))

            ax.scatter(idx, y_test_np, s=10, alpha=0.7, label="True")
            ax.fill_between(idx, int_pred[s][:, 0], int_pred[s][:, 1], alpha=0.4)
            ax.plot(idx, y_pred[s], linewidth=1, label="Pred")

            ax.set_title(f"{strategies[s]} ({model_name})")
            ax.legend()

            buf = io.BytesIO()
            fig.savefig(buf, format="png")
            buf.seek(0)
            plt.close(fig)

            img = Image(buf)
            ps = f"NEXCP_TabNet_{s}"
            base = ps
            i = 1
            while ps in wb.sheetnames:
                ps = f"{base}_{i}"
                i += 1

            ws = wb.create_sheet(ps)
            ws.add_image(img, "A1")

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))

    return summary_df


conformal_predictions_NEXCP_tabnet(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    best_params=best_params,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    decay=0.99,
    n_folds=5,
    n_bootstrap=50
)

epoch 0  | loss: 20515.02539|  0:00:00s
epoch 1  | loss: 19362.41797|  0:00:00s
epoch 2  | loss: 18733.70703|  0:00:00s
epoch 3  | loss: 17943.61328|  0:00:00s


epoch 4  | loss: 17168.80664|  0:00:00s
epoch 5  | loss: 16311.9043|  0:00:00s
epoch 6  | loss: 15311.12402|  0:00:00s
epoch 7  | loss: 14406.68066|  0:00:00s
epoch 8  | loss: 13744.61719|  0:00:00s


epoch 9  | loss: 12864.39453|  0:00:00s
epoch 10 | loss: 11992.45898|  0:00:00s
epoch 11 | loss: 11312.65918|  0:00:00s
epoch 12 | loss: 10556.44238|  0:00:00s
epoch 13 | loss: 9720.99805|  0:00:00s


epoch 14 | loss: 8987.28125|  0:00:00s
epoch 15 | loss: 8063.77246|  0:00:00s
epoch 16 | loss: 7364.9873|  0:00:00s
epoch 17 | loss: 6668.06934|  0:00:00s
epoch 18 | loss: 5944.29395|  0:00:01s


epoch 19 | loss: 5337.32471|  0:00:01s
epoch 20 | loss: 4621.72266|  0:00:01s
epoch 21 | loss: 3936.42822|  0:00:01s
epoch 22 | loss: 3195.92798|  0:00:01s
epoch 23 | loss: 2563.80713|  0:00:01s


epoch 24 | loss: 1899.76501|  0:00:01s
epoch 25 | loss: 1439.422|  0:00:01s
epoch 26 | loss: 1113.2915|  0:00:01s
epoch 27 | loss: 733.51111|  0:00:01s
epoch 28 | loss: 577.4032|  0:00:01s


... (50,206 outputs left out)

epoch 87 | loss: 23.54015|  0:00:03s
epoch 88 | loss: 15.76434|  0:00:03s
epoch 89 | loss: 16.07187|  0:00:03s
epoch 90 | loss: 12.65162|  0:00:03s
epoch 91 | loss: 14.0799 |  0:00:03s


epoch 92 | loss: 18.4484 |  0:00:04s
epoch 93 | loss: 13.64027|  0:00:04s
epoch 94 | loss: 9.07601 |  0:00:04s
epoch 95 | loss: 6.85753 |  0:00:04s
epoch 96 | loss: 5.74996 |  0:00:04s


epoch 97 | loss: 14.40928|  0:00:04s
epoch 98 | loss: 11.61552|  0:00:04s
epoch 99 | loss: 9.06658 |  0:00:04s

===== NEXCP TABNET RESULTS =====
NexCP Split | Coverage: 91.489% | Width: 136.862
NexCP Full | Coverage: 97.872% | Width: 256.015
NexCP Jackknife+ aB | Coverage: 97.872% | Width: 108.860
NexCP CV+ | Coverage: 100.000% | Width: 185.601


,Strategy,Coverage,Width,RMSE,CWC,Note
0,NexCP Split,0.914894,136.862152,46.986503,136.862152,
1,NexCP Full,0.978723,256.014771,46.986503,256.014771,44 of 47 intervals reach the edge of the y-gri...
2,NexCP Jackknife+ aB,0.978723,108.860235,22.070255,108.860235,
3,NexCP CV+,1.000000,185.601301,25.627940,185.601301,


In [74]:
y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

conformal_predictions_online_super(
    model_class=TabNetRegressorCP,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    strategies="all",
    window_min=50,
    window_max=500,
    window_grid=(25, 50, 100, 200, 400),
    n_bootstrap=30
)


===== ONLINE CP RESULTS =====
               Strategy  Coverage       Width       RMSE         CWC Note
0          Online Split  0.914894  136.862152  46.986503  136.862152     
1            Online CV+  1.000000  184.799922  46.986503  184.799922     
2  Online Jackknife+ aB  0.978723   95.529167  21.728309   95.529167     
3                 SAOCP  0.914894  161.459780  46.986503  161.459780     
4                SF-OGD  0.936170  180.603220  46.986503  180.603220     
5                  FACI  0.914894  134.968582  46.986503  134.968582     
6                   ACI  0.893617  135.550123  46.986503  149.970349     
7            MFCS Split  0.914894  136.862152  46.986503  136.862152     
8             MFCS Full  0.936170  122.189940  31.331525  122.189940     


,Strategy,Coverage,Width,RMSE,CWC,Note
0,Online Split,0.914894,136.862152,46.986503,136.862152,
1,Online CV+,1.000000,184.799922,46.986503,184.799922,
2,Online Jackknife+ aB,0.978723,95.529167,21.728309,95.529167,
3,SAOCP,0.914894,161.459780,46.986503,161.459780,
4,SF-OGD,0.936170,180.603220,46.986503,180.603220,
5,FACI,0.914894,134.968582,46.986503,134.968582,
6,ACI,0.893617,135.550123,46.986503,149.970349,
7,MFCS Split,0.914894,136.862152,46.986503,136.862152,
8,MFCS Full,0.936170,122.189940,31.331525,122.189940,


In [75]:
conformal_predictions_COP(
    model_class=TabNetRegressorCP,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    lr=0.05,
    T_burnin=COP_T_BURNIN,
    scale=0.5
)


===== CONFORMAL OPTIMISTIC PREDICTION (TabNet) =====
Coverage : 91.489%
Avg Width: 138.085
RMSE     : 46.987
Final q  : 69.166


,Strategy,Coverage,Width,RMSE,Alpha,LR,Scale
0,COP,0.914894,138.085102,46.986503,0.1,0.05,0.5


In [76]:
conformal_predictions_EXTREME(
    model_class=TabNetRegressorCP,
    model_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
    calib_size=0.4,
    gpd_threshold_quantile=0.95,
    n_bootstrap=30
)


 threshold_level  threshold_u  n_exceedances     sigma        xi  q_threshold
        0.473684     0.508956             10 45.654122 -0.428309     54.76378

===== EXTREME CONFORMAL PREDICTION (TabNet) =====
Coverage : 95.745%
Width    : 54.764
RMSE     : 34.574
q_extreme: 54.764
u        : 0.509
#Exceed. : 10


In [77]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def conformal_predictions_AlphaNet_tabnet(
    X_train,
    y_train,
    X_test,
    y_test,
    best_params,
    model_name="TabNet",
    excel_file_path=None,
    alpha=0.05,
    lambda_reg=20.0,
    num_epochs=50,
    batch_size=32,
    lr_alpha=1e-2,  # FIX A3 (was 1e-3)
    calib_fraction=0.5,
    num_runs=4,
    random_state=123,
):
    """
    Learned-α Conformal Prediction via AlphaNet — TabNet-specific variant.

    Handles:
    - TabNetRegressorCP wrapper (y reshape, predict flatten)
    - Removes 'verbose' from best_params if present
    - Precomputed LOO scores for speed
    - Progress tracking with tqdm + ETA
    """

    warnings.filterwarnings("ignore")
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    t_start_total = time.time()

    # =========================================================
    # TabNet-specific: clean params & set model class
    # =========================================================
    tabnet_params = dict(best_params)
    tabnet_params.pop("verbose", None)           # TabNet doesn't use verbose here
    model_class = TabNetRegressorCP

    # =========================================================
    # data conversion
    # =========================================================
    Xtr = X_train.to_numpy() if hasattr(X_train, "to_numpy") else np.asarray(X_train)
    Xte = X_test.to_numpy()  if hasattr(X_test,  "to_numpy") else np.asarray(X_test)
    ytr = to_numpy_1d(y_train)
    yte = to_numpy_1d(y_test)

    # =========================================================
    # 1. Split training data → fit + calibration
    # =========================================================
    n_total = len(Xtr)
    n_fit   = int(n_total * (1 - calib_fraction))
    perm    = np.random.permutation(n_total)

    fit_idx   = perm[:n_fit]
    calib_idx = perm[n_fit:]

    X_fit,   y_fit   = Xtr[fit_idx],   ytr[fit_idx]
    X_calib, y_calib = Xtr[calib_idx],  ytr[calib_idx]
    n_calib = len(X_calib)

    # alpha_min: must satisfy α̃·(n+1) − 1 > 0  →  α̃ > 1/(n+1)
    alpha_min = 1.0 / (n_calib + 1) + 0.01
    print(f"  [Info] n_calib={n_calib} | alpha_min={alpha_min:.4f}")

    # =========================================================
    # 2. Train base TabNet model
    # =========================================================
    print(f"  [Step 2/9] Training base {model_name} model...")
    t0 = time.time()
    base_model = model_class(**tabnet_params)
    base_model.fit(X_fit, y_fit)
    print(f"  [Step 2/9] Done in {_fmt_time(time.time()-t0)}")

    # =========================================================
    # 3. PRECOMPUTE all LOO scores (KEY SPEEDUP)
    # =========================================================
    print(f"  [Step 3/9] Precomputing {n_calib} LOO score vectors...")
    t0 = time.time()

    precomputed_scores = []
    precomputed_sums   = []

    iterator = (
        tqdm(range(n_calib), desc="    LOO precompute", leave=False)
        if HAS_TQDM else range(n_calib)
    )

    for i in iterator:
        loo_X = np.delete(X_calib, i, axis=0)
        loo_y = np.delete(y_calib, i, axis=0)
        preds_loo = base_model.predict(loo_X).ravel()
        scores_loo = np.abs(preds_loo - loo_y)

        scores_tensor = torch.tensor(scores_loo, dtype=torch.float32).to(device)
        precomputed_scores.append(scores_tensor)
        precomputed_sums.append(scores_loo.sum())

    elapsed_precompute = time.time() - t0
    print(f"  [Step 3/9] Done in {_fmt_time(elapsed_precompute)}")

    # Build feature matrix (sum of LOO scores per sample)
    # FIX A2: divide the feature (sum of LOO scores) by its mean so the network input is O(1)
    # whatever the units of the target (was: raw sums, e.g. ~1e3, which saturate the network).
    feature_scale = float(np.mean(precomputed_sums)) or 1.0
    X_alpha = torch.tensor(
        (np.array(precomputed_sums) / feature_scale).reshape(-1, 1),
        dtype=torch.float32,
    ).to(device)

    # =========================================================
    # 4. Train AlphaNet (multiple runs, keep best)
    # =========================================================
    # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
    # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
    # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
    # now matches the exact minimiser of the same loss for every lambda.
    num_epochs = max(num_epochs, int(np.ceil(2000 / np.ceil(n_calib / batch_size))))
    print(f"  [Step 4/9] Training AlphaNet: {num_runs} runs × {num_epochs} epochs")

    best_net  = None
    best_loss = np.inf
    history   = {"losses": [], "sizes": [], "alphas": []}

    t0_train = time.time()

    for run in range(num_runs):
        np.random.seed(run)
        torch.manual_seed(run)

        alpha_net = AlphaNet(input_dim=1, alpha_min=alpha_min).to(device)
        optimizer = optim.Adam(alpha_net.parameters(), lr=lr_alpha)

        Dataset = TensorDataset(X_alpha, torch.arange(n_calib))
        loader  = DataLoader(Dataset, batch_size=batch_size, shuffle=True)

        run_losses, run_sizes, run_alphas = [], [], []

        epoch_iter = (
            tqdm(range(num_epochs), desc=f"    Run {run+1}/{num_runs}", leave=False)
            if HAS_TQDM else range(num_epochs)
        )

        for epoch in epoch_iter:
            t_epoch_start = time.time()
            epoch_loss = epoch_size = epoch_alpha = 0.0

            for x_batch, idx_batch in loader:
                x_batch = x_batch.to(device)
                alpha_pred = alpha_net(x_batch)

                batch_sizes = []
                for j, idx in enumerate(idx_batch):
                    # USE PRECOMPUTED scores — no predict() call here
                    scores_t = precomputed_scores[idx.item()]
                    size_j = _interval_size(scores_t, alpha_pred[j])
                    batch_sizes.append(size_j)

                batch_sizes = torch.stack(batch_sizes)
                loss = (batch_sizes + lambda_reg * alpha_pred).mean()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss  += loss.item()
                epoch_size  += batch_sizes.mean().item()
                epoch_alpha += alpha_pred.mean().item()

            n_batches = len(loader)
            run_losses.append(epoch_loss  / n_batches)
            run_sizes.append(epoch_size   / n_batches)
            run_alphas.append(epoch_alpha / n_batches)

            epoch_time = time.time() - t_epoch_start

            if epoch % max(10, num_epochs // 10) == 0 or epoch == num_epochs - 1:  # FIX A3: ~10 log lines per run
                epochs_done  = (run * num_epochs) + epoch + 1
                epochs_total = num_runs * num_epochs
                elapsed_train = time.time() - t0_train
                eta = (elapsed_train / epochs_done) * (epochs_total - epochs_done)

                print(
                    f"  λ={lambda_reg} | Run {run+1}/{num_runs} | "
                    f"Epoch {epoch}/{num_epochs} | "
                    f"Loss: {run_losses[-1]:.4f} | "
                    f"Size: {run_sizes[-1]:.4f} | "
                    f"α̃: {run_alphas[-1]:.4f} | "
                    f"⏱ {epoch_time:.2f}s/epoch | "
                    f"ETA: {_fmt_time(eta)}"
                )

        if run_losses[-1] < best_loss:
            best_loss = run_losses[-1]
            best_net  = alpha_net

        history["losses"].append(run_losses)
        history["sizes"].append(run_sizes)
        history["alphas"].append(run_alphas)

        print(f"  ✓ Run {run+1} complete | Final loss: {run_losses[-1]:.4f}")

    train_time = time.time() - t0_train
    print(f"  [Step 4/9] Training done in {_fmt_time(train_time)}")

    alpha_net = best_net
    alpha_net.eval()

    # =========================================================
    # 5. Compute conformal intervals on TEST set
    # =========================================================
    print(f"  [Step 5/9] Computing conformal intervals...")

    calib_preds = base_model.predict(X_calib).ravel()
    scores_calib = np.abs(calib_preds - y_calib)
    sum_scores   = scores_calib.sum()

    with torch.no_grad():
        feat_test = torch.tensor([sum_scores / feature_scale], dtype=torch.float32).to(device)  # FIX A2: same scale; FIX B11: .to(device) was inside this comment
        alpha_val = alpha_net(feat_test).item()

    denom = alpha_val * (n_calib + 1) - 1
    if denom <= 0:
        q = np.quantile(scores_calib, 1 - alpha)
        half_width = q
        print(f"  [Warning] α̃ too small (denom={denom:.4f}); fallback q={q:.4f}")
    else:
        half_width = sum_scores / denom

    y_pred_test = base_model.predict(Xte).ravel()
    ymin = y_pred_test - half_width
    ymax = y_pred_test + half_width

    # =========================================================
    # 6. Metrics
    # =========================================================
    is_covered = ((yte >= ymin) & (yte <= ymax)).astype(int)
    coverage   = is_covered.mean()
    avg_width  = np.mean(ymax - ymin)
    rmse       = np.sqrt(mean_squared_error(yte, y_pred_test))

    total_time = time.time() - t_start_total

    print(f"\n{'='*55}")
    print(f"  ALPHANET CONFORMAL ({model_name})")
    print(f"{'='*55}")
    print(f"  Learned α̃     : {alpha_val:.4f}")
    print(f"  Half-width (r) : {half_width:.4f}")
    print(f"  Coverage       : {coverage*100:.3f}%")
    print(f"  Avg Width      : {avg_width:.4f}")
    print(f"  RMSE           : {rmse:.4f}")
    print(f"  Total time     : {_fmt_time(total_time)}")
    print(f"    ├─ Precompute: {_fmt_time(elapsed_precompute)}")
    print(f"    └─ Training  : {_fmt_time(train_time)}")
    print(f"{'='*55}")

    # =========================================================
    # 7. Summary & Values
    # =========================================================
    summary_df = pd.DataFrame([{
        "Strategy":       "AlphaNet Conformal",
        "Coverage":       coverage,
        "Width":          avg_width,
        "RMSE":           rmse,
        "Alpha_nominal (not used for the interval)":  alpha,   # FIX A1: the interval uses Alpha_learned
        "Target_coverage": 1 - alpha_val,
        "Miss_over_alpha": (1 - coverage) / alpha_val,
        "Alpha_learned":  alpha_val,
        "Lambda_reg":     lambda_reg,
        "Total_time_sec": round(total_time, 2),
    }])

    values_df = pd.DataFrame({
        "sample_index": np.arange(len(yte)),
        "y_true":       yte,
        "y_pred":       y_pred_test,
        "ymin":         ymin,
        "ymax":         ymax,
        "width":        ymax - ymin,
        "is_covered":   is_covered,
    })

    # =========================================================
    # 8. Plotting (SAVE ONLY — NO SHOW)
    # =========================================================
    print(f"  [Step 8/9] Generating plots...")
    idx_arr = np.arange(len(yte))

    # --- Prediction-interval plot ---
    fig_pi, ax_pi = plt.subplots(figsize=(10, 4))
    ax_pi.scatter(idx_arr, yte, color="blue", s=10, alpha=0.7, label="True")
    ax_pi.fill_between(idx_arr, ymin, ymax, color="gray", alpha=0.5,
                       label=f"AlphaNet Interval (α̃={alpha_val:.3f})")
    ax_pi.plot(idx_arr, y_pred_test, color="red", lw=1, label="Predicted mean")
    ax_pi.set_title(f"AlphaNet Conformal ({model_name})")
    ax_pi.set_xlabel("Sample Number")
    ax_pi.set_ylabel(f"{TARGET_LABEL}")
    ax_pi.legend(loc="upper left")
    plt.tight_layout()

    buf_pi = io.BytesIO()
    fig_pi.savefig(buf_pi, format="png")
    buf_pi.seek(0)
    plt.close(fig_pi)

    # --- Training-curves plot ---
    arr_losses = np.array(history["losses"])
    arr_sizes  = np.array(history["sizes"])
    arr_alphas = np.array(history["alphas"])
    epochs_arr = np.arange(1, num_epochs + 1)

    fig_tc, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, ylabel, title in zip(
        axs,
        [arr_losses, arr_sizes, arr_alphas],
        ["Loss", "Mean Interval Size", r"Mean $\tilde{\alpha}$"],
        ["Training Loss", "Interval Size", r"Learned $\tilde{\alpha}$"],
    ):
        m = data.mean(axis=0)
        s = data.std(axis=0)
        ax.plot(epochs_arr, m, color="#0077bb")
        ax.fill_between(epochs_arr, m - s, m + s, color="#0077bb", alpha=0.25)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, linewidth=0.5)
    plt.tight_layout()

    buf_tc = io.BytesIO()
    fig_tc.savefig(buf_tc, format="png")
    buf_tc.seek(0)
    plt.close(fig_tc)

    # =========================================================
    # 9. Excel Export (append-safe)
    # =========================================================
    if excel_file_path:
        print(f"  [Step 9/9] Saving to Excel...")
        try:
            wb = load_workbook(_ensure_excel_file(excel_file_path))
        except FileNotFoundError:
            from openpyxl import Workbook
            wb = Workbook()
            wb.remove(wb.active)

        def _unique_sheet(wb, name):
            base, i = name, 1
            while name in wb.sheetnames:
                name = f"{base}_{i}"; i += 1
            return name

        # ---- Summary ----
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Summary")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(summary_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        # ---- PI Plot ----
        img_pi = Image(buf_pi)
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Plot_PI")
        ws = wb.create_sheet(sn)
        ws.add_image(img_pi, "A1")

        # ---- Training Curves Plot ----
        img_tc = Image(buf_tc)
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Plot_Train")
        ws = wb.create_sheet(sn)
        ws.add_image(img_tc, "A1")

        # ---- Values ----
        sn = _unique_sheet(wb, "AlphaNet_TabNet_Values")
        ws = wb.create_sheet(sn)
        for r, row in enumerate(dataframe_to_rows(values_df, index=False, header=True), 1):
            for c, val in enumerate(row, 1):
                ws.cell(row=r, column=c, value=val)

        _ensure_parent_dir(excel_file_path)
        wb.save(_ensure_parent_dir(excel_file_path))
        print(f"  [Step 9/9] Saved!")

    print(f"\n  ✅ AlphaNet TabNet complete in {_fmt_time(total_time)}")
    return summary_df

In [78]:
conformal_predictions_AlphaNet_tabnet(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    best_params=best_params,
    model_name="TabNet",
    excel_file_path="./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/conformal_predictions_nexcp_adaptivecp_mfcs/TabNet.xlsx",
    alpha=ALPHA,  # FIX M4
)

  [Info] n_calib=24 | alpha_min=0.0500
  [Step 2/9] Training base TabNet model...
epoch 0  | loss: 22321.71289|  0:00:00s
epoch 1  | loss: 21614.06836|  0:00:00s
epoch 2  | loss: 20632.70508|  0:00:00s


epoch 3  | loss: 19789.15625|  0:00:00s
epoch 4  | loss: 19202.41797|  0:00:00s
epoch 5  | loss: 18580.11133|  0:00:00s
epoch 6  | loss: 17816.31836|  0:00:00s
epoch 7  | loss: 17147.2832|  0:00:00s
epoch 8  | loss: 16391.25195|  0:00:00s


epoch 9  | loss: 15479.97168|  0:00:00s
epoch 10 | loss: 14604.83398|  0:00:00s
epoch 11 | loss: 13801.57715|  0:00:00s
epoch 12 | loss: 13238.26855|  0:00:00s
epoch 13 | loss: 12205.17969|  0:00:00s
epoch 14 | loss: 11447.24707|  0:00:00s


epoch 15 | loss: 10412.01465|  0:00:00s
epoch 16 | loss: 9595.07129|  0:00:00s
epoch 17 | loss: 8994.08691|  0:00:00s
epoch 18 | loss: 8402.87598|  0:00:00s
epoch 19 | loss: 7643.47363|  0:00:00s
epoch 20 | loss: 6831.99609|  0:00:00s


epoch 21 | loss: 6199.52783|  0:00:00s
epoch 22 | loss: 5456.97656|  0:00:00s
epoch 23 | loss: 4769.0752|  0:00:00s
epoch 24 | loss: 4195.79688|  0:00:00s
epoch 25 | loss: 3484.59302|  0:00:01s
epoch 26 | loss: 2825.08765|  0:00:01s


epoch 27 | loss: 2126.82568|  0:00:01s
epoch 28 | loss: 1562.89978|  0:00:01s
epoch 29 | loss: 969.00977|  0:00:01s
epoch 30 | loss: 521.02344|  0:00:01s
epoch 31 | loss: 208.87434|  0:00:01s
epoch 32 | loss: 36.57402|  0:00:01s


... (61 outputs left out)

  λ=20.0 | Run 4/4 | Epoch 1999/2000 | Loss: 80.0614 | Size: 60.0614 | α̃: 1.0000 | ⏱ 0.00s/epoch | ETA: 0s
  ✓ Run 4 complete | Final loss: 80.0614
  [Step 4/9] Training done in 49s
  [Step 5/9] Computing conformal intervals...

  ALPHANET CONFORMAL (TabNet)
  Learned α̃     : 1.0000
  Half-width (r) : 30.0307
  Coverage       : 48.936%
  Avg Width      : 60.0614
  RMSE           : 63.7426
  Total time     : 53s
    ├─ Precompute: 0s
    └─ Training  : 49s
  [Step 8/9] Generating plots...


  [Step 9/9] Saving to Excel...


  [Step 9/9] Saved!

  ✅ AlphaNet TabNet complete in 53s


,Strategy,Coverage,Width,RMSE,Alpha_nominal (not used for the interval),Target_coverage,Miss_over_alpha,Alpha_learned,Lambda_reg,Total_time_sec
0,AlphaNet Conformal,0.489362,60.061355,63.742576,0.1,0.0,0.510638,1.0,20.0,53.06
